# Result Analyses
Evaluatie van vier LLM-architecturen voor strategische adviesgeneratie op basis van menukaartdata:
- **Experiment A** — Baseline (enkel prompt + casecontext)
- **Experiment B** — RAG (menu engineering + scenariotheorie)
- **Experiment C** — Web search (real-time marktinformatie)
- **Experiment D** — Self-Refine (iteratieve zelfreflectie: generate → critique → refine)

**Deelvragen geadresseerd in dit notebook:**
| Sectie | Evaluatiedimensie | Onderzoeksvraag |
|--------|-------------------|------------------|
| §4.1 | Basisanalyse | Overzicht runs, tokens, latency en eerste stabiliteitsberekening |
| §4.2 | Parameter sensitivity | *Zijn de bevindingen robuust over verschillende temperatuurinstellingen?* |
| §4.3 | Kwaliteit (LLM-as-judge) | *Wat is de inhoudelijke kwaliteit van de gegenereerde adviezen?* |
| §4.4 | CoT stap-analyse | *Levert elke redeneerstap betere kwaliteit op en worden inzichten doorvertaald?* |
| §4.5 | Consistentie | *Hoe stabiel/reproduceerbaar zijn de adviezen over meerdere runs?* |
| §4.6 | Efficiëntie | *Hoe efficiënt zijn de architecturen qua tokens en verwerkingstijd?* |
| §4.7 | Semantische vergelijking | *Hoe sterk verschillen de outputs van A, B, C en D?* |
| §4.8 | CGR & Hallucinatie | *In hoeverre zijn de adviezen gegrond in de aangeleverde context?* |
| §4.9 | Scenariokwaliteit | *Hoe divers zijn de voorgestelde scenario’s binnen één advies?* |
| §4.10 | Integrale vergelijking | *Welke architectuur presteert het beste op de gewogen totaalscore?* |
| §4.11 | Statistische toetsing | *Zijn de gevonden verschillen statistisch significant?* |
| §4.12 | Verbosity bias | *Beoordeelt de judge langere antwoorden systematisch hoger?* |
| §4.13 | Judge-Ranking contradictie | *Waarom geven rubric en pairwise tegengestelde conclusies?* |
| §4.14 | Kosten-baten | *Is extra rekentijd gerechtvaardigd door betere kwaliteit?* |
| §4.15 | Convergentie strategische opties | *Hoe dicht bij elkaar liggen de gegenereerde strategieën?* |

### Imports en setup
Installatie van benodigde packages en laden van de OpenAI client voor LLM-as-judge beoordelingen.

In [0]:
%pip install sentence-transformers scikit-learn pandas numpy matplotlib seaborn plotly openai -q

import os, sys, json, glob, re, itertools, math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# Project imports
sys.path.insert(0, "/Workspace/Users/rgoedings@hanos.nl/AI_op_het_menu")
from utils.openai_client import get_client

openai_client = get_client()
MODEL = "gpt-4o-mini"

### Configuratie
Paden, experimenten, embedding model (paraphrase-multilingual-MiniLM-L12-v2) en gewichten voor de totaalscore.

In [0]:
PROJECT_ROOT = "/Workspace/Users/rgoedings@hanos.nl/AI_op_het_menu"
RESULTS_DIR = os.path.join(PROJECT_ROOT, "data", "results_data")

EXPERIMENTS = ["a", "b", "c", "d"]
RETRIEVAL_EXPERIMENTS = ["b", "c"]  # experimenten met externe context

# Experiment labels en kleuren (gecentreerd — niet meer per cel herdefinieren)
EXP_LABELS = {
    "a": "Baseline",
    "b": "RAG",
    "c": "Web search",
    "d": "Self-Refine",
}

palette = {"a": "#4C72B0", "b": "#55A868", "c": "#C44E52", "d": "#DD8452"}

EMBEDDING_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# Gewichten voor totaalscore (uit documentatie)
WEIGHTS = {
    "stability": 0.4,
    "cgr": 0.3,
    "hallucination_inverse": 0.15,
    "context_precision": 0.15,
}

# Uitgebreide gewichten (incl. judge kwaliteit en diversiteit)
WEIGHTS_EXTENDED = {
    "stability": 0.20,
    "judge_score": 0.25,      # genormaliseerd van 0-100 naar 0-1
    "cgr": 0.20,
    "hallucination_inverse": 0.10,
    "context_precision": 0.10,
    "diversity": 0.15,         # al 0-1 schaal
}

print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"OpenAI model: {MODEL}")
print(f"Experimenten: {EXPERIMENTS}")
print(f"Experiment labels: {EXP_LABELS}")
print(f"Gewichten origineel: {WEIGHTS}")
print(f"Gewichten uitgebreid: {WEIGHTS_EXTENDED}")

# ═══════════════════════════════════════════════════════════════════════
# AUTO-SAVE: elke plt.show() slaat de figuur ook op als PNG
# ═══════════════════════════════════════════════════════════════════════
FIG_DIR = os.path.join(PROJECT_ROOT, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

_original_plt_show = plt.show
_fig_counter = [0]

def _auto_save_and_show(*args, **kwargs):
    """Override plt.show(): sla alle open figuren op + toon ze."""
    import re
    for fig_num in plt.get_fignums():
        fig = plt.figure(fig_num)
        _fig_counter[0] += 1
        
        # Gebruik de figuur-titel als bestandsnaam
        title = ""
        for ax in fig.get_axes():
            t = ax.get_title()
            if t:
                title = t
                break
        if not title:
            # Fallback: gebruik suptitle
            title = fig._suptitle.get_text() if fig._suptitle else ""
        
        if title:
            # Sanitize: verwijder speciale tekens, beperk lengte
            safe_name = re.sub(r'[^a-zA-Z0-9_\- ]', '', title.split('\n')[0])
            safe_name = safe_name.strip().replace(' ', '_')[:60]
            filename = f"fig_{_fig_counter[0]:02d}_{safe_name}.png"
        else:
            filename = f"fig_{_fig_counter[0]:02d}.png"
        
        path = os.path.join(FIG_DIR, filename)
        fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
        print(f"  💾 {filename}")
    
    _original_plt_show(*args, **kwargs)

plt.show = _auto_save_and_show
print(f"\n✅ Auto-save actief: elke plt.show() slaat op naar {FIG_DIR}/")

### Resultaten laden
Alle run-bestanden (5 runs × 4 experimenten × 5 cases = 100 antwoorden) worden ingelezen vanuit de JSON-bestanden en samengevoegd tot één DataFrame `df_runs`. Experiment D (Self-Refine) heeft voorlopig 1 testrun.

In [0]:
# Importeer gedeelde step-extractie (voor backward compatibility met oude data)
from utils.steps import extract_steps, COT_STEPS, STEP_LABELS

rows = []
steps_data = []  # Per-stap data voor CoT-analyse

for exp in ["experiment_a", "experiment_b", "experiment_c", "experiment_d"]:

    exp_dir = os.path.join(RESULTS_DIR, exp)

    if not os.path.exists(exp_dir):
        continue

    for file in os.listdir(exp_dir):

        if not file.startswith("run_"):
            continue

        path = os.path.join(exp_dir, file)

        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        run = data["run"]
        experiment = data["experiment"]

        for case in data["cases"]:

            usage = case.get("usage", {})

            rows.append(
                {
                    "experiment": experiment,
                    "run": run,
                    "case_id": case["case_id"],
                    "answer": case["answer"],
                    "latency_seconds": case.get("latency_seconds"),
                    "prompt_tokens": usage.get("prompt_tokens"),
                    "completion_tokens": usage.get("completion_tokens"),
                    "total_tokens": usage.get("total_tokens"),
                }
            )

            # ── CoT-stappen laden ──
            # Gebruik opgeslagen steps als beschikbaar, anders extraheer on-the-fly
            steps = case.get("steps")
            if not steps and case.get("answer"):
                steps = extract_steps(case["answer"])  # backward compat

            if steps:
                for step_num, step_data in steps.items():
                    steps_data.append({
                        "experiment": experiment,
                        "run": run,
                        "case_id": case["case_id"],
                        "step_num": int(step_num),
                        "step_label": step_data.get("label", STEP_LABELS.get(step_num, f"Stap {step_num}")),
                        "step_text": step_data.get("text", ""),
                        "step_chars": len(step_data.get("text", "")),
                        "phase": "final",  # het uiteindelijke antwoord
                    })

            # Voor experiment D: ook de generate-fase stappen laden
            if experiment == "d" and case.get("phases"):
                gen_steps = case["phases"].get("generate", {}).get("steps")
                if not gen_steps:
                    gen_answer = case["phases"].get("generate", {}).get("answer", "")
                    if gen_answer:
                        gen_steps = extract_steps(gen_answer)
                if gen_steps:
                    for step_num, step_data in gen_steps.items():
                        steps_data.append({
                            "experiment": experiment,
                            "run": run,
                            "case_id": case["case_id"],
                            "step_num": int(step_num),
                            "step_label": step_data.get("label", STEP_LABELS.get(step_num, f"Stap {step_num}")),
                            "step_text": step_data.get("text", ""),
                            "step_chars": len(step_data.get("text", "")),
                            "phase": "generate",  # vóór Self-Refine
                        })

df_runs = pd.DataFrame(rows)
df_steps = pd.DataFrame(steps_data)

print(f"Geladen: {len(df_runs)} antwoorden, {len(df_steps)} individuele stappen")
print(f"Experimenten: {sorted(df_runs['experiment'].unique())}")
print(f"Stappen per experiment:")
for exp in EXPERIMENTS:
    n = len(df_steps[(df_steps['experiment'] == exp) & (df_steps['phase'] == 'final')])
    print(f"  {exp}: {n} stappen")
    if exp == 'd':
        n_gen = len(df_steps[(df_steps['experiment'] == 'd') & (df_steps['phase'] == 'generate')])
        print(f"  d (generate-fase): {n_gen} stappen")

df_runs.head()

## 4.1 Basisanalyse
> **Adresseert:** Stabiliteit + Efficiëntie — eerste verkenning van de data

Overzicht van de experimenten: aantal runs, cases, gemiddelde latency, tokenverbruik en stabiliteit per architectuur. De stabiliteit (S) wordt berekend als de gemiddelde pairwise cosine similarity tussen alle runs:

**S** = 2 / (R(R−1)) × Σ sim(Tᵢ, Tⱼ)

In [0]:
# Stabiliteit: mean pairwise cosine similarity per experiment/case

def mean_pairwise_cosine(texts, model):
    texts = [t.strip() for t in texts if isinstance(t, str) and t.strip()]
    if len(texts) < 2:
        return np.nan

    emb = model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
    sim = cosine_similarity(emb)
    iu = np.triu_indices_from(sim, k=1)
    return float(sim[iu].mean())

rows = []
for (experiment, case_id), g in df_runs.groupby(["experiment", "case_id"]):
    texts = g.sort_values("run")["answer"].fillna("").tolist()
    stability = mean_pairwise_cosine(texts, embedding_model)

    rows.append({
        "experiment": experiment,
        "case_id": case_id,
        "stability": stability,
    })

stability_by_case = pd.DataFrame(rows)

In [0]:
stability_summary = (
    stability_by_case.groupby("experiment")
    .agg(
        stability=("stability", "mean"),
        stability_std=("stability", "std"),
    )
    .reset_index()
)

stability_summary

In [0]:
overview = (
    df_runs.groupby("experiment")
    .agg(
        runs=("run", "nunique"),
        cases=("case_id", "nunique"),
        avg_latency_seconds=("latency_seconds", "mean"),
        std_latency_seconds=("latency_seconds", "std"),
        avg_prompt_tokens=("prompt_tokens", "mean"),
        avg_completion_tokens=("completion_tokens", "mean"),
        avg_total_tokens=("total_tokens", "mean"),
    )
    .reset_index()
)

overview = overview.merge(
    stability_summary[["experiment", "stability", "stability_std"]],
    on="experiment",
    how="left"
)

overview = overview.rename(
    columns={
        "stability": "avg_stability",
        "stability_std": "std_stability",
    }
)

# Rond af op 2 decimalen voor leesbaarheid
overview = overview.sort_values("experiment").round(2)

print(f"Overzicht: {len(EXPERIMENTS)} experimenten")
for exp in EXPERIMENTS:
    exp_data = overview[overview['experiment'] == exp]
    if not exp_data.empty:
        r = exp_data.iloc[0]
        stab = f"{r['avg_stability']:.3f}" if pd.notna(r['avg_stability']) else 'N/A (1 run)'
        print(f"  {exp.upper()}: {int(r['runs'])} runs × {int(r['cases'])} cases | "
              f"latency={r['avg_latency_seconds']:.1f}s | tokens={r['avg_total_tokens']:.0f} | "
              f"stabiliteit={stab}")

overview

## 4.2 Parameter sensitivity analyse
> **Adresseert:** *Zijn de bevindingen robuust over verschillende LLM-temperatuurinstellingen? Heeft temperatuur effect op scenario-diversiteit?*

Alle experimenten zijn tot nu toe gedraaid met identieke parameters (`temperature=0.8`, `top_p=1.0`). De vraag is of de ranking (A > B > C > D in pairwise, D > B > A > C op rubric) standhoudt bij andere temperatures.

**Aanpak:**
- Hergebruikt de **exacte messages** (incl. RAG/web context) uit de bestaande runs
- Varieert **alleen** `temperature` (0.3, 0.5, 0.8, 1.0) — `top_p` vast op 1.0
- 3 runs per configuratie, 2 representatieve cases
- Beoordeling: identieke strenge 0–100 judge

Dit isoleert het effect van temperature van retrieval-randomness.

**Analyses:**
| Cel | Onderwerp | Methode |
|-----|-----------|---------|
| 4.2 | Kwaliteit × temperatuur | Judge scores + ranking per temp |
| 4.2 viz | Sensitivity visualisatie | Lijnplots, heatmap, ranking-stabiliteit |
| **4.2.1** | **Scenario diversiteit × temperatuur** | **Embedding-diversiteit per temp, Spearman correlatie, stabiliteit-diversiteit trade-off** |

**Geschatte omvang:** \~200 API calls + embedding-berekeningen, \~12 minuten

**Data:** `hypertune_results.csv` (scores) + `hypertune_answers.json` (ruwe antwoorden voor diversiteitsanalyse)

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# 4.2 Parameter sensitivity: Hypertune uitvoeren
# ═══════════════════════════════════════════════════════════════════════
# Draait de sensitivity analyse over alle experimenten en parameterconfigs.
# Hergebruikt exacte messages uit bestaande runs, varieert alleen temperature.
#
# Output:
#   - data/results_data/hypertune_results.csv   (geaggregeerde scores)
#   - data/results_data/hypertune_answers.json   (ruwe antwoorden voor diversiteitsanalyse)

import importlib
import sys

# Importeer en draai de hypertune
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import Hypertune
importlib.reload(Hypertune)

df_hypertune = Hypertune.run_tuning()

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# 4.2 Sensitivity analyse: visualisatie
# ═══════════════════════════════════════════════════════════════════════

# Laad resultaten (als df_hypertune niet in memory is)
try:
    df_ht = df_hypertune.copy()
except NameError:
    ht_csv = os.path.join(PROJECT_ROOT, "data", "results_data", "hypertune_results.csv")
    df_ht = pd.read_csv(ht_csv)
    print(f"Geladen uit CSV: {len(df_ht)} trials")

temps = sorted(df_ht["temperature"].unique())

# ─── Plot 1: 2×2 overzicht ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1a. Lijnplot: gem. judge score per temp
ax = axes[0, 0]
for exp in EXPERIMENTS:
    sub = df_ht[df_ht["experiment"] == exp]
    means = sub.groupby("temperature")["judge_total"].mean()
    stds = sub.groupby("temperature")["judge_total"].std()
    ax.plot(means.index, means.values, marker="o", label=f"{exp.upper()} ({EXP_LABELS[exp]})",
            color=palette[exp], linewidth=2)
    ax.fill_between(means.index, means - stds, means + stds, alpha=0.15, color=palette[exp])
ax.set_xlabel("Temperature")
ax.set_ylabel("Gemiddelde judge score (0–100)")
ax.set_title("Judge score vs temperature per experiment")
ax.legend(fontsize=9)
ax.set_xticks(temps)
ax.grid(True, alpha=0.3)

# 1b. Lijnplot: stabiliteit per temp
ax = axes[0, 1]
for exp in EXPERIMENTS:
    sub = df_ht[df_ht["experiment"] == exp]
    means = sub.groupby("temperature")["stability"].mean()
    ax.plot(means.index, means.values, marker="s", label=f"{exp.upper()}",
            color=palette[exp], linewidth=2)
ax.set_xlabel("Temperature")
ax.set_ylabel("Stabiliteit (cosine similarity)")
ax.set_title("Stabiliteit vs temperature per experiment")
ax.legend(fontsize=9)
ax.set_xticks(temps)
ax.grid(True, alpha=0.3)

# 1c. Ranking per temperature
ax = axes[1, 0]
for exp in EXPERIMENTS:
    ranks_at_temp = []
    for temp in temps:
        sub = df_ht[df_ht["temperature"] == temp].groupby("experiment")["judge_total"].mean()
        rank = sub.rank(ascending=False)
        ranks_at_temp.append(rank.get(exp, np.nan))
    ax.plot(temps, ranks_at_temp, marker="D", label=f"{exp.upper()}",
            color=palette[exp], linewidth=2)
ax.set_xlabel("Temperature")
ax.set_ylabel("Rank (1=best)")
ax.set_title("Ranking per temperature\n(verandert de volgorde?)")
ax.invert_yaxis()
ax.set_yticks([1, 2, 3, 4])
ax.set_xticks(temps)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 1d. Heatmap: experiment × temperature → judge score
ax = axes[1, 1]
heatmap_data = df_ht.pivot_table(
    index="experiment", columns="temperature",
    values="judge_total", aggfunc="mean"
).reindex(EXPERIMENTS)
sns.heatmap(heatmap_data, annot=True, fmt=".1f", cmap="RdYlGn",
            vmin=heatmap_data.values.min() - 5, vmax=heatmap_data.values.max() + 5,
            linewidths=0.5, ax=ax)
ax.set_title("Judge score heatmap\n(experiment × temperature)")
ax.set_ylabel("Experiment")
ax.set_xlabel("Temperature")

plt.suptitle("§4.2 Temperature Sensitivity Analyse", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# ─── Robuustheid samenvatting ────────────────────────────────────────────
print("\n" + "=" * 70)
print("ROBUUSTHEID VAN DE RANKING")
print("=" * 70)

# Bereken ranking per temperature
ranking_results = []
for temp in temps:
    sub = df_ht[df_ht["temperature"] == temp]
    means = sub.groupby("experiment")["judge_total"].mean()
    ranking = means.rank(ascending=False)
    ranking_results.append({
        "temperature": temp,
        **{f"rank_{e}": ranking.get(e, np.nan) for e in EXPERIMENTS},
        "ranking": " > ".join(f"{e.upper()}" for e in means.sort_values(ascending=False).index)
    })

df_rankings = pd.DataFrame(ranking_results)
print("\nRanking per temperature:")
for _, row in df_rankings.iterrows():
    print(f"  temp={row['temperature']:.1f}: {row['ranking']}")

# Consistentie
ranking_counts = df_rankings["ranking"].value_counts()
print(f"\nMeest voorkomende ranking: {ranking_counts.index[0]} ({ranking_counts.iloc[0]}/{len(df_rankings)} temps)")
print(f"Unieke rankings: {len(ranking_counts)}")

# Gemiddelde rank per experiment over alle temps
print("\nGemiddelde rank over alle temperatures:")
for exp in EXPERIMENTS:
    col = f"rank_{exp}"
    if col in df_rankings.columns:
        avg_r = df_rankings[col].mean()
        std_r = df_rankings[col].std()
        print(f"  {exp.upper()} ({EXP_LABELS[exp]}): gem.rank = {avg_r:.2f} (±{std_r:.2f})")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# 4.2.1 Scenario diversiteit vs temperatuur
# ═══════════════════════════════════════════════════════════════════════
# Berekent de intra-advies scenario diversiteit (embedding-based) voor
# elke temperatuurinstelling uit de hypertune data.
#
# Vereist: data/results_data/hypertune_answers.json
#   (gegenereerd door Hypertune.py na uitbreiding met antwoordopslag)
# ═══════════════════════════════════════════════════════════════════════

import json
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, kruskal
from sklearn.metrics.pairwise import cosine_similarity

# ── Helper: stap-tekst extraheren (nodig vóór cel 25 waar dit normaal staat) ──
def _step_to_text(step_val):
    """Haal tekst op uit extract_steps(), ongeacht dict of string formaat."""
    if isinstance(step_val, dict):
        return step_val.get("text", "")
    return str(step_val)

# ── Inline scenario-extractie ──
def extract_scenarios_from_answer(answer_text):
    """Extraheer individuele scenario's uit stap 5 (Scenario-analyse) van een antwoord."""
    steps = extract_steps(answer_text)
    # Stap 5 = Scenario-analyse
    scenario_text = ""
    if "5" in steps:
        scenario_text = _step_to_text(steps["5"])
    if not scenario_text or len(scenario_text.strip()) < 30:
        return []
    
    # Split op scenario-headers: "Scenario 1:", "**Scenario 1:**", "1.", etc.
    patterns = [
        r'\*\*Scenario\s+\d+[^*]*\*\*',  # **Scenario 1: Titel**
        r'Scenario\s+\d+\s*[:.]',          # Scenario 1:
        r'(?:^|\n)\s*\d+\.\s+\*\*',       # 1. **Titel**
    ]
    
    for pat in patterns:
        parts = re.split(pat, scenario_text)
        # Eerste deel is vaak een inleiding, skip als kort
        scenarios = [p.strip() for p in parts if len(p.strip()) > 50]
        if len(scenarios) >= 2:
            return scenarios[:3]  # max 3 scenario's
    
    # Fallback: split op dubbele newlines
    paragraphs = [p.strip() for p in scenario_text.split("\n\n") if len(p.strip()) > 50]
    if len(paragraphs) >= 2:
        return paragraphs[:3]
    
    return [scenario_text]  # geen split mogelijk

# ── Laad hypertune antwoorden ──
HT_ANSWERS_PATH = os.path.join(PROJECT_ROOT, "data", "results_data", "hypertune_answers.json")

if not os.path.exists(HT_ANSWERS_PATH):
    print("⚠️  hypertune_answers.json niet gevonden.")
    print("   Draai eerst cel '4.2 Hypertune uitvoeren' opnieuw")
    print("   (Hypertune.py slaat nu ook ruwe antwoorden op).")
else:
    with open(HT_ANSWERS_PATH, encoding="utf-8") as f:
        ht_answers = json.load(f)
    print(f"Geladen: {len(ht_answers)} trial-groepen uit hypertune_answers.json")

    ht_div_rows = []
    for key, trial in ht_answers.items():
        exp = trial["experiment"]
        case_id = trial["case_id"]
        temp = trial["temperature"]
        answers = trial["answers"]

        for run_idx, answer in enumerate(answers):
            scenarios = extract_scenarios_from_answer(answer)
            n_scen = len(scenarios)

            if n_scen >= 2:
                embs = embedding_model.encode(scenarios[:3], normalize_embeddings=True)
                sim = cosine_similarity(embs)
                iu = np.triu_indices_from(sim, k=1)
                diversity = float(1.0 - sim[iu].mean())
            else:
                diversity = np.nan

            ht_div_rows.append({
                "experiment": exp,
                "case_id": case_id,
                "temperature": temp,
                "run": run_idx + 1,
                "n_scenarios": n_scen,
                "diversity": round(diversity, 4) if not np.isnan(diversity) else np.nan,
            })

    df_ht_div = pd.DataFrame(ht_div_rows)
    df_ht_valid = df_ht_div.dropna(subset=["diversity"])

    print(f"\nTotaal: {len(df_ht_div)} antwoorden, {len(df_ht_valid)} met ≥2 scenario's")

    # ── Samenvatting per temperatuur ──
    print("\n" + "=" * 70)
    print("SCENARIO DIVERSITEIT PER TEMPERATUUR")
    print("=" * 70)
    temp_summary = df_ht_valid.groupby("temperature")["diversity"].agg(
        ["count", "mean", "std", "min", "median", "max"]
    ).round(4)
    display(temp_summary)

    # ── Correlatie temperatuur ↔ diversiteit ──
    rho, p_val = spearmanr(df_ht_valid["temperature"], df_ht_valid["diversity"])
    print(f"\nSpearman correlatie temperatuur ↔ diversiteit: ρ={rho:.3f}, p={p_val:.4f}")
    print(f"→ {'Significante' if p_val < 0.05 else 'Geen significante'} monotone relatie")

    # Kruskal-Wallis: verschilt diversiteit significant per temperatuur?
    temps_sorted = sorted(df_ht_valid["temperature"].unique())
    groups = [df_ht_valid[df_ht_valid["temperature"] == t]["diversity"].values for t in temps_sorted]
    groups_valid = [g for g in groups if len(g) >= 2]
    if len(groups_valid) >= 2:
        h_stat, h_p = kruskal(*groups_valid)
        print(f"Kruskal-Wallis (diversiteit ~ temperatuur): H={h_stat:.3f}, p={h_p:.4f}")

    # ── Visualisatie ──
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # 1. Boxplot: diversiteit per temperatuur
    ax = axes[0, 0]
    sns.boxplot(data=df_ht_valid, x="temperature", y="diversity", ax=ax,
                color="#4C72B0", showmeans=True,
                meanprops={"marker": "D", "markerfacecolor": "white", "markersize": 8})
    for i, t in enumerate(temps_sorted):
        vals = df_ht_valid[df_ht_valid["temperature"] == t]["diversity"]
        if len(vals) > 0:
            ax.text(i, vals.mean() + 0.008, f"μ={vals.mean():.3f}", ha="center", fontsize=9, fontweight="bold")
    ax.set_title("Scenario diversiteit per temperatuur")
    ax.set_ylabel("Diversiteit (1 − gem. cosine sim)")
    ax.set_xlabel("Temperature")
    ax.grid(True, alpha=0.3)

    # 2. Per experiment: diversiteit vs temperatuur
    ax = axes[0, 1]
    for exp in EXPERIMENTS:
        sub = df_ht_valid[df_ht_valid["experiment"] == exp]
        means = sub.groupby("temperature")["diversity"].mean()
        stds = sub.groupby("temperature")["diversity"].std()
        ax.plot(means.index, means.values, "o-", color=palette[exp],
                label=f"{exp.upper()} ({EXP_LABELS[exp]})", linewidth=2)
        ax.fill_between(means.index, means - stds, means + stds, alpha=0.15, color=palette[exp])
    ax.set_xlabel("Temperature")
    ax.set_ylabel("Gem. diversiteit")
    ax.set_title("Diversiteit per experiment × temperatuur")
    ax.legend(fontsize=9)
    ax.set_xticks(temps_sorted)
    ax.grid(True, alpha=0.3)

    # 3. Heatmap: experiment × temperatuur → diversiteit
    ax = axes[1, 0]
    hm_data = df_ht_valid.pivot_table(index="experiment", columns="temperature",
                                       values="diversity", aggfunc="mean")
    hm_data = hm_data.reindex(EXPERIMENTS)
    sns.heatmap(hm_data, annot=True, fmt=".3f", cmap="RdYlGn",
                vmin=0, vmax=hm_data.values.max() * 1.2,
                linewidths=0.5, ax=ax)
    ax.set_title("Diversiteit heatmap\n(experiment × temperature)")
    ax.set_ylabel("Experiment")
    ax.set_xlabel("Temperature")

    # 4. Scatter: stabiliteit vs diversiteit (trade-off?)
    ax = axes[1, 1]
    # Merge met hypertune CSV voor stabiliteitsdata
    ht_csv = os.path.join(PROJECT_ROOT, "data", "results_data", "hypertune_results.csv")
    df_ht_csv = pd.read_csv(ht_csv)
    ht_merged = df_ht_valid.groupby(["experiment", "case_id", "temperature"])["diversity"].mean().reset_index()
    ht_merged = ht_merged.merge(df_ht_csv[["experiment", "case_id", "temperature", "stability"]],
                                 on=["experiment", "case_id", "temperature"], how="left")
    for exp in EXPERIMENTS:
        sub = ht_merged[ht_merged["experiment"] == exp]
        ax.scatter(sub["stability"], sub["diversity"], color=palette[exp],
                   label=f"{exp.upper()}", s=80, alpha=0.7, edgecolors="white")
    if len(ht_merged.dropna(subset=["stability", "diversity"])) >= 4:
        rho_sd, p_sd = spearmanr(ht_merged["stability"].dropna(), ht_merged["diversity"].dropna())
        ax.set_xlabel(f"Stabiliteit (cosine sim)\nρ={rho_sd:.3f}, p={p_sd:.3f}")
    else:
        ax.set_xlabel("Stabiliteit (cosine sim)")
    ax.set_ylabel("Scenario diversiteit")
    ax.set_title("Trade-off: stabiliteit vs diversiteit\n(hogere temp → minder stabiel maar diverser?)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    plt.suptitle("§4.2.1 Temperatuur × scenario diversiteit (hypertune data)",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

    # ── Conclusie ──
    print("\n" + "=" * 70)
    print("CONCLUSIE: TEMPERATUUR → SCENARIO DIVERSITEIT")
    print("=" * 70)
    if p_val < 0.05:
        direction = "positieve" if rho > 0 else "negatieve"
        print(f"✅ Significante {direction} correlatie (ρ={rho:.3f}, p={p_val:.4f})")
        print(f"   → Hogere temperatuur leidt tot {'meer' if rho > 0 else 'minder'} diverse scenario's")
    else:
        print(f"❌ Geen significante correlatie (ρ={rho:.3f}, p={p_val:.4f})")
        print("   → Temperatuur heeft geen meetbaar effect op scenario diversiteit")
        print("   → De architectuur-keuze (A/B/C/D) is belangrijker dan de temperatuur")

### 4.1.3 Invloed van temperatuur op consistentie (DV1)
> **Deelvraag:** Worden de antwoorden minder consistent wanneer de temperatuur stijgt?

Naast kwaliteit en diversiteit is onderzocht in hoeverre temperatuur invloed heeft op de **consistentie** van de gegenereerde scenario's. Consistentie is gemeten met cosine similarity tussen meerdere runs van hetzelfde experiment. Per temperatuurinstelling zijn 8 metingen beschikbaar (4 architecturen × 2 cases).

| Temperatuur | Gem. Cosine Similarity | Std |
|---|---|---|
| 0.3 | 0.9652 | 0.0113 |
| 0.5 | 0.9647 | 0.0117 |
| 0.8 | 0.9606 | 0.0102 |
| 1.0 | 0.9592 | 0.0069 |

Figuur 4.1.3 laat zien dat de gemiddelde cosine similarity monotoon afneemt naarmate de temperatuur stijgt, van 0.965 bij t=0.3 naar 0.959 bij t=1.0 (Δ = −0.006). Hoewel het absolute verschil beperkt is, bevestigt de monotone daling dat hogere temperatuurinstellingen consequent leiden tot een lagere reproduceerbaarheid van de gegenereerde output.

In [0]:
from IPython.display import Image, display

# ── Figuur laden ──────────────────────────────────────────────────────────────
fig_path = os.path.join(PROJECT_ROOT, "figures", "fig_4_1_3_temperatuur_consistentie.png")
display(Image(filename=fig_path, width=700))

# ── Samenvattingstabel ────────────────────────────────────────────────────────
ht_df = pd.read_csv(os.path.join(RESULTS_DIR, "hypertune_results.csv"))
temp_agg = (
    ht_df.groupby("temperature")["stability"]
    .agg(mean="mean", std="std", n="count")
    .round(4)
    .reset_index()
)
temp_agg.columns = ["Temperatuur", "Gem. Cosine Similarity", "Std", "n"]
display(temp_agg)

delta = temp_agg["Gem. Cosine Similarity"].iloc[-1] - temp_agg["Gem. Cosine Similarity"].iloc[0]
print(f"Δ (t=1.0 − t=0.3) = {delta:+.4f}")
print(f"Hoogste consistentie bij t={temp_agg.loc[temp_agg['Gem. Cosine Similarity'].idxmax(), 'Temperatuur']}")
print(f"Laagste  consistentie bij t={temp_agg.loc[temp_agg['Gem. Cosine Similarity'].idxmin(), 'Temperatuur']}")

## 4.3 Kwaliteit van de gegenereerde adviezen (LLM-as-judge)
> **Adresseert:** *Wat is de inhoudelijke kwaliteit van de strategische adviezen per architectuur?*

De kwaliteit wordt beoordeeld met een **LLM-as-a-judge** benadering (gpt-4o-mini, temperature=0) op vijf criteria:
1. **Probleemdefinitie** — Eén concreet, afgebakend strategisch probleem?
2. **Analyse** — Externe én interne analyse met verbanden?
3. **Scenario’s** — Onderscheidend, plausibel, relevant?
4. **Advies** — Logisch voortbouwend, concreet, uitvoerbaar?
5. **CoT samenhang** — Logische doorvertaling tussen stappen?

Per experiment/case wordt run 1 beoordeeld om API-kosten te beperken (15 beoordelingen totaal).

In [0]:
# ── JSON parser voor LLM-judge output ──
import json, re

def parse_judge_json(text: str) -> dict:
    """Parse JSON uit LLM-response, inclusief markdown code blocks."""
    if not text or not text.strip():
        return {}
    
    text = text.strip()
    
    # Strategie 1: Verwijder markdown code block markers
    if "```" in text:
        # Match ```json ... ``` of ``` ... ```
        match = re.search(r'```(?:json)?\s*\n?(.*?)\n?```', text, re.DOTALL)
        if match:
            text = match.group(1).strip()
    
    # Strategie 2: Zoek eerste { ... } blok
    brace_start = text.find('{')
    brace_end = text.rfind('}')
    if brace_start != -1 and brace_end > brace_start:
        text = text[brace_start:brace_end + 1]
    
    # Parse JSON
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # Fallback: probeer common issues te fixen
        try:
            # Verwijder trailing comma's voor sluitende } of ]
            fixed = re.sub(r',\s*([}\]])', r'\1', text)
            return json.loads(fixed)
        except json.JSONDecodeError:
            print(f"  ⚠️ JSON parse error: {text[:100]}...")
            return {}

print("✅ parse_judge_json() gedefinieerd")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Strenge Judge: 0–100 per criterium
# ═══════════════════════════════════════════════════════════════════════

SCORE_KEYS = ["probleemdefinitie", "analyse", "scenarios", "advies", "cot_samenhang"]

RUBRIC_PROMPT = """Je bent een ZEER STRENGE academische beoordelaar van strategische adviezen voor restaurants.
Je beoordeelt als een examinator die zelden hoge cijfers geeft.

Beoordeel het advies op vijf criteria. Geef per criterium een score van 0 tot 100.

CRITERIATABEL — VOLG DEZE STRIKT:

| Score     | Niveau          | Beschrijving                                                                        |
|-----------|-----------------|-------------------------------------------------------------------------------------|
|  0–15     | Onvoldoende     | Criterium AFWEZIG of feitelijk onjuist                                              |
| 16–30     | Zwak            | Oppervlakkig, vaag, geen onderbouwing, alleen algemeenheden                         |
| 31–45     | Matig           | Aanwezig maar GENERIEK — zou op elk restaurant kunnen slaan                        |
| 46–58     | Voldoende       | Enige specificiteit, verwijst naar de case maar mist diepgang                       |
| 59–70     | Ruim voldoende  | Concreet en specifiek, verwijst naar data uit de case, redelijke onderbouwing       |
| 71–80     | Goed            | Specifiek, onderbouwd met CONCRETE CIJFERS, logische verbanden uitgewerkt           |
| 81–90     | Zeer goed       | Diepgaand, sterk geïntegreerd, concrete acties met KPI's en tijdlijn              |
| 91–100    | Uitmuntend      | UITZONDERLIJK — alleen als NIETS ontbreekt en alles meetbaar is onderbouwd        |

STRENGE REGELS:
- Een advies dat GENERIEK is (toepasbaar op elk restaurant) scoort MAXIMAAL 45 per criterium.
- Pas als het advies SPECIFIEKE data uit de case noemt (prijzen, percentages, concurrenten),
  mag de score boven 58 komen.
- Scores boven 80 vereisen CONCRETE cijfers, KPI's, tijdlijnen en risicoanalyse.
- Score 90+ is BIJNA ONMOGELIJK — alleen bij echt foutloos werk.
- Gebruik GEEN ronde getallen. Gebruik bijv. 37, 52, 64, 73.
- Als je twijfelt tussen twee scores, kies de LAGERE.
- De GEMIDDELDE score over alle adviezen hoort rond 45–60 te liggen, niet rond 70–80.

CRITERIA:

1. PROBLEEMDEFINITIE (0–100):
   Hoe scherp, afgebakend en onderbouwd is het strategisch probleem gedefinieerd?
   Let op: één helder probleem, kwantificering met data UIT DE CASE, oorzaak-gevolg.
   - ≤45: Probleem benoemd maar generiek ("omzet daalt" zonder specifieke cijfers)
   - 46–58: Probleem met enige kwantificering maar mist oorzaakanalyse
   - 59–70: Scherp afgebakend met specifieke cijfers uit de case
   - 71–80: Volledig gekwantificeerd met oorzaak-gevolg-keten en urgentiebepaling
   - ≥81: Messcherp, meerdere databronnen, prioritering van deelproblemen

2. ANALYSE (0–100):
   Hoe diep en verbindend is de externe en interne analyse?
   Let op: concrete datapunten, eigen conclusies (niet alleen beschrijving), verbanden.
   - ≤45: Beschrijvend, somt feiten op zonder verbanden te leggen
   - 46–58: Enige analyse maar conclusies zijn oppervlakkig
   - 59–70: Diepgaand met concrete data én eigen conclusies
   - 71–80: Synthese van interne+externe factoren met verrassende inzichten
   - ≥81: Volledig geïntegreerde analyse, SWOT/PESTEL met doorvertaling

3. SCENARIOS (0–100):
   Hoe onderscheidend, plausibel en PASSEND bij de restaurantcontext zijn de scenario's?
   Let op: ≥2 verschillende scenario's, concrete implicaties, passendheid bij het TYPE restaurant.
   - ≤45: Scenario's zijn generiek of onrealistisch voor dit type restaurant
   - 46–58: Scenario's aanwezig maar niet onderscheidend of slecht passend
   - 59–70: Scenario's zijn plausibel en passend bij de context
   - 71–80: Duidelijk verschillende én contextspecifieke scenario's met doorgerekende impact
   - ≥81: Scenario's met kwantitatieve doorrekening, risicobepaling en strategische trade-offs

4. ADVIES (0–100):
   Hoe concreet, uitvoerbaar en onderbouwd zijn de aanbevelingen?
   Let op: link met analyse, concrete acties, prioritering, risico's, KPI's.
   - ≤45: Generieke aanbevelingen ("verbeter de marketingstrategie") zonder concretisering
   - 46–58: Enigszins concreet maar mist actieplan of KPI's
   - 59–70: Concreet advies met prioritering en onderbouwing
   - 71–80: Volledig actieplan met tijdlijn, verantwoordelijkheden en meetbare KPI's
   - ≥81: Advies met implementatiefasen, risicomitigatie, budget en succescriteria

5. COT_SAMENHANG (0–100):
   Hoe sterk is de logische doorvertaling van begin tot eind?
   Let op: analyse verwijst naar probleem, scenario's bouwen op analyse,
   advies verwijst naar scenario's, geen logische sprongen.
   - ≤45: Secties staan los van elkaar, geen doorvertaling
   - 46–58: Enige samenhang maar met logische hiaten of sprongen
   - 59–70: Sterke doorvertaling, inzichten worden meegenomen
   - 71–80: Rode draad van probleem tot implementatie, elke stap bouwt voort
   - ≥81: Perfecte integratie, cross-referenties tussen secties

ADVIES OM TE BEOORDELEN:
{text}

Geef ALLEEN valid JSON terug:
{{
  "probleemdefinitie": {{"score": <0-100>, "toelichting": "<1 zin>"}},
  "analyse":           {{"score": <0-100>, "toelichting": "<1 zin>"}},
  "scenarios":         {{"score": <0-100>, "toelichting": "<1 zin>"}},
  "advies":            {{"score": <0-100>, "toelichting": "<1 zin>"}},
  "cot_samenhang":     {{"score": <0-100>, "toelichting": "<1 zin>"}}
}}
"""

def judge_answer_rubric_100(text: str) -> tuple:
    """
    Beoordeel een advies: 5 criteria × 0–100.
    Totaal = gemiddelde van de 5 criteria (0–100).
    Returns: (scores_dict, details_dict)
    """
    prompt = RUBRIC_PROMPT.format(text=text[:6000])

    try:
        resp = openai_client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=800,
        )
        parsed = parse_judge_json(resp.choices[0].message.content or "")
    except Exception as e:
        print(f"FOUT: {e}")
        parsed = {}

    scores = {}
    details = {}
    for crit in SCORE_KEYS:
        crit_data = parsed.get(crit, {})
        if isinstance(crit_data, dict):
            score = crit_data.get("score", 50)
            note = crit_data.get("toelichting", "")
        elif isinstance(crit_data, (int, float)):
            score = crit_data
            note = ""
        else:
            score = 50
            note = "parse error"
        scores[crit] = max(0, min(100, int(score)))  # clamp 0–100
        details[crit] = {"score": scores[crit], "toelichting": note}

    return scores, details

judge_answer_rubric = judge_answer_rubric_100

print("Strenge Judge klaar: 0–100 per criterium.")
print(f"Criteria: {SCORE_KEYS}")
print(f"\nCriteriatabel (STRENG):")
print(f"   0–15  Onvoldoende  |  16–30  Zwak       |  31–45  Matig")
print(f"  46–58  Voldoende    |  59–70  Ruim vold. |  71–80  Goed")
print(f"  81–90  Zeer goed    |  91–100 Uitmuntend (bijna onmogelijk)")
print(f"\nVerwacht gemiddelde: 45–60 (niet 70–80 zoals voorheen)")

In [0]:
# ── Beoordeling uitvoeren: 0–100 per criterium ──

df_to_judge = df_runs[df_runs["run"] == 1].copy()

print(f"Te beoordelen: {len(df_to_judge)} antwoorden (0–100 per criterium)")
print("Dit kan enkele minuten duren...\n")

rubric_rows = []
rubric_details = []

for idx, row in df_to_judge.iterrows():
    exp = row["experiment"]
    case = row["case_id"]
    print(f"  {exp} / {case}...", end=" ")

    scores, details = judge_answer_rubric_100(row["answer"])
    scores["experiment"] = exp
    scores["case_id"] = case
    # Totaal = gemiddelde van 5 criteria (0–100)
    scores["totaal"] = round(sum(v for k, v in scores.items() if k in SCORE_KEYS) / len(SCORE_KEYS), 1)
    rubric_rows.append(scores)
    rubric_details.append({"experiment": exp, "case_id": case, **details})

    score_str = " | ".join(f"{k[:6]}={scores[k]}" for k in SCORE_KEYS)
    print(f"{score_str} | GEM={scores['totaal']}/100")

df_judge = pd.DataFrame(rubric_rows)
df_judge_details = pd.DataFrame(rubric_details)

print(f"\nKlaar. {len(df_judge)} beoordelingen.")
print(f"\nScore verdeling (0–100 per criterium, gemiddelde 0–100):")
for crit in SCORE_KEYS:
    vals = df_judge[crit]
    print(f"  {crit:20s}: min={vals.min():3.0f}, max={vals.max():3.0f}, mean={vals.mean():.1f}, std={vals.std():.1f}")
print(f"  {'GEMIDDELD':20s}: min={df_judge['totaal'].min():.1f}, max={df_judge['totaal'].max():.1f}, "
      f"mean={df_judge['totaal'].mean():.1f}, std={df_judge['totaal'].std():.1f}")

df_judge

In [0]:
# ── Visualisatie: 0–100 per criterium ──

# 1. Barplot scores per criterium per experiment (0–100 schaal)
judge_melted = df_judge.melt(
    id_vars=["experiment", "case_id"],
    value_vars=SCORE_KEYS,
    var_name="criterium",
    value_name="score"
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=judge_melted, x="criterium", y="score", hue="experiment",
            order=SCORE_KEYS, hue_order=EXPERIMENTS, ax=ax)
ax.set_ylim(0, 105)
ax.set_title("Score per criterium per experiment (0–100)")
ax.set_ylabel("Score (0–100)")
ax.set_xlabel("Criterium")
ax.axhline(y=55, color="gray", linestyle="--", alpha=0.5, label="voldoende (55)")
ax.axhline(y=75, color="green", linestyle="--", alpha=0.3, label="goed (75)")
plt.xticks(rotation=15)
plt.legend(title="Experiment")
plt.tight_layout()
plt.show()

# 2. Totaalscore per experiment (gemiddelde 0–100)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

totaal_by_exp = df_judge.groupby("experiment")["totaal"].agg(["mean", "std"]).reset_index()
totaal_by_exp = totaal_by_exp.sort_values("experiment")

sns.barplot(data=df_judge, x="experiment", y="totaal", hue="experiment",
            order=EXPERIMENTS, hue_order=EXPERIMENTS,
            palette={"a": "#4C72B0", "b": "#55A868", "c": "#C44E52", "d": "#8172B2"},
            legend=False, ax=axes[0])
axes[0].set_title("Gemiddelde totaalscore per experiment (0–100)")
axes[0].set_ylabel("Score (0–100)")
axes[0].set_ylim(0, 105)
for i, row in totaal_by_exp.iterrows():
    idx = EXPERIMENTS.index(row["experiment"])
    axes[0].text(idx, row["mean"] + 1.5, f"{row['mean']:.1f}", ha="center", fontweight="bold")

# 3. Heatmap: score per criterium per experiment
heatmap_data = df_judge.groupby("experiment")[SCORE_KEYS].mean()
heatmap_data = heatmap_data.reindex(EXPERIMENTS)
sns.heatmap(heatmap_data.T, annot=True, fmt=".1f", cmap="RdYlGn", vmin=0, vmax=100,
            linewidths=0.5, ax=axes[1])
axes[1].set_title("Gemiddelde score per criterium per experiment (0–100)")
axes[1].set_ylabel("Criterium")
axes[1].set_xlabel("Experiment")

plt.tight_layout()
plt.show()

# 4. Toelichtingen tonen
print("\n=== Toelichtingen per experiment/case ===")
for _, row in df_judge_details.iterrows():
    exp, case = row["experiment"], row["case_id"]
    notes = []
    for crit in SCORE_KEYS:
        d = row.get(crit, {})
        if isinstance(d, dict) and d.get("toelichting"):
            notes.append(f"{crit[:8]}({d['score']}): {d['toelichting']}")
    if notes:
        print(f"  {exp}/{case}: {' | '.join(notes)}")

# Samenvattingstabel
judge_summary = df_judge.groupby("experiment")[SCORE_KEYS + ["totaal"]].mean().round(1)
judge_summary = judge_summary.reindex(EXPERIMENTS)
print("\nGemiddelde scores per experiment (0–100 per criterium):")
judge_summary

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# DEEL B: Pairwise ranking met positiebias-correctie
# ═══════════════════════════════════════════════════════════════════════
# Per case worden alle experimenten naast elkaar gelegd in WILLEKEURIGE
# volgorde. Dit wordt N_PERMS keer herhaald met verschillende volgordes
# om positiebias te elimineren. Rankings worden gemiddeld over permutaties.

import random

N_PERMS = 4  # aantal random volgordes per case

EXP_LABELS = {
    "a": "Baseline",
    "b": "RAG",
    "c": "Web search",
    "d": "Self-Refine",
}

def _build_pairwise_prompt(texts: dict, exp_order: list) -> str:
    """Bouw pairwise ranking prompt met OPGEGEVEN volgorde van experimenten."""
    n = len(texts)
    rank_range = f"1-{n}"

    # Bouw adviesblokken in de opgegeven volgorde
    advice_blocks = []
    for exp in exp_order:
        label = EXP_LABELS.get(exp, exp.upper())
        advice_blocks.append(f"ADVIES {exp.upper()} ({label}):\n{texts[exp]}")

    ranking_template = ", ".join(
        f'"{exp}": <{rank_range}>' for exp in sorted(texts.keys())
    )

    criteria_json = []
    for crit in SCORE_KEYS:
        criteria_json.append(
            f'  "{crit}": {{"ranking": {{{ranking_template}}}, "toelichting": "..."}}'
        )

    return (
        f"Je vergelijkt {n} strategische adviezen voor HETZELFDE restaurant.\n"
        "Beoordeel welk advies het BESTE is per criterium.\n\n"
        "GEBRUIK DEZE CRITERIA:\n"
        "1. probleemdefinitie: Welk advies formuleert het scherpste, best afgebakende probleem?\n"
        "2. analyse: Welk advies heeft de diepste analyse met de meeste verbanden?\n"
        "3. scenarios: Welk advies heeft de meest plausibele en onderscheidende scenario's?\n"
        "4. advies: Welk advies geeft de meest concrete, uitvoerbare aanbevelingen?\n"
        "5. cot_samenhang: Welk advies heeft de sterkste logische doorvertaling van begin tot eind?\n\n"
        f"RANK per criterium van 1 (beste) tot {n} (slechtste).\n"
        "Bij gelijkwaardig: geef dezelfde rank.\n"
        "Geef per criterium een korte toelichting (max 1 zin).\n\n"
        + "\n\n".join(advice_blocks)
        + "\n\nGeef ALLEEN valid JSON terug:\n{\n"
        + ",\n".join(criteria_json)
        + "\n}\n"
    )

# ── Pairwise ranking met permutaties ──
cases = sorted(df_runs["case_id"].unique())
run1 = df_runs[df_runs["run"] == 1]
available_exps = sorted(run1["experiment"].unique())

print(f"Pairwise ranking voor {len(cases)} cases × {len(available_exps)} experimenten")
print(f"Positiebias-correctie: {N_PERMS} random permutaties per case")
print(f"Totaal API-calls: {len(cases) * N_PERMS}\n")

# Verzamel alle ranks per permutatie
all_perm_rows = []   # (case_id, criterium, experiment, rank, perm_idx)
pairwise_notes = []

for case_id in cases:
    texts = {}
    for exp in available_exps:
        mask = (run1["experiment"] == exp) & (run1["case_id"] == case_id)
        ans = run1.loc[mask, "answer"].values
        if len(ans) > 0 and ans[0]:
            texts[exp] = ans[0][:4000]

    if len(texts) < 2:
        print(f"  {case_id}: overgeslagen (< 2 adviezen)")
        continue

    exp_list = list(texts.keys())

    for perm_idx in range(N_PERMS):
        shuffled = exp_list.copy()
        random.shuffle(shuffled)
        order_str = ",".join(e.upper() for e in shuffled)

        prompt = _build_pairwise_prompt(texts, shuffled)

        print(f"  {case_id} perm {perm_idx+1}/{N_PERMS} (volgorde: {order_str})...", end=" ")
        try:
            resp = openai_client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0, max_tokens=1000,
            )
            parsed = parse_judge_json(resp.choices[0].message.content or "")
        except Exception as e:
            print(f"FOUT: {e}")
            parsed = {}

        for crit in SCORE_KEYS:
            crit_data = parsed.get(crit, {})
            ranking = crit_data.get("ranking", {})
            note = crit_data.get("toelichting", "")
            for exp in available_exps:
                rank = ranking.get(exp, len(available_exps) // 2 + 1)
                all_perm_rows.append({
                    "case_id": case_id,
                    "criterium": crit,
                    "experiment": exp,
                    "rank": int(rank),
                    "perm_idx": perm_idx,
                    "order": order_str,
                })
            # Sla toelichtingen op van eerste permutatie
            if perm_idx == 0 and note:
                pairwise_notes.append({"case_id": case_id, "criterium": crit, "toelichting": note})

        # Korte samenvatting
        for crit in SCORE_KEYS[:2]:
            crit_data = parsed.get(crit, {})
            ranking = crit_data.get("ranking", {})
            print(f"{crit[:6]}={ranking}", end=" ")
        print("...")

df_pairwise_all = pd.DataFrame(all_perm_rows)
df_pairwise_notes = pd.DataFrame(pairwise_notes) if pairwise_notes else pd.DataFrame(
    columns=["case_id", "criterium", "toelichting"]
)

# ── Gemiddelde rank over permutaties ──
if not df_pairwise_all.empty:
    df_pairwise = (
        df_pairwise_all
        .groupby(["case_id", "criterium", "experiment"])["rank"]
        .mean()
        .reset_index()
    )
    df_pairwise["rank"] = df_pairwise["rank"].round(2)

    # Win-tabel (gemiddelde rank ≤ 1.25 telt als win)
    wins = df_pairwise[df_pairwise["rank"] <= 1.25].groupby("experiment").size().reset_index(name="wins")
    for exp in available_exps:
        if exp not in wins["experiment"].values:
            wins = pd.concat([wins, pd.DataFrame([{"experiment": exp, "wins": 0}])], ignore_index=True)
    total_rankings = len(cases) * len(SCORE_KEYS)
    wins["win_rate"] = (wins["wins"] / total_rankings).round(3)
    wins = wins.sort_values("experiment")
    print(f"\n=== Win-tabel (gem. rank ≤ 1.25 over {N_PERMS} permutaties, {total_rankings} rankings) ===")
    display(wins)

    # Gemiddelde rank per experiment per criterium
    avg_ranks = df_pairwise.groupby(["experiment", "criterium"])["rank"].mean().unstack()
    avg_ranks["gemiddeld"] = avg_ranks.mean(axis=1).round(2)
    print("\nGemiddelde rank per experiment (1=best, gemiddeld over permutaties):")
    display(avg_ranks.round(2))

    # Positiebias-check: gemiddelde rank per promptpositie
    df_pairwise_all["prompt_position"] = df_pairwise_all.apply(
        lambda r: r["order"].split(",").index(r["experiment"].upper()) + 1, axis=1
    )
    pos_effect = df_pairwise_all.groupby("prompt_position")["rank"].mean()
    print("\nPositiebias-check (gem. rank per positie in prompt):")
    for pos, avg_rank in pos_effect.items():
        bias = "✅ neutraal" if abs(avg_rank - 2.5) < 0.3 else "⚠️ mogelijke bias"
        print(f"  Positie {pos}: gem. rank = {avg_rank:.2f}  {bias}")

    # ── Visualisatie: 3 subplots ──
    palette = {"a": "#4C72B0", "b": "#55A868", "c": "#C44E52", "d": "#8172B2"}
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    # 1. Gemiddelde rank per experiment (lager = beter)
    avg_rank_by_exp = df_pairwise.groupby("experiment")["rank"].mean().reindex(EXPERIMENTS)
    bar_colors = [palette.get(e, "#999") for e in EXPERIMENTS]
    bars = axes[0].bar(EXPERIMENTS, avg_rank_by_exp.values, color=bar_colors)
    axes[0].set_title(f"Gemiddelde rank per experiment\n(1=best, {N_PERMS} permutaties)")
    axes[0].set_ylabel("Gemiddelde rank")
    axes[0].set_xlabel("Experiment")
    axes[0].set_ylim(0, len(available_exps) + 0.5)
    axes[0].axhline(y=2.5, color="gray", linestyle="--", alpha=0.5, label="gemiddeld")
    axes[0].invert_yaxis()  # lager = beter → bovenaan
    axes[0].set_ylim(len(available_exps) + 0.3, 0.5)  # inversie: 1 bovenaan
    for i, v in enumerate(avg_rank_by_exp.values):
        axes[0].text(i, v + 0.08, f"{v:.2f}", ha="center", fontweight="bold", fontsize=10)
    axes[0].legend(fontsize=8)

    # 2. Win rate per experiment
    wins_sorted = wins.set_index("experiment").reindex(EXPERIMENTS).reset_index()
    bars2 = axes[1].bar(EXPERIMENTS, wins_sorted["win_rate"].values, color=bar_colors)
    axes[1].set_title(f"Win rate per experiment\n(gem. rank ≤ 1.25)")
    axes[1].set_ylabel("Fractie #1 rankings")
    axes[1].set_xlabel("Experiment")
    axes[1].set_ylim(0, max(0.5, wins_sorted["win_rate"].max() * 1.3 + 0.05))
    for i, v in enumerate(wins_sorted["win_rate"].values):
        y_pos = max(v, 0.01) + 0.01
        axes[1].text(i, y_pos, f"{v:.0%}", ha="center", fontweight="bold", fontsize=10)

    # 3. Heatmap per criterium
    rank_heatmap = df_pairwise.pivot_table(
        index="criterium", columns="experiment", values="rank", aggfunc="mean"
    )
    rank_heatmap = rank_heatmap[EXPERIMENTS]  # vaste volgorde
    sns.heatmap(rank_heatmap, annot=True, cmap="RdYlGn_r", vmin=1, vmax=len(available_exps),
                fmt=".2f", ax=axes[2], linewidths=0.5)
    axes[2].set_title("Ranking per criterium\n(1=best, bias-gecorrigeerd)")

    plt.tight_layout()
    plt.show()
else:
    print("Geen pairwise data beschikbaar.")

### 4.4 Chain-of-Thought (CoT) stap-voor-stap analyse
> **Adresseert:** *Levert elke redeneerstap betere kwaliteit op en worden inzichten consequent doorvertaald?*

De prompt schrijft 11 gestructureerde stappen voor (probleemdefinitie → externe analyse → interne analyse → synthese → scenario’s → opties → evaluatie → advies → implementatie → monitoring → aannames).

Deze analyse meet twee dimensies:
1. **Stapkwaliteit** — Hoe goed is elke individuele stap uitgevoerd? (1–5)
2. **Doorvertaling** — Worden inzichten uit stap N consequent meegenomen in stap N+1? (1–5)

Dit sluit aan bij het concept van Explainable AI (XAI): niet alleen het eindantwoord telt, maar ook de kwaliteit en traceerbaarheid van het redeneerproces.

In [0]:
# ── CoT stap-voor-stap analyse ───────────────────────────────────────────
# Isoleert elke redeneerstap, beoordeelt kwaliteit per stap,
# en meet doorvertaling van inzichten tussen opeenvolgende stappen.

COT_STEPS = [
    ("1", "Probleemdefinitie en doel"),
    ("2", "Externe analyse"),
    ("3", "Interne analyse"),
    ("4", "Synthese"),
    ("5", "Scenario-analyse"),
    ("6", "Strategische opties"),
    ("7", "Evaluatie en keuze"),
    ("8", "Strategisch advies"),
    ("9", "Implementatie"),
    ("10", "Monitoring en bijsturing"),
    ("11", "Aannames en beperkingen"),
]

def extract_steps(answer):
    """Splits een advies in secties op basis van markdown headers (### N. of **N.)."""
    # Stap 1: vind alle section headers met hun positie en stapnummer
    header_pattern = r'(?:^|\n)\s*((?:#{1,3}|\*\*)\s*(\d{1,2})\.\s.+)'
    headers = list(re.finditer(header_pattern, answer))
    
    if not headers:
        return {}
    
    # Stap 2: bouw een geordende lijst van (positie, stapnummer, header_tekst)
    found = []
    for h in headers:
        num = h.group(2)
        found.append((h.start(), num, h.group(1).strip()))
    found.sort(key=lambda x: x[0])
    
    # Stap 3: extraheer tekst per stap (van header tot volgende header)
    steps = {}
    for i, (pos, num, header) in enumerate(found):
        end = found[i + 1][0] if i + 1 < len(found) else len(answer)
        steps[num] = answer[pos:end].strip()
    
    return steps

JSON_FMT = '{"score": <1-5>, "toelichting": "<korte uitleg>"}'

def judge_step_quality(step_num, step_label, step_text):
    """Beoordeel de kwaliteit van een redeneerstap (1-5)."""
    prompt = (
        "Beoordeel de kwaliteit van de volgende redeneerstap uit een strategisch advies.\n\n"
        + f"STAP: {step_num}. {step_label}\n\n"
        + f"TEKST:\n{step_text[:1500]}\n\n"
        + "Beoordeel op:\n"
        + "- Volledigheid: worden alle relevante aspecten behandeld?\n"
        + "- Specificiteit: zijn uitspraken concreet en onderbouwd met data?\n"
        + "- Analytische diepte: wordt er echt geanalyseerd of alleen beschreven?\n\n"
        + f"Geef JSON terug: {JSON_FMT}\n"
    )
    try:
        resp = openai_client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0, max_tokens=200,
        )
        parsed = parse_judge_json(resp.choices[0].message.content or "")
        return int(parsed.get("score", 3)), parsed.get("toelichting", "")
    except Exception as e:
        return 3, str(e)

def judge_step_coherence(prev_num, prev_label, prev_text, curr_num, curr_label, curr_text):
    """Meet of inzichten uit de vorige stap doorvertaald worden."""
    prompt = (
        "Beoordeel of de HUIDIGE stap inzichten uit de VORIGE stap consequent doorvertaalt.\n\n"
        + f"VORIGE STAP ({prev_num}. {prev_label}):\n{prev_text[:800]}\n\n"
        + f"HUIDIGE STAP ({curr_num}. {curr_label}):\n{curr_text[:800]}\n\n"
        + "Beoordeel de doorvertaling:\n"
        + "- Worden bevindingen/inzichten uit de vorige stap expliciet gebruikt?\n"
        + "- Bouwt de huidige stap logisch voort op de vorige?\n"
        + "- Zijn er losse eindjes die niet worden opgepakt?\n\n"
        + f"Geef JSON terug: {JSON_FMT}\n"
    )
    try:
        resp = openai_client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0, max_tokens=200,
        )
        parsed = parse_judge_json(resp.choices[0].message.content or "")
        return int(parsed.get("score", 3)), parsed.get("toelichting", "")
    except Exception as e:
        return 3, str(e)

# ── Analyse uitvoeren (run 1 per experiment/case) ──
df_cot_run1 = df_runs[df_runs["run"] == 1].copy()

print(f"CoT stap-analyse voor {len(df_cot_run1)} antwoorden...")
print("Dit kan 5-10 minuten duren...\n")

cot_rows = []
for _, row in df_cot_run1.iterrows():
    exp, case_id, answer = row["experiment"], row["case_id"], row["answer"]
    steps = extract_steps(answer)
    
    # Map gevonden stappen naar COT_STEPS labels
    step_labels = {num: label for num, label in COT_STEPS}
    found_nums = sorted(steps.keys(), key=lambda x: int(x))
    
    print(f"  {exp}/{case_id}: {len(steps)} stappen gevonden ({', '.join(found_nums)})")
    
    prev_num, prev_label, prev_text = None, None, None
    
    for num in found_nums:
        label = step_labels.get(num, f"Stap {num}")
        text = steps[num]
        
        quality, q_note = judge_step_quality(num, label, text)
        
        if prev_text is not None:
            coherence, c_note = judge_step_coherence(
                prev_num, prev_label, prev_text, num, label, text
            )
        else:
            coherence, c_note = np.nan, "eerste stap"
        
        cot_rows.append({
            "experiment": exp, "case_id": case_id,
            "step_num": int(num), "step_label": label,
            "quality": quality, "quality_note": q_note,
            "coherence": coherence, "coherence_note": c_note,
            "step_length": len(text),
        })
        
        prev_num, prev_label, prev_text = num, label, text

df_cot = pd.DataFrame(cot_rows)
print(f"\nTotaal: {len(df_cot)} stap-beoordelingen")
df_cot.head(10)

In [0]:
# ── CoT visualisatie ─────────────────────────────────────────────────

# 1. Stapkwaliteit per experiment (lijn-grafiek)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

quality_by_step = df_cot.groupby(["experiment", "step_num"])["quality"].mean().reset_index()
for exp in EXPERIMENTS:
    subset = quality_by_step[quality_by_step["experiment"] == exp]
    axes[0].plot(subset["step_num"], subset["quality"],
                 marker="o", label=f"Experiment {exp.upper()}", linewidth=2)
axes[0].set_xlabel("Stap")
axes[0].set_ylabel("Kwaliteit (1-5)")
axes[0].set_title("Stapkwaliteit per architectuur")
axes[0].set_xticks(range(1, 12))
axes[0].set_ylim(1, 5.5)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Doorvertaling (coherence) per stap-overgang
coherence_by_step = df_cot.dropna(subset=["coherence"]).groupby(
    ["experiment", "step_num"]
)["coherence"].mean().reset_index()
for exp in EXPERIMENTS:
    subset = coherence_by_step[coherence_by_step["experiment"] == exp]
    axes[1].plot(subset["step_num"], subset["coherence"],
                 marker="s", label=f"Experiment {exp.upper()}", linewidth=2)
axes[1].set_xlabel("Stap")
axes[1].set_ylabel("Doorvertaling (1-5)")
axes[1].set_title("Doorvertaling van inzichten per stap-overgang")
axes[1].set_xticks(range(1, 12))
axes[1].set_ylim(1, 5.5)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 3. Heatmap: kwaliteit per stap per experiment
heatmap_data = df_cot.pivot_table(
    index="step_label", columns="experiment", values="quality", aggfunc="mean"
)
# Sorteer op stapvolgorde
step_order = [label for _, label in COT_STEPS]
heatmap_data = heatmap_data.reindex([l for l in step_order if l in heatmap_data.index])

plt.figure(figsize=(8, 7))
sns.heatmap(heatmap_data, annot=True, fmt=".1f", cmap="RdYlGn", vmin=1, vmax=5,
            linewidths=0.5)
plt.title("Stapkwaliteit per architectuur (gemiddeld over cases)")
plt.xlabel("Experiment")
plt.ylabel("")
plt.tight_layout()
plt.show()

# 4. Samenvattingstabel
print("\nGemiddelde kwaliteit en doorvertaling per experiment:")
cot_summary = df_cot.groupby("experiment").agg(
    gem_kwaliteit=("quality", "mean"),
    gem_doorvertaling=("coherence", "mean"),
    stappen_gevonden=("step_length", lambda x: (x > 0).sum()),
    stappen_totaal=("step_num", "count"),
).round(2)
display(cot_summary)

# 5. Zwakste stappen identificeren
print("\nZwakste stappen (kwaliteit < 4):")
weak = df_cot[df_cot["quality"] < 4].sort_values("quality")
if len(weak) > 0:
    display(weak[["experiment", "case_id", "step_label", "quality", "quality_note"]].head(10))
else:
    print("Geen stappen met kwaliteit < 4 gevonden.")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# 4.4.1 Informatiestroom door de CoT-keten
# ═══════════════════════════════════════════════════════════════════════
# Drie kernvragen:
#   1. Hoeveel informatie stroomt van stap N naar stap N+1? (adjacente overlap)
#   2. Hoeveel van ALLE voorgaande stappen keert terug in stap N? (cumulatieve retentie)
#   3. Draagt stapkwaliteit bij aan de eindkwaliteit? (stap → judge correlatie)
#
# Methode: embedding-gebaseerd (cosine similarity op stapteksten)
# Dit is objectiever dan LLM-as-judge doorvertaling.

from sklearn.metrics.pairwise import cosine_similarity as cos_sim
from scipy.stats import spearmanr

def _step_to_text(step_val):
    """Haal tekst op uit extract_steps(), ongeacht dict of string formaat.
    utils.steps versie retourneert {"label": ..., "text": ...},
    lokale versie retourneert plain strings."""
    if isinstance(step_val, dict):
        return step_val.get("text", "")
    return str(step_val)

df_run1 = df_runs[df_runs["run"] == 1].copy()

flow_rows = []       # adjacente overdracht (stap N → N+1)
retention_rows = []  # cumulatieve retentie (stap 1..N-1 → N)
full_matrices = {}   # volledige sim matrices per exp

for _, row in df_run1.iterrows():
    exp, case_id, answer = row["experiment"], row["case_id"], row["answer"]
    steps = extract_steps(answer)
    if len(steps) < 3:
        continue

    # Sorteer stappen op nummer, extraheer tekst
    sorted_nums = sorted(steps.keys(), key=lambda x: int(x))
    raw_texts = [_step_to_text(steps[n]) for n in sorted_nums]
    raw_nums = [int(n) for n in sorted_nums]

    # Filter lege stappen
    valid = [(n, t) for n, t in zip(raw_nums, raw_texts) if t and len(t.strip()) > 20]
    if len(valid) < 3:
        continue
    step_nums, step_texts = zip(*valid)
    step_nums, step_texts = list(step_nums), list(step_texts)

    # Encode alle stappen in één batch
    embs = embedding_model.encode(step_texts, convert_to_numpy=True, normalize_embeddings=True)

    # Volledige similarity matrix
    sim_matrix = cos_sim(embs)

    # Bewaar per experiment
    if exp not in full_matrices:
        full_matrices[exp] = []
    full_matrices[exp].append((step_nums, sim_matrix))

    # Adjacente overdracht: sim(stap_i, stap_i+1)
    for i in range(len(step_texts) - 1):
        flow_rows.append({
            "experiment": exp, "case_id": case_id,
            "from_step": step_nums[i], "to_step": step_nums[i + 1],
            "transition": f"{step_nums[i]}→{step_nums[i+1]}",
            "similarity": float(sim_matrix[i, i + 1]),
        })

    # Cumulatieve retentie: sim(concat(stap_1..N-1), stap_N)
    for i in range(1, len(step_texts)):
        prev_concat = " ".join(step_texts[:i])
        prev_emb = embedding_model.encode([prev_concat], convert_to_numpy=True, normalize_embeddings=True)
        cum_sim = float(cos_sim(prev_emb, [embs[i]])[0][0])

        retention_rows.append({
            "experiment": exp, "case_id": case_id,
            "step_num": step_nums[i],
            "n_prev_steps": i,
            "cumulative_retention": cum_sim,
        })

df_flow = pd.DataFrame(flow_rows)
df_retention = pd.DataFrame(retention_rows)

print(f"Berekend: {len(df_flow)} adjacente overgangen, {len(df_retention)} cumulatieve retentiemetingen")

# ───────────────────────────────────────────────────────────────────────
# VISUALISATIE
# ───────────────────────────────────────────────────────────────────────
from utils.steps import STEP_LABELS
step_short = {str(i): STEP_LABELS.get(str(i), f"Stap {i}")[:18] for i in range(1, 12)}

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# ── Plot 1: Adjacente informatie-overdracht per stap-overgang ──
ax = axes[0, 0]
for exp in EXPERIMENTS:
    sub = df_flow[df_flow["experiment"] == exp]
    means = sub.groupby("from_step")["similarity"].mean().sort_index()
    ax.plot(means.index, means.values, "o-", color=palette[exp],
            label=f"{exp.upper()} ({EXP_LABELS[exp]})", linewidth=2, markersize=6)
ax.set_xlabel("Van stap")
ax.set_ylabel("Cosine similarity met volgende stap")
ax.set_title("Informatie-overdracht naar de volgende stap\n(hoger = meer inhoudelijke overlap)")
ax.set_xticks(range(1, 11))
ax.set_xticklabels([f"{i}→{i+1}" for i in range(1, 11)], fontsize=8, rotation=45)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.3, 1.0)

# ── Plot 2: Cumulatieve retentie — hoeveel van alles daarvoor komt terug? ──
ax = axes[0, 1]
for exp in EXPERIMENTS:
    sub = df_retention[df_retention["experiment"] == exp]
    means = sub.groupby("step_num")["cumulative_retention"].mean().sort_index()
    ax.plot(means.index, means.values, "s-", color=palette[exp],
            label=f"{exp.upper()}", linewidth=2, markersize=6)
ax.set_xlabel("Stap")
ax.set_ylabel("Cosine similarity met alle voorgaande stappen")
ax.set_title("Cumulatieve informatieretentie\n(hoeveel van stap 1..N-1 zit in stap N?)")
ax.set_xticks(range(2, 12))
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.3, 1.0)
ax.axhline(y=0.7, color="gray", linestyle="--", alpha=0.3)
ax.text(2.1, 0.71, "hoge retentie (≥0.7)", fontsize=8, color="gray")

# ── Plot 3: Volledige stap-stap similarity matrix ──
ax = axes[1, 0]
avg_matrix = np.zeros((11, 11))
counts = np.zeros((11, 11))
for exp_m, matrices in full_matrices.items():
    for step_nums_m, sim_mat in matrices:
        for i, si in enumerate(step_nums_m):
            for j, sj in enumerate(step_nums_m):
                avg_matrix[si-1, sj-1] += sim_mat[i, j]
                counts[si-1, sj-1] += 1
counts[counts == 0] = 1
avg_matrix /= counts

step_labels_11 = [step_short[str(i)] for i in range(1, 12)]

im = ax.imshow(avg_matrix, cmap="YlOrRd", vmin=0.3, vmax=0.9, aspect="auto")
for i in range(11):
    for j in range(11):
        val = avg_matrix[i, j]
        color = "white" if val > 0.7 else "black"
        if i != j:
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=6, color=color)
ax.set_xticks(range(11))
ax.set_yticks(range(11))
ax.set_xticklabels([str(i+1) for i in range(11)], fontsize=8)
ax.set_yticklabels(step_labels_11, fontsize=7)
ax.set_xlabel("Stap (kolom)")
ax.set_ylabel("Stap (rij)")
ax.set_title("Informatieoverlap tussen ALLE stapparen\n(embedding similarity, gemiddeld over alle antwoorden)")
plt.colorbar(im, ax=ax, shrink=0.8, label="Cosine sim")

# ── Plot 4: Stapkwaliteit vs eindkwaliteit ──
ax = axes[1, 1]
if 'df_judge' in dir() and not df_judge.empty:
    cot_avg = df_cot.groupby(["experiment", "case_id"]).agg(
        gem_quality=("quality", "mean"),
        gem_coherence=("coherence", "mean"),
    ).reset_index()
    cot_judge = cot_avg.merge(
        df_judge[["experiment", "case_id", "totaal"]],
        on=["experiment", "case_id"], how="inner"
    )
    if len(cot_judge) > 3:
        for exp in EXPERIMENTS:
            sub = cot_judge[cot_judge["experiment"] == exp]
            ax.scatter(sub["gem_quality"], sub["totaal"], color=palette[exp],
                       s=120, label=f"{exp.upper()}", edgecolors="white", alpha=0.8)
        from numpy.polynomial.polynomial import polyfit
        x, y = cot_judge["gem_quality"].values, cot_judge["totaal"].values
        mask_v = ~(np.isnan(x) | np.isnan(y))
        if mask_v.sum() > 3:
            b, m = polyfit(x[mask_v], y[mask_v], 1)
            x_line = np.linspace(x[mask_v].min(), x[mask_v].max(), 50)
            ax.plot(x_line, b + m * x_line, "--", color="gray", alpha=0.6)
            rho, p = spearmanr(x[mask_v], y[mask_v])
            ax.set_title(f"Draagt stapkwaliteit bij aan eindkwaliteit?\n(Spearman ρ={rho:.3f}, p={p:.3f})")
        ax.set_xlabel("Gemiddelde CoT stapkwaliteit (1–5)")
        ax.set_ylabel("Judge totaalscore (0–100)")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, "Onvoldoende data", ha="center", va="center", transform=ax.transAxes)
else:
    ax.text(0.5, 0.5, "df_judge niet beschikbaar", ha="center", va="center", transform=ax.transAxes)

plt.suptitle("§4.4.1 Informatiestroom door de Chain-of-Thought keten",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# ───────────────────────────────────────────────────────────────────────
# NUMERIEKE SAMENVATTING
# ───────────────────────────────────────────────────────────────────────

print("\n" + "=" * 75)
print("INFORMATIESTROOM DOOR DE COT-KETEN")
print("=" * 75)

print("\n1. ADJACENTE OVERDRACHT (stap N → stap N+1):")
print("   Hoeveel van stap N is direct terug te vinden in stap N+1?\n")
flow_summary = df_flow.groupby("experiment")["similarity"].agg(["mean", "std", "min", "max"]).round(3)
flow_summary.index = [f"{e.upper()} ({EXP_LABELS[e]})" for e in flow_summary.index]
display(flow_summary)

print("\n   Sterkste overgangen (meeste overlap):")
flow_by_trans = df_flow.groupby("transition")["similarity"].mean().sort_values(ascending=False)
for trans, sim in flow_by_trans.head(3).items():
    print(f"     {trans}: sim={sim:.3f}")
print("\n   Zwakste overgangen (informatiebreuk):")
for trans, sim in flow_by_trans.tail(3).items():
    print(f"     {trans}: sim={sim:.3f}")

print("\n2. CUMULATIEVE RETENTIE (stap 1..N-1 → stap N):")
print("   Hoeveel van ALLE voorgaande redenering is terug te vinden?\n")
ret_by_exp = df_retention.groupby("experiment")["cumulative_retention"].agg(["mean", "std"]).round(3)
ret_by_exp.index = [f"{e.upper()} ({EXP_LABELS[e]})" for e in ret_by_exp.index]
display(ret_by_exp)

ret_by_step = df_retention.groupby("step_num")["cumulative_retention"].mean()
if len(ret_by_step) > 2:
    ret_diff = ret_by_step.diff()
    worst_drop = ret_diff.idxmin()
    prev_val = ret_by_step.get(worst_drop - 1, ret_by_step.iloc[0])
    print(f"\n   Grootste retentiedaling bij stap {worst_drop}: "
          f"Δ={ret_diff[worst_drop]:.3f} (van {prev_val:.3f} naar {ret_by_step[worst_drop]:.3f})")
    print(f"   → Hier gaat de meeste voorgaande informatie 'verloren'")

print("\n3. DRAAGT STAPKWALITEIT BIJ AAN EINDKWALITEIT?")
if 'cot_judge' in dir() and len(cot_judge) > 3:
    rho_q, p_q = spearmanr(cot_judge["gem_quality"], cot_judge["totaal"])
    rho_c, p_c = spearmanr(cot_judge["gem_coherence"].dropna(), 
                            cot_judge.loc[cot_judge["gem_coherence"].notna(), "totaal"])
    print(f"   Stapkwaliteit ↔ Judge totaal:  ρ={rho_q:.3f}, p={p_q:.3f} "
          f"{'(✅ significant)' if p_q < 0.05 else '(❌ niet significant)'}")
    print(f"   Doorvertaling ↔ Judge totaal:  ρ={rho_c:.3f}, p={p_c:.3f} "
          f"{'(✅ significant)' if p_c < 0.05 else '(❌ niet significant)'}")

    print("\n   Welke individuele stap voorspelt het best de eindkwaliteit?")
    step_corrs = []
    for sn in range(1, 12):
        sq = df_cot[df_cot["step_num"] == sn][["experiment", "case_id", "quality"]]
        mg = sq.merge(df_judge[["experiment", "case_id", "totaal"]], 
                      on=["experiment", "case_id"], how="inner")
        if len(mg) >= 5:
            rho_s, p_s = spearmanr(mg["quality"], mg["totaal"])
            label = STEP_LABELS.get(str(sn), f"Stap {sn}")
            step_corrs.append((sn, label, rho_s, p_s))
    step_corrs.sort(key=lambda x: abs(x[2]), reverse=True)
    for num, label, rho_s, p_s in step_corrs[:5]:
        sig = "★" if p_s < 0.05 else " "
        print(f"   {sig} Stap {num:2d} ({label:25s}): ρ={rho_s:+.3f} (p={p_s:.3f})")

print("\n" + "=" * 75)
print("CONCLUSIE")
print("=" * 75)
mean_adj = df_flow["similarity"].mean()
mean_ret = df_retention["cumulative_retention"].mean()
print(f"\n  Gemiddelde adjacente overlap:      {mean_adj:.3f}")
print(f"  Gemiddelde cumulatieve retentie:   {mean_ret:.3f}")
if mean_adj > 0.6:
    print("\n  ✅ De CoT-stappen bouwen substantieel op elkaar voort")
    print("     (adjacente sim > 0.6 = significante inhoudelijke overlap)")
else:
    print("\n  ⚠️  Beperkte overdracht tussen opeenvolgende stappen")
    print("     De stappen functioneren meer als losse secties")
if mean_ret > 0.65:
    print("  ✅ Hoge cumulatieve retentie: latere stappen integreren voorgaand redeneren")
else:
    print("  ⚠️  Lage cumulatieve retentie: latere stappen verliezen context")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# 4.4.2 Retentie per experiment: waar verschilt de informatiestroom?
# ═══════════════════════════════════════════════════════════════════════
# Bouwt voort op df_flow, df_retention en full_matrices uit §4.4.1.
# STEP_LABELS, step_short etc. zijn al geïmporteerd in §4.4.1.
#
# Visualisaties:
#   1. Retentie-heatmap: experiment × stap → cumulatieve retentie
#   2. Adjacente overlap heatmap: experiment × overgang
#   3. Per-experiment 11×11 similarity matrices (informatiestroom)
#   4. Delta-heatmap: verschil t.o.v. gemiddelde (waar wijkt elk exp af?)

# Hergebruik step_short uit cel hierboven (of herdefinieer als vangnet)
if 'step_short' not in dir():
    step_short = {str(i): STEP_LABELS.get(str(i), f"Stap {i}")[:15] for i in range(1, 12)}

# ───────────────────────────────────────────────────────────────────────
# Figuur 1: Retentie + adjacente heatmaps (2×2)
# ───────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 13))

# 1a. Cumulatieve retentie heatmap
ax = axes[0, 0]
ret_pivot = df_retention.pivot_table(
    index="experiment", columns="step_num",
    values="cumulative_retention", aggfunc="mean"
).reindex(EXPERIMENTS)

sns.heatmap(ret_pivot, annot=True, fmt=".2f", cmap="RdYlGn",
            vmin=0.4, vmax=0.9, linewidths=0.5, ax=ax,
            xticklabels=[f"{int(c)}" for c in ret_pivot.columns],
            yticklabels=[f"{e.upper()} ({EXP_LABELS[e]})" for e in EXPERIMENTS])
ax.set_title("Cumulatieve informatieretentie per experiment\n(hoeveel van stap 1..N-1 zit in stap N?)")
ax.set_xlabel("Stap")
ax.set_ylabel("")

# 1b. Adjacente overlap heatmap
ax = axes[0, 1]
adj_pivot = df_flow.pivot_table(
    index="experiment", columns="from_step",
    values="similarity", aggfunc="mean"
).reindex(EXPERIMENTS)

sns.heatmap(adj_pivot, annot=True, fmt=".2f", cmap="RdYlGn",
            vmin=0.3, vmax=0.8, linewidths=0.5, ax=ax,
            xticklabels=[f"{int(c)}→{int(c)+1}" for c in adj_pivot.columns],
            yticklabels=[f"{e.upper()}" for e in EXPERIMENTS])
ax.set_title("Adjacente informatie-overdracht per experiment\n(cosine sim stap N → stap N+1)")
ax.set_xlabel("Overgang")
ax.set_ylabel("")

# 1c. Delta retentie vs gemiddelde
ax = axes[1, 0]
ret_mean_col = ret_pivot.mean(axis=0)
ret_delta = ret_pivot.subtract(ret_mean_col, axis=1)

sns.heatmap(ret_delta, annot=True, fmt="+.3f", cmap="RdYlGn", center=0,
            vmin=-0.06, vmax=0.06, linewidths=0.5, ax=ax,
            xticklabels=[f"{int(c)}" for c in ret_delta.columns],
            yticklabels=[f"{e.upper()} ({EXP_LABELS[e]})" for e in EXPERIMENTS])
ax.set_title("Δ Retentie vs gemiddelde\n(groen = behoudt meer info dan gemiddeld)")
ax.set_xlabel("Stap")
ax.set_ylabel("")

# 1d. Delta adjacente overdracht vs gemiddelde
ax = axes[1, 1]
adj_mean_col = adj_pivot.mean(axis=0)
adj_delta = adj_pivot.subtract(adj_mean_col, axis=1)

sns.heatmap(adj_delta, annot=True, fmt="+.3f", cmap="RdYlGn", center=0,
            vmin=-0.06, vmax=0.06, linewidths=0.5, ax=ax,
            xticklabels=[f"{int(c)}→{int(c)+1}" for c in adj_delta.columns],
            yticklabels=[f"{e.upper()}" for e in EXPERIMENTS])
ax.set_title("Δ Adjacente overdracht vs gemiddelde\n(groen = sterkere koppeling dan gemiddeld)")
ax.set_xlabel("Overgang")
ax.set_ylabel("")

plt.suptitle("§4.4.2 Informatiestroom per experiment: heatmaps",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# ───────────────────────────────────────────────────────────────────────
# Figuur 2: Per-experiment 11×11 similarity matrices
# ───────────────────────────────────────────────────────────────────────
fig, axes_m = plt.subplots(2, 2, figsize=(16, 14))
step_labels_11 = [step_short.get(str(i), f"Stap {i}")[:15] for i in range(1, 12)]

for idx, exp in enumerate(EXPERIMENTS):
    ax = axes_m[idx // 2, idx % 2]

    exp_matrix = np.zeros((11, 11))
    exp_counts = np.zeros((11, 11))

    if exp in full_matrices:
        for step_nums_m, sim_mat in full_matrices[exp]:
            for i, si in enumerate(step_nums_m):
                for j, sj in enumerate(step_nums_m):
                    exp_matrix[si-1, sj-1] += sim_mat[i, j]
                    exp_counts[si-1, sj-1] += 1

    exp_counts[exp_counts == 0] = 1
    exp_matrix /= exp_counts

    im = ax.imshow(exp_matrix, cmap="YlOrRd", vmin=0.3, vmax=0.9, aspect="auto")
    for i in range(11):
        for j in range(11):
            val = exp_matrix[i, j]
            if i != j and val > 0:
                color = "white" if val > 0.7 else "black"
                ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=5.5, color=color)
    ax.set_xticks(range(11))
    ax.set_yticks(range(11))
    ax.set_xticklabels([str(i+1) for i in range(11)], fontsize=7)
    ax.set_yticklabels(step_labels_11, fontsize=6)
    ax.set_title(f"{exp.upper()} ({EXP_LABELS[exp]})", fontsize=12, fontweight="bold",
                 color=palette[exp])

    # Highlight adjacente band
    for k in range(10):
        rect = plt.Rectangle((k + 0.5, k - 0.5), 1, 1, fill=False,
                              edgecolor=palette[exp], linewidth=1.5, linestyle="--")
        ax.add_patch(rect)

plt.suptitle("§4.4.2 Stap-stap similarity matrices per experiment\n(stippellijn = adjacente overdracht)",
             fontsize=14, fontweight="bold", y=1.01)
fig.subplots_adjust(right=0.92)
cbar_ax = fig.add_axes([0.94, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cbar_ax, label="Cosine similarity")
plt.tight_layout(rect=[0, 0, 0.92, 0.96])
plt.show()

# ───────────────────────────────────────────────────────────────────────
# SAMENVATTING
# ───────────────────────────────────────────────────────────────────────
print("\n" + "=" * 75)
print("RETENTIE-PROFIEL PER EXPERIMENT")
print("=" * 75)

for exp in EXPERIMENTS:
    ret_exp = ret_pivot.loc[exp] if exp in ret_pivot.index else pd.Series(dtype=float)
    adj_exp = adj_pivot.loc[exp] if exp in adj_pivot.index else pd.Series(dtype=float)

    print(f"\n  {exp.upper()} ({EXP_LABELS[exp]}):")
    if not ret_exp.empty:
        print(f"    Gem. retentie:          {ret_exp.mean():.3f}")
        best_step = ret_exp.idxmax()
        worst_step = ret_exp.idxmin()
        print(f"    Hoogste retentie:       stap {int(best_step)} ({step_short.get(str(int(best_step)), '?')}) = {ret_exp[best_step]:.3f}")
        print(f"    Laagste retentie:       stap {int(worst_step)} ({step_short.get(str(int(worst_step)), '?')}) = {ret_exp[worst_step]:.3f}")
    if not adj_exp.empty:
        best_trans = adj_exp.idxmax()
        worst_trans = adj_exp.idxmin()
        print(f"    Sterkste koppeling:     {int(best_trans)}→{int(best_trans)+1} = {adj_exp[best_trans]:.3f}")
        print(f"    Zwakste koppeling:      {int(worst_trans)}→{int(worst_trans)+1} = {adj_exp[worst_trans]:.3f}")

print("\n" + "-" * 75)
ret_means = ret_pivot.mean(axis=1)
best_exp = ret_means.idxmax()
worst_exp = ret_means.idxmin()
print(f"  Beste informatieretentie:     {best_exp.upper()} ({EXP_LABELS[best_exp]}) = {ret_means[best_exp]:.3f}")
print(f"  Zwakste informatieretentie:   {worst_exp.upper()} ({EXP_LABELS[worst_exp]}) = {ret_means[worst_exp]:.3f}")
print(f"  Verschil:                     Δ={ret_means[best_exp] - ret_means[worst_exp]:.3f}")

In [0]:
# Export specifieke figuren uit cellen 15, 24, 27, 35, 43, 60, 63
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

FIG_DIR = "/Workspace/Users/rgoedings@hanos.nl/AI_op_het_menu/figures"
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(fig, name, dpi=150):
    path = os.path.join(FIG_DIR, f"{name}.png")
    fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  ✅ {name}.png")

print("Exporteren extra figuren...\n")

# ═══ CEL 15: Scenario diversiteit vs temperatuur (2x2 panel) ═══════════════
if 'df_ht_valid' in dir() and not df_ht_valid.empty:
    temps_sorted = sorted(df_ht_valid["temperature"].unique())
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    sns.boxplot(data=df_ht_valid, x="temperature", y="diversity", ax=axes[0, 0],
                color="#4C72B0", showmeans=True,
                meanprops={"marker": "D", "markerfacecolor": "white", "markersize": 8})
    for i, t in enumerate(temps_sorted):
        vals = df_ht_valid[df_ht_valid["temperature"] == t]["diversity"]
        if len(vals) > 0:
            axes[0, 0].text(i, vals.mean() + 0.008, f"μ={vals.mean():.3f}", ha="center", fontsize=9, fontweight="bold")
    axes[0, 0].set_title("Scenario diversiteit per temperatuur")
    axes[0, 0].set_ylabel("Diversiteit (1 − gem. cosine sim)")
    axes[0, 0].set_xlabel("Temperature")
    axes[0, 0].grid(True, alpha=0.3)

    for exp in EXPERIMENTS:
        sub = df_ht_valid[df_ht_valid["experiment"] == exp]
        means = sub.groupby("temperature")["diversity"].mean()
        stds = sub.groupby("temperature")["diversity"].std()
        axes[0, 1].plot(means.index, means.values, "o-", color=palette[exp],
                        label=f"{exp.upper()} ({EXP_LABELS[exp]})", linewidth=2)
        axes[0, 1].fill_between(means.index, means - stds, means + stds, alpha=0.15, color=palette[exp])
    axes[0, 1].set_xlabel("Temperature")
    axes[0, 1].set_ylabel("Gem. diversiteit")
    axes[0, 1].set_title("Diversiteit per experiment × temperatuur")
    axes[0, 1].legend(fontsize=9)
    axes[0, 1].grid(True, alpha=0.3)

    hm_data = df_ht_valid.pivot_table(index="experiment", columns="temperature",
                                       values="diversity", aggfunc="mean").reindex(EXPERIMENTS)
    sns.heatmap(hm_data, annot=True, fmt=".3f", cmap="RdYlGn",
                vmin=0, vmax=hm_data.values.max() * 1.2, linewidths=0.5, ax=axes[1, 0])
    axes[1, 0].set_title("Diversiteit heatmap (experiment × temperature)")
    axes[1, 0].set_ylabel("Experiment")
    axes[1, 0].set_xlabel("Temperature")

    # Scatter: stabiliteit vs diversiteit
    ht_csv = os.path.join(PROJECT_ROOT, "data", "results_data", "hypertune_results.csv")
    if os.path.exists(ht_csv):
        df_ht_csv = pd.read_csv(ht_csv)
        ht_merged = df_ht_valid.groupby(["experiment", "case_id", "temperature"])["diversity"].mean().reset_index()
        ht_merged = ht_merged.merge(df_ht_csv[["experiment", "case_id", "temperature", "stability"]],
                                     on=["experiment", "case_id", "temperature"], how="left")
        for exp in EXPERIMENTS:
            sub = ht_merged[ht_merged["experiment"] == exp]
            axes[1, 1].scatter(sub["stability"], sub["diversity"], color=palette[exp],
                               label=f"{exp.upper()}", s=80, alpha=0.7, edgecolors="white")
        axes[1, 1].set_xlabel("Stabiliteit (cosine sim)")
        axes[1, 1].set_ylabel("Scenario diversiteit")
        axes[1, 1].set_title("Trade-off: stabiliteit vs diversiteit")
        axes[1, 1].legend(fontsize=9)
        axes[1, 1].grid(True, alpha=0.3)

    plt.suptitle("§4.2.1 Temperatuur × scenario diversiteit", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    savefig(fig, "fig_4_2_temperatuur_diversiteit")
else:
    print("  ⚠️ cel 15: df_ht_valid niet beschikbaar")

# ═══ CEL 24: CoT visualisatie (stapkwaliteit + doorvertaling) ═════════════
if 'df_cot' in dir() and not df_cot.empty:
    # Fig 1: lijngrafieken
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    quality_by_step = df_cot.groupby(["experiment", "step_num"])["quality"].mean().reset_index()
    for exp in EXPERIMENTS:
        subset = quality_by_step[quality_by_step["experiment"] == exp]
        axes[0].plot(subset["step_num"], subset["quality"], marker="o",
                     label=f"Experiment {exp.upper()}", linewidth=2, color=palette[exp])
    axes[0].set_xlabel("Stap")
    axes[0].set_ylabel("Kwaliteit (1-5)")
    axes[0].set_title("Stapkwaliteit per architectuur")
    axes[0].set_xticks(range(1, 12))
    axes[0].set_ylim(1, 5.5)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    coherence_by_step = df_cot.dropna(subset=["coherence"]).groupby(
        ["experiment", "step_num"])["coherence"].mean().reset_index()
    for exp in EXPERIMENTS:
        subset = coherence_by_step[coherence_by_step["experiment"] == exp]
        axes[1].plot(subset["step_num"], subset["coherence"], marker="s",
                     label=f"Experiment {exp.upper()}", linewidth=2, color=palette[exp])
    axes[1].set_xlabel("Stap")
    axes[1].set_ylabel("Doorvertaling (1-5)")
    axes[1].set_title("Doorvertaling van inzichten per stap-overgang")
    axes[1].set_xticks(range(1, 12))
    axes[1].set_ylim(1, 5.5)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    savefig(fig, "fig_4_3_cot_kwaliteit_lijnen")

    # Fig 2: heatmap
    from utils.steps import COT_STEPS
    heatmap_data = df_cot.pivot_table(index="step_label", columns="experiment", values="quality", aggfunc="mean")
    step_order = [label for _, label in COT_STEPS]
    heatmap_data = heatmap_data.reindex([l for l in step_order if l in heatmap_data.index])
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(heatmap_data, annot=True, fmt=".1f", cmap="RdYlGn", vmin=1, vmax=5, linewidths=0.5, ax=ax)
    ax.set_title("Stapkwaliteit per architectuur (gemiddeld over cases)")
    ax.set_xlabel("Experiment")
    ax.set_ylabel("")
    plt.tight_layout()
    savefig(fig, "fig_4_3_cot_heatmap")
else:
    print("  ⚠️ cel 24: df_cot niet beschikbaar (vereist API calls)")

# ═══ CEL 27: Analyse→Strategie informatiedoorvertaling ════════════════
ANALYSE_STAPPEN = [1, 2, 3, 4]
TARGET_ADVIES = 8

rows_totaal = []
rows_per_stap = []
for _, row in df_run1.iterrows():
    exp, case_id = row["experiment"], row["case_id"]
    steps = extract_steps(row["answer"])
    if not steps or str(TARGET_ADVIES) not in steps:
        continue
    analyse_texts = []
    for s in ANALYSE_STAPPEN:
        if str(s) in steps:
            t = _step_to_text(steps[str(s)])
            if t and len(t.strip()) > 20:
                analyse_texts.append((s, t))
    if len(analyse_texts) < 2:
        continue
    advies_text = _step_to_text(steps[str(TARGET_ADVIES)])
    if not advies_text or len(advies_text.strip()) < 20:
        continue
    # Totale analyse → advies
    concat_analyse = " ".join([t for _, t in analyse_texts])
    embs = embedding_model.encode([concat_analyse, advies_text], convert_to_numpy=True, normalize_embeddings=True)
    sim_total = float(cosine_similarity([embs[0]], [embs[1]])[0][0])
    rows_totaal.append({"experiment": exp, "case_id": case_id, "similarity": sim_total})
    # Per stap
    emb_advies = embs[1:2]
    for s, t in analyse_texts:
        emb_s = embedding_model.encode([t], convert_to_numpy=True, normalize_embeddings=True)
        sim_s = float(cosine_similarity(emb_s, emb_advies)[0][0])
        rows_per_stap.append({"experiment": exp, "case_id": case_id, "stap": s, "similarity": sim_s})

df_a2a_total = pd.DataFrame(rows_totaal)
df_a2a_stap = pd.DataFrame(rows_per_stap)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Links: totale similarity per experiment
for i, exp in enumerate(EXPERIMENTS):
    vals = df_a2a_total[df_a2a_total["experiment"] == exp]["similarity"]
    axes[0].bar(i, vals.mean(), color=palette[exp], yerr=vals.std(), capsize=5)
    axes[0].text(i, vals.mean() + vals.std() + 0.01, f"{vals.mean():.3f}", ha="center", fontweight="bold")
axes[0].set_xticks(range(len(EXPERIMENTS)))
axes[0].set_xticklabels([e.upper() for e in EXPERIMENTS])
axes[0].set_title("Analyse → Advies informatiedoorvertaling\n(cosine sim: concat analysestappen vs advies)")
axes[0].set_ylabel("Cosine similarity")
axes[0].set_ylim(0, 0.85)
axes[0].grid(True, alpha=0.3, axis="y")

# Rechts: per analysestap
from utils.steps import STEP_LABELS
stap_means = df_a2a_stap.groupby("stap")["similarity"].mean()
stap_labels = [STEP_LABELS.get(str(s), f"Stap {s}")[:20] for s in stap_means.index]
axes[1].barh(stap_labels, stap_means.values, color="#3498DB")
for i, v in enumerate(stap_means.values):
    axes[1].text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)
axes[1].set_xlabel("Cosine similarity met advies (stap 8)")
axes[1].set_title("Welke analysestap landt het sterkst in het advies?")
axes[1].set_xlim(0, 0.75)
plt.tight_layout()
savefig(fig, "fig_4_4f_analyse_advies_doorvertaling")

# ═══ CEL 35: Stap-niveau vergelijking (investering + divergentie + refine) ═══
if 'df_steps' in dir() and not df_steps.empty:
    df_steps_r1 = df_steps[(df_steps["run"] == 1) & (df_steps["phase"] == "final")].copy()
    # Fig 1: Tekstlengte per stap
    length_pivot = df_steps_r1.pivot_table(index="step_num", columns="experiment",
                                            values="step_chars", aggfunc="mean").reindex(range(1, 12))
    length_pivot = length_pivot[[e for e in EXPERIMENTS if e in length_pivot.columns]]
    length_pivot.index = [STEP_LABELS.get(str(i), f"Stap {i}") for i in range(1, 12)]

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    sns.heatmap(length_pivot, annot=True, fmt=".0f", cmap="YlOrRd", linewidths=0.5, ax=axes[0])
    axes[0].set_title("Tekstlengte per stap per experiment (karakters)")
    axes[0].set_ylabel("")

    for exp in EXPERIMENTS:
        if exp in length_pivot.columns:
            axes[1].plot(range(1, 12), length_pivot[exp].values, marker="o",
                         label=f"{exp.upper()}", linewidth=2, color=palette.get(exp))
    axes[1].set_xlabel("Stap")
    axes[1].set_ylabel("Gem. tekstlengte (karakters)")
    axes[1].set_title("Investering per redeneerstap")
    axes[1].set_xticks(range(1, 12))
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    savefig(fig, "fig_4_5b_stap_investering")
else:
    print("  ⚠️ cel 35: df_steps niet beschikbaar")

# ═══ CEL 43: Stabiliteit boxplot (BETER GEIJKT) ═════════════════════
if 'stability_by_case' in dir() and not stability_by_case.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.boxplot(data=stability_by_case, x="experiment", y="stability",
                order=EXPERIMENTS, ax=ax, palette=palette, width=0.5)
    # Zoom y-as naar daadwerkelijk bereik
    y_min = stability_by_case["stability"].min()
    y_max = stability_by_case["stability"].max()
    margin = (y_max - y_min) * 0.15
    ax.set_ylim(max(0, y_min - margin), min(1.0, y_max + margin))
    # Gemiddelde annotatie
    for i, exp in enumerate(EXPERIMENTS):
        vals = stability_by_case[stability_by_case["experiment"] == exp]["stability"].dropna()
        if not vals.empty:
            ax.text(i, vals.mean() + 0.003, f"μ={vals.mean():.3f}", ha="center", fontsize=9, fontweight="bold")
    ax.set_title("Stabiliteit per experiment (geijkt op werkelijk bereik)", fontsize=13)
    ax.set_ylabel("Cosine similarity")
    ax.set_xlabel("Experiment")
    ax.grid(True, alpha=0.2, axis="y")
    plt.tight_layout()
    savefig(fig, "fig_4_5_stabiliteit_boxplot_geijkt")
else:
    print("  ⚠️ cel 43: stability_by_case niet beschikbaar")

# ═══ CEL 60: Intra-advies scenario diversiteit (alle runs) ═════════════
print("  Scenario-diversiteit berekenen...")
div_rows_all = []
for _, row in df_runs.iterrows():
    scenarios = extract_scenarios_from_answer(row["answer"])
    n_scen = len(scenarios)
    if n_scen >= 2:
        embs = embedding_model.encode(scenarios[:3], normalize_embeddings=True)
        sim = cosine_similarity(embs)
        iu = np.triu_indices_from(sim, k=1)
        diversity = float(1.0 - sim[iu].mean())
    else:
        diversity = np.nan
    div_rows_all.append({"experiment": row["experiment"], "run": row["run"],
                         "case_id": row["case_id"], "diversity": diversity, "n_scenarios": n_scen})

df_div_all = pd.DataFrame(div_rows_all)
df_div_valid = df_div_all.dropna(subset=["diversity"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df_div_valid, x="experiment", y="diversity", order=EXPERIMENTS,
            palette=palette, ax=axes[0])
axes[0].set_title("Intra-advies scenario diversiteit per experiment")
axes[0].set_ylabel("Diversiteit (1 − gem. cosine sim)")
axes[0].grid(True, alpha=0.3, axis="y")

# Per case
sns.boxplot(data=df_div_valid, x="case_id", y="diversity", hue="experiment",
            hue_order=EXPERIMENTS, palette=palette, ax=axes[1])
axes[1].set_title("Scenario diversiteit per case per experiment")
axes[1].set_ylabel("Diversiteit")
axes[1].legend(title="Experiment", fontsize=8)
axes[1].grid(True, alpha=0.3, axis="y")
plt.tight_layout()
savefig(fig, "fig_4_9_scenario_diversiteit")

# ═══ CEL 63: Radar/windroos (integrale vergelijking) ═════════════════
# Herbereken final als dat nodig is
if 'stability_summary' not in dir():
    stability_summary = stability_by_case.groupby("experiment").agg(
        stability=("stability", "mean")).reset_index()

final = stability_summary[["experiment", "stability"]].copy()
# Judge scores
if 'df_judge' in dir() and not df_judge.empty:
    SCORE_KEYS = ["probleemdefinitie", "analyse", "scenarios", "advies", "cot_samenhang"]
    judge_summary = df_judge.groupby("experiment")[SCORE_KEYS + ["totaal"]].mean().round(1)
    judge_avg = judge_summary[["totaal"]].reset_index()
    judge_avg.columns = ["experiment", "judge_score"]
    final = final.merge(judge_avg, on="experiment", how="left")
    for crit in SCORE_KEYS:
        crit_avg = judge_summary[[crit]].reset_index()
        crit_avg.columns = ["experiment", f"judge_{crit}"]
        final = final.merge(crit_avg, on="experiment", how="left")
else:
    SCORE_KEYS = ["probleemdefinitie", "analyse", "scenarios", "advies", "cot_samenhang"]
    final["judge_score"] = np.nan
    for crit in SCORE_KEYS:
        final[f"judge_{crit}"] = np.nan

# Diversiteit
div_summary = df_div_valid.groupby("experiment")["diversity"].mean().reset_index()
div_summary.columns = ["experiment", "avg_diversity"]
final = final.merge(div_summary, on="experiment", how="left")

# Efficiëntie
eff = df_runs.groupby("experiment").agg(
    avg_latency=("latency_seconds", "mean"), avg_tokens=("total_tokens", "mean")).round(1).reset_index()
final = final.merge(eff, on="experiment", how="left")
for col, out_col in [("avg_latency", "latency_score"), ("avg_tokens", "token_score")]:
    vals = final[col].dropna()
    if len(vals) > 0 and vals.max() != vals.min():
        final[out_col] = 0.2 + 0.8 * (1 - (final[col] - vals.min()) / (vals.max() - vals.min()))
    else:
        final[out_col] = 0.5

# CGR
if 'df_cgr' in dir() and not df_cgr.empty:
    cgr_avg = df_cgr.groupby("experiment")[["CGR"]].mean().reset_index()
    cgr_avg.columns = ["experiment", "avg_CGR"]
    final = final.merge(cgr_avg, on="experiment", how="left")
else:
    final["avg_CGR"] = np.nan

# Radar chart
radar_metrics = [
    ("stability", "Stabiliteit", "direct"),
    ("judge_probleemdefinitie", "Probleem-\ndefinitie", "div100"),
    ("judge_analyse", "Analyse", "div100"),
    ("judge_scenarios", "Scenario's", "div100"),
    ("judge_advies", "Advies", "div100"),
    ("judge_cot_samenhang", "CoT\nsamenhang", "div100"),
    ("avg_CGR", "CGR", "direct"),
    ("avg_diversity", "Diversiteit", "direct"),
    ("latency_score", "Snelheid\n(↑=sneller)", "direct"),
    ("token_score", "Token-\nefficiëntie\n(↑=zuiniger)", "direct"),
]

metric_keys = [m[0] for m in radar_metrics]
metric_labels = [m[1] for m in radar_metrics]
metric_norms = [m[2] for m in radar_metrics]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
angles = np.linspace(0, 2 * np.pi, len(metric_keys), endpoint=False).tolist()
angles += angles[:1]

for exp in EXPERIMENTS:
    exp_rows = final[final["experiment"] == exp]
    if exp_rows.empty:
        continue
    row = exp_rows.iloc[0]
    values = []
    for key, norm in zip(metric_keys, metric_norms):
        v = row.get(key, np.nan)
        if pd.isna(v):
            values.append(0)
        elif norm == "div100":
            values.append(v / 100)
        else:
            values.append(float(v))
    values += values[:1]
    ax.plot(angles, values, "o-", linewidth=2,
            label=f"{exp.upper()} ({EXP_LABELS[exp]})", color=palette.get(exp))
    ax.fill(angles, values, alpha=0.08, color=palette.get(exp))

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metric_labels, fontsize=8)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=7, color="grey")
ax.set_title("Integrale vergelijking per evaluatiedimensie", pad=25, fontsize=12)
ax.legend(loc="upper right", bbox_to_anchor=(1.45, 1.1))
plt.tight_layout()
savefig(fig, "fig_4_10_radar_windroos")

print(f"\n{'='*70}")
print(f"KLAAR — Bestanden in: {FIG_DIR}")
print(f"{'='*70}")
for f in sorted(os.listdir(FIG_DIR)):
    if f.endswith(".png"):
        size_kb = os.path.getsize(os.path.join(FIG_DIR, f)) / 1024
        print(f"  📊 {f} ({size_kb:.0f} KB)")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# §4.4.3  Komt de analyse terug in het advies?
# ═══════════════════════════════════════════════════════════════════════
#
# De CoT-prompt schrijft eerst 4 analysestappen voor:
#   Stap 1 – Probleemdefinitie
#   Stap 2 – Externe analyse
#   Stap 3 – Interne analyse
#   Stap 4 – Synthese
#
# Daarna volgt stap 8 – Strategisch advies.
#
# KERNVRAAG: Hoeveel van die analyse-informatie komt daadwerkelijk
#            terug in het uiteindelijke advies?
#
# Methode:
#   • Embed elke analysestap (1-4) apart én geconcateneerd
#   • Embed het advies (stap 8)
#   • Cosine similarity = proxy voor informatiedoorvertaling
#   • Similarity 1.0 = identieke inhoud, 0.0 = geen overlap
#
# In de praktijk liggen embedding-similarities voor gerelateerde teksten
# typisch tussen 0.3 (zwak verwant) en 0.8 (sterk verwant).
# ═══════════════════════════════════════════════════════════════════════

ANALYSE_STAPPEN = [1, 2, 3, 4]
TARGET_ADVIES   = 8

# ── 1. Bereken similarity: hele analysefase (concat 1-4) → advies ──
rows_totaal = []
for _, row in df_run1.iterrows():
    exp, case_id = row["experiment"], row["case_id"]
    steps = extract_steps(row["answer"])
    if not steps or str(TARGET_ADVIES) not in steps:
        continue

    analyse_texts = []
    for s in ANALYSE_STAPPEN:
        if str(s) in steps:
            t = _step_to_text(steps[str(s)])
            if t and len(t.strip()) > 20:
                analyse_texts.append(t)
    if len(analyse_texts) < 2:
        continue

    advies_text = _step_to_text(steps[str(TARGET_ADVIES)])
    if not advies_text or len(advies_text.strip()) < 20:
        continue

    emb_analyse = embedding_model.encode(
        [" ".join(analyse_texts)], convert_to_numpy=True, normalize_embeddings=True)
    emb_advies = embedding_model.encode(
        [advies_text], convert_to_numpy=True, normalize_embeddings=True)
    sim = float(cos_sim(emb_analyse, emb_advies)[0][0])

    rows_totaal.append({"experiment": exp, "case_id": case_id, "sim": sim})

df_totaal = pd.DataFrame(rows_totaal)

# ── 2. Bereken per individuele analysestap → advies ──
rows_detail = []
for _, row in df_run1.iterrows():
    exp, case_id = row["experiment"], row["case_id"]
    steps = extract_steps(row["answer"])
    if not steps or str(TARGET_ADVIES) not in steps:
        continue

    advies_text = _step_to_text(steps[str(TARGET_ADVIES)])
    if not advies_text or len(advies_text.strip()) < 20:
        continue
    emb_advies = embedding_model.encode(
        [advies_text], convert_to_numpy=True, normalize_embeddings=True)

    for s in ANALYSE_STAPPEN:
        if str(s) not in steps:
            continue
        s_text = _step_to_text(steps[str(s)])
        if not s_text or len(s_text.strip()) < 20:
            continue
        emb_s = embedding_model.encode(
            [s_text], convert_to_numpy=True, normalize_embeddings=True)
        sim = float(cos_sim(emb_s, emb_advies)[0][0])
        rows_detail.append({
            "experiment": exp, "case_id": case_id,
            "stap": s,
            "stap_label": STEP_LABELS.get(str(s), f"Stap {s}"),
            "sim": sim,
        })

df_detail = pd.DataFrame(rows_detail)

print(f"Totaal: {len(df_totaal)} metingen (analysefase → advies)")
print(f"Detail: {len(df_detail)} metingen (per analysestap → advies)")

# ═══════════════════════════════════════════════════════════════════════
# VISUALISATIE  (3 panelen)
# ═══════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.3)

stap_namen = {
    1: "Probleem-\ndefinitie",
    2: "Externe\nanalyse",
    3: "Interne\nanalyse",
    4: "Synthese",
}

# ──────────────────────────────────────────────────────────────────────
# PLOT 1 (boven-links):  Totale doorvertaling per experiment
#   "Hoeveel van de HELE analysefase komt terug in het advies?"
# ──────────────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
exp_means = df_totaal.groupby("experiment")["sim"].agg(["mean", "std"]).reindex(EXPERIMENTS)
colors_exp = [palette[e] for e in EXPERIMENTS]

bars = ax1.bar(range(len(EXPERIMENTS)), exp_means["mean"],
               yerr=exp_means["std"], capsize=5,
               color=colors_exp, edgecolor="white", linewidth=1.5, alpha=0.9)

for i, (exp, row_m) in enumerate(exp_means.iterrows()):
    ax1.text(i, row_m["mean"] + row_m["std"] + 0.012,
             f"{row_m['mean']:.2f}", ha="center", fontsize=13, fontweight="bold")

ax1.set_xticks(range(len(EXPERIMENTS)))
ax1.set_xticklabels([f"{e.upper()}\n{EXP_LABELS[e]}" for e in EXPERIMENTS], fontsize=10)
ax1.set_ylabel("Cosine similarity", fontsize=11)
ax1.set_title("Hoeveel analyse-informatie\nkomt terug in het advies?", fontsize=13, fontweight="bold")
ax1.set_ylim(0.25, 0.85)
ax1.axhspan(0.25, 0.40, alpha=0.08, color="red",   label="Zwak (<0.4)")
ax1.axhspan(0.40, 0.55, alpha=0.08, color="orange", label="Matig (0.4-0.55)")
ax1.axhspan(0.55, 0.70, alpha=0.08, color="green",  label="Goed (0.55-0.7)")
ax1.axhspan(0.70, 0.85, alpha=0.08, color="#006400", label="Sterk (>0.7)")
ax1.legend(loc="upper right", fontsize=7, framealpha=0.8)
ax1.grid(True, alpha=0.2, axis="y")

# ──────────────────────────────────────────────────────────────────────
# PLOT 2 (boven-rechts):  Per analysestap → advies (gemiddeld)
#   "Welke analysestap komt het meest terug in het advies?"
# ──────────────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
stap_means = df_detail.groupby("stap")["sim"].mean().reindex(ANALYSE_STAPPEN)
stap_stds  = df_detail.groupby("stap")["sim"].std().reindex(ANALYSE_STAPPEN)
stap_colors = ["#E74C3C", "#3498DB", "#2ECC71", "#F39C12"]

bars2 = ax2.bar(range(4), stap_means.values,
                yerr=stap_stds.values, capsize=5,
                color=stap_colors, edgecolor="white", linewidth=1.5, alpha=0.9)

for i, (val, std) in enumerate(zip(stap_means.values, stap_stds.values)):
    ax2.text(i, val + std + 0.012, f"{val:.2f}",
             ha="center", fontsize=13, fontweight="bold")

ax2.set_xticks(range(4))
ax2.set_xticklabels([stap_namen[s] for s in ANALYSE_STAPPEN], fontsize=10)
ax2.set_ylabel("Cosine similarity met advies", fontsize=11)
ax2.set_title("Welke analysestap komt\nhet meest terug in het advies?", fontsize=13, fontweight="bold")
ax2.set_ylim(0.25, 0.85)
ax2.axhspan(0.25, 0.40, alpha=0.08, color="red")
ax2.axhspan(0.40, 0.55, alpha=0.08, color="orange")
ax2.axhspan(0.55, 0.70, alpha=0.08, color="green")
ax2.axhspan(0.70, 0.85, alpha=0.08, color="#006400")
ax2.grid(True, alpha=0.2, axis="y")

# ──────────────────────────────────────────────────────────────────────
# PLOT 3 (onder, breed):  Heatmap experiment × analysestap → advies
#   "Welke architectuur benut welke analysestap het best?"
# ──────────────────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, :])
heat = df_detail.pivot_table(
    index="experiment", columns="stap",
    values="sim", aggfunc="mean"
).reindex(EXPERIMENTS)

# Voeg totaal-kolom toe
heat["TOTAAL\n(concat 1-4)"] = exp_means["mean"].values

heat.index = [f"{e.upper()} ({EXP_LABELS[e]})" for e in EXPERIMENTS]
heat.columns = [stap_namen.get(c, c) if isinstance(c, int) else c for c in heat.columns]

# Annotaties met interpretatie
annot_text = heat.copy()
for c in heat.columns:
    for r in heat.index:
        v = heat.loc[r, c]
        if v >= 0.60:   level = "\n✓ goed"
        elif v >= 0.50: level = "\n~ matig"
        else:           level = "\n✗ zwak"
        annot_text.loc[r, c] = f"{v:.2f}{level}"

sns.heatmap(heat, annot=annot_text, fmt="", cmap="RdYlGn",
            vmin=0.35, vmax=0.75, linewidths=2, ax=ax3,
            cbar_kws={"label": "Cosine similarity (analyse → advies)", "shrink": 0.6},
            annot_kws={"fontsize": 11})
ax3.set_title("Per architectuur × per analysestap: hoeveel komt terug in het advies?",
              fontsize=13, fontweight="bold")
ax3.set_xlabel("")
ax3.set_ylabel("")

# Markeer de totaal-kolom
ax3.axvline(x=4, color="black", linewidth=2.5)

plt.suptitle("§4.4.3  Analyse → Strategisch advies: wordt de onderbouwing benut?",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════════════
# SAMENVATTING
# ═══════════════════════════════════════════════════════════════════════
print("\n" + "═" * 70)
print("  SAMENVATTING: Analyse → Advies doorvertaling")
print("═" * 70)

print("\n  Totale doorvertaling (analysefase → advies):")
for exp in EXPERIMENTS:
    v = exp_means.loc[exp, "mean"]
    level = "goed" if v >= 0.60 else "matig" if v >= 0.50 else "zwak"
    print(f"    {exp.upper()} ({EXP_LABELS[exp]:12s}):  {v:.2f}  [{level}]")

best  = exp_means["mean"].idxmax()
worst = exp_means["mean"].idxmin()
print(f"\n  → Beste:   {best.upper()} ({EXP_LABELS[best]}) benut de analyse het best ({exp_means.loc[best,'mean']:.2f})")
print(f"  → Zwakste: {worst.upper()} ({EXP_LABELS[worst]}) verliest de meeste analyse-info ({exp_means.loc[worst,'mean']:.2f})")

print("\n  Per analysestap (gem. over alle experimenten):")
for s in ANALYSE_STAPPEN:
    v = stap_means[s]
    print(f"    Stap {s} ({STEP_LABELS.get(str(s),'?'):25s}):  {v:.2f}")

best_stap  = stap_means.idxmax()
worst_stap = stap_means.idxmin()
print(f"\n  → Meest benut: Stap {best_stap} ({STEP_LABELS.get(str(best_stap),'?')}) = {stap_means[best_stap]:.2f}")
print(f"  → Minst benut: Stap {worst_stap} ({STEP_LABELS.get(str(worst_stap),'?')}) = {stap_means[worst_stap]:.2f}")
print(f"    ↳ {STEP_LABELS.get(str(worst_stap),'?')} informatie verdampt het snelst op weg naar het advies")

# Interpretatie
print("\n" + "─" * 70)
print("  INTERPRETATIE")
print("─" * 70)
gem = exp_means['mean'].mean()
print(f"\n  Gemiddelde doorvertaling: {gem:.2f}")
if gem >= 0.55:
    print("  → De LLM's benutten de analysefase REDELIJK GOED in hun advies")
    print("    De CoT-structuur dwingt informatiehergebruik af")
else:
    print("  → De analysefase wordt MATIG benut in het advies")
    print("    Er gaat substantiële informatie verloren tussen analyse en advies")
print(f"\n  Opvallend: Synthese (stap 4) scoort het hoogst — dit is logisch")
print(f"  omdat de synthese al een samenvatting IS van stap 1-3.")
print(f"  Het advies bouwt dus vooral op de synthese, niet op de ruwe analyses.")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# §4.4.4  Thematische traceerbaarheid: Analyse → Opties → Advies → Implementatie
# ═══════════════════════════════════════════════════════════════════════
#
# KERNVRAAG: Welke strategische thema's identificeert de analyse,
#            en komen die thema's ook daadwerkelijk terug in de
#            strategische opties, het advies en het implementatieplan?
#
# Methode (embedding-based, geen API calls):
#   1. Definieer 12 strategische thema's met beschrijvingen
#   2. Embed elk thema + elke CoT-sectie
#   3. Cosine similarity = thema-aanwezigheidsscore per sectie
#   4. Track het thema door de keten: analyse → opties → advies → implementatie
# ═══════════════════════════════════════════════════════════════════════

# ── Strategische thema's voor restaurantadvies ──
THEMAS = {
    "Marketing & promotie": (
        "Marketing, promotie, reclame, social media, online zichtbaarheid, "
        "branding, naamsbekendheid, marketingstrategie, campagnes, "
        "klantwerving, bereik vergroten, Google reviews, Instagram"
    ),
    "Menu & aanbod": (
        "Menukaart, gerechten, menu-engineering, assortiment, menu-innovatie, "
        "seizoensgebonden aanbod, specials, menusamenstelling, food concept, "
        "gerechtontwikkeling, smaken, culinair aanbod, vegetarisch, vegan"
    ),
    "Klantervaring": (
        "Klantervaring, gastvrijheid, service, beleving, sfeer, interieur, "
        "klanttevredenheid, hospitality, ambiance, customer experience, "
        "gasttevredenheid, terugkeergedrag, loyaliteit, NPS"
    ),
    "Prijsstrategie": (
        "Prijzen, prijsstrategie, prijsniveau, value for money, food cost, "
        "marge, winstgevendheid per gerecht, pricing, kortingen, aanbiedingen, "
        "prijs-kwaliteitverhouding, omzet per couvert"
    ),
    "Personeel & organisatie": (
        "Personeel, medewerkers, personeelstekort, werving, opleiding, "
        "training, teambuilding, planning, roostering, HR, arbeidsmarkt, "
        "personeelskosten, ziekteverzuim, bedrijfscultuur, leiderschap"
    ),
    "Financieel & kosten": (
        "Financiën, kosten, kostenbeheersing, budget, liquiditeit, "
        "break-even, investering, rendement, omzet, winst, cashflow, "
        "kostprijsberekening, overhead, exploitatiekosten"
    ),
    "Digitalisering & technologie": (
        "Digitalisering, technologie, online bestellen, reserveringssysteem, "
        "kassasysteem, POS, bezorgplatform, website, app, automatisering, "
        "digitale menukaart, QR bestellen, data-analyse"
    ),
    "Duurzaamheid": (
        "Duurzaamheid, milieu, lokale inkoop, biologisch, voedselverspilling, "
        "afvalreductie, CO2-reductie, seizoensproducten, farm-to-table, "
        "circulair, energiebesparing, groene bedrijfsvoering"
    ),
    "Concurrentie & marktpositie": (
        "Concurrentie, marktanalyse, concurrenten, marktpositie, "
        "onderscheidend vermogen, USP, positionering, concurrentievoordeel, "
        "marktsegment, doelgroep, niche, benchmarking"
    ),
    "Groei & expansie": (
        "Groei, expansie, schaalvergroting, tweede vestiging, franchise, "
        "catering, cateringtak, nieuwe markt, omzetgroei, capaciteitsuitbreiding, "
        "bezetting verhogen, extra inkomstenbronnen"
    ),
    "Operatie & logistiek": (
        "Operationeel, keukenproces, inkoop, leveranciers, supply chain, "
        "voorraadbeheer, efficiëntie, workflow, keukenindeling, "
        "bereidingsprocessen, hygiëne, HACCP, kwaliteitscontrole"
    ),
    "Trends & innovatie": (
        "Trends, innovatie, foodtrends, dark kitchen, ghost kitchen, "
        "plant-based, streetfood, fusion, pop-up, experiential dining, "
        "nieuwe concepten, markttrends, consumentengedrag, generatie Z"
    ),
}

print(f"Gedefinieerd: {len(THEMAS)} strategische thema's")
for naam in THEMAS:
    print(f"  • {naam}")

# ── Embed alle thema's ──
thema_namen = list(THEMAS.keys())
thema_beschrijvingen = list(THEMAS.values())
emb_themas = embedding_model.encode(
    thema_beschrijvingen, convert_to_numpy=True, normalize_embeddings=True
)
print(f"\nThema-embeddings: {emb_themas.shape}")

# ── CoT-fasen: 4 opeenvolgende schakels ──
FASEN = {
    "Analyse":        [1, 2, 3, 4],   # probleemdefinitie + externe + interne + synthese
    "Opties":         [6],             # strategische opties
    "Advies":         [8],             # strategisch advies
    "Implementatie":  [9],             # implementatieplan
}
fase_order = ["Analyse", "Opties", "Advies", "Implementatie"]
fase_kleuren = {"Analyse": "#3498DB", "Opties": "#E67E22",
                "Advies": "#2ECC71", "Implementatie": "#9B59B6"}

# ── Meet thema-aanwezigheid per CoT-fase ──
rows_thema = []

for _, row in df_run1.iterrows():
    exp, case_id = row["experiment"], row["case_id"]
    steps = extract_steps(row["answer"])
    if not steps:
        continue

    for fase_naam, fase_stappen in FASEN.items():
        teksten = []
        for s in fase_stappen:
            if str(s) in steps:
                t = _step_to_text(steps[str(s)])
                if t and len(t.strip()) > 20:
                    teksten.append(t)
        if not teksten:
            continue

        fase_tekst = " ".join(teksten)
        emb_fase = embedding_model.encode(
            [fase_tekst], convert_to_numpy=True, normalize_embeddings=True
        )
        sims = cosine_similarity(emb_fase, emb_themas)[0]

        for i, (thema, sim_score) in enumerate(zip(thema_namen, sims)):
            rows_thema.append({
                "experiment": exp, "case_id": case_id,
                "fase": fase_naam, "thema": thema,
                "score": float(sim_score),
            })

df_thema = pd.DataFrame(rows_thema)
print(f"\nBerekend: {len(df_thema)} thema-metingen")
print(f"  ({len(df_thema) // len(thema_namen)} fase-metingen × {len(thema_namen)} thema's)")

# ── Bepaal VASTE thema-volgorde (op analyse-score, hoogste bovenaan) ──
thema_rank = (
    df_thema[df_thema["fase"] == "Analyse"]
    .groupby("thema")["score"].mean()
    .sort_values(ascending=False)
)
THEMA_ORDER = list(thema_rank.index)  # vaste volgorde voor alle plots
print(f"\nVaste thema-volgorde (op analyse-sterkte):")
for i, t in enumerate(THEMA_ORDER, 1):
    print(f"  {i:2d}. {t:30s}  ({thema_rank[t]:.3f})")

# ═══════════════════════════════════════════════════════════════════════
# FIGUUR 1: Thema-aanwezigheid per fase (vaste volgorde)
# ═══════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 4, figsize=(24, 7), sharey=True)

# Vaste volgorde (omgekeerd voor barh: laagste bovenaan, hoogste onderaan)
thema_order_barh = list(reversed(THEMA_ORDER))

for idx, fase in enumerate(fase_order):
    ax = axes[idx]
    sub = df_thema[df_thema["fase"] == fase]
    means = sub.groupby("thema")["score"].mean().reindex(thema_order_barh)

    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(means)))
    bars = ax.barh(range(len(means)), means.values, color=colors, edgecolor="white")

    for i, (val, thema) in enumerate(zip(means.values, means.index)):
        ax.text(val + 0.005, i, f"{val:.2f}", va="center", fontsize=9, fontweight="bold")

    ax.set_yticks(range(len(means)))
    ax.set_yticklabels(means.index, fontsize=9)
    ax.set_xlim(0.10, 0.55)
    ax.set_title(f"{fase}\n(stap {'+'.join(str(s) for s in FASEN[fase])})",
                 fontsize=13, fontweight="bold", color=fase_kleuren[fase])
    ax.set_xlabel("Thema-aanwezigheid (cosine sim)")
    ax.grid(True, alpha=0.2, axis="x")

plt.suptitle("§4.4.4  Welke thema's domineren in elke fase?",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════════════
# FIGUUR 2: Thema-doorvertaling + opeenvolgende delta's
# ═══════════════════════════════════════════════════════════════════════
heat_flow = df_thema.pivot_table(
    index="thema", columns="fase",
    values="score", aggfunc="mean"
)[fase_order].reindex(THEMA_ORDER)

# Opeenvolgende delta's: elke schakel in de keten
heat_flow["Δ Analyse\n→ Opties"] = heat_flow["Opties"] - heat_flow["Analyse"]
heat_flow["Δ Opties\n→ Advies"] = heat_flow["Advies"] - heat_flow["Opties"]
heat_flow["Δ Advies\n→ Implementatie"] = heat_flow["Implementatie"] - heat_flow["Advies"]

fig, axes = plt.subplots(1, 2, figsize=(22, 8), gridspec_kw={"width_ratios": [4, 3]})

# Links: absolute scores per fase (4 kolommen)
ax = axes[0]
sns.heatmap(heat_flow[fase_order], annot=True, fmt=".2f",
            cmap="YlOrRd", vmin=0.15, vmax=0.55,
            linewidths=1.5, ax=ax,
            cbar_kws={"label": "Thema-aanwezigheid", "shrink": 0.8})
ax.set_title("Thema-aanwezigheid per fase\n(vaste volgorde op analyse-sterkte)",
             fontsize=13, fontweight="bold")
ax.set_ylabel("")
ax.set_xlabel("")

# Rechts: opeenvolgende delta's (3 kolommen)
ax = axes[1]
delta_cols = ["Δ Analyse\n→ Opties", "Δ Opties\n→ Advies", "Δ Advies\n→ Implementatie"]
sns.heatmap(heat_flow[delta_cols], annot=True, fmt="+.3f",
            cmap="RdBu_r", center=0, vmin=-0.08, vmax=0.08,
            linewidths=1.5, ax=ax,
            cbar_kws={"label": "Δ score", "shrink": 0.8})
ax.set_title("Verschuiving per schakel\n(rood=groeit, blauw=krimpt)",
             fontsize=13, fontweight="bold")
ax.set_ylabel("")
ax.set_xlabel("")

plt.suptitle("§4.4.4  Thematische verschuiving door de CoT-keten",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════════════
# FIGUUR 3: Per experiment — thematisch profiel (vaste volgorde)
# ═══════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(22, 14))

for idx, exp in enumerate(EXPERIMENTS):
    ax = axes[idx // 2][idx % 2]
    sub = df_thema[df_thema["experiment"] == exp]

    pivot = sub.pivot_table(
        index="thema", columns="fase",
        values="score", aggfunc="mean"
    )[fase_order].reindex(THEMA_ORDER)  # vaste volgorde

    # Opeenvolgende delta's
    pivot["Δ Ana\n→Opt"] = pivot["Opties"] - pivot["Analyse"]
    pivot["Δ Opt\n→Adv"] = pivot["Advies"] - pivot["Opties"]
    pivot["Δ Adv\n→Imp"] = pivot["Implementatie"] - pivot["Advies"]

    # Annotatie: .2f voor fasen, +.3f voor delta's
    annot_vals = pivot.copy().astype(str)
    for r in pivot.index:
        for c in fase_order:
            annot_vals.loc[r, c] = f"{pivot.loc[r, c]:.2f}"
        for dc in ["Δ Ana\n→Opt", "Δ Opt\n→Adv", "Δ Adv\n→Imp"]:
            annot_vals.loc[r, dc] = f"{pivot.loc[r, dc]:+.2f}"

    sns.heatmap(pivot, annot=annot_vals, fmt="",
                cmap="YlOrRd", vmin=0.12, vmax=0.58,
                linewidths=1, ax=ax,
                cbar=idx in [1, 3],
                cbar_kws={"shrink": 0.7} if idx in [1, 3] else {})
    ax.set_title(f"{exp.upper()} ({EXP_LABELS[exp]})",
                 fontsize=13, fontweight="bold", color=palette[exp])
    ax.set_ylabel("" if idx % 2 else "Thema")
    ax.set_xlabel("")
    # Verticale lijn tussen fasen en delta's
    ax.axvline(x=4, color="black", linewidth=2)

plt.suptitle("§4.4.4  Thematisch profiel per architectuur",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════════════
# NUMERIEKE SAMENVATTING
# ═══════════════════════════════════════════════════════════════════════
print("\n" + "═" * 75)
print("  THEMATISCHE TRACEERBAARHEID")
print("═" * 75)

# Top-3 thema's per fase
for fase in fase_order:
    sub = df_thema[df_thema["fase"] == fase]
    top3 = sub.groupby("thema")["score"].mean().nlargest(3)
    print(f"\n  Top-3 thema's in {fase}:")
    for thema, score in top3.items():
        print(f"    {thema:30s}  {score:.3f}")

# Opeenvolgende verschuivingen
print(f"\n{'─' * 75}")
print("  THEMA-VERSCHUIVINGEN PER SCHAKEL:")
print(f"{'─' * 75}")
for delta_label, delta_col in [("Analyse → Opties", "Δ Analyse\n→ Opties"),
                                ("Opties → Advies", "Δ Opties\n→ Advies"),
                                ("Advies → Implementatie", "Δ Advies\n→ Implementatie")]:
    deltas = heat_flow[delta_col].sort_values()
    print(f"\n  {delta_label}:")
    print(f"    Meest krimpend:  {deltas.index[0]:30s} ({deltas.iloc[0]:+.3f})")
    print(f"    Meest groeiend:  {deltas.index[-1]:30s} ({deltas.iloc[-1]:+.3f})")

# Per experiment: thematische consistentie
print(f"\n{'─' * 75}")
print("  THEMATISCHE CONSISTENTIE PER EXPERIMENT:")
print(f"{'─' * 75}")
for exp in EXPERIMENTS:
    sub = df_thema[df_thema["experiment"] == exp]
    pivot = sub.pivot_table(index="thema", columns="fase",
                            values="score", aggfunc="mean")
    shifts = []
    for f_from, f_to in [("Analyse","Opties"),("Opties","Advies"),("Advies","Implementatie")]:
        if f_from in pivot.columns and f_to in pivot.columns:
            shifts.append((pivot[f_to] - pivot[f_from]).abs().mean())
    gem = np.mean(shifts) if shifts else 0
    print(f"  {exp.upper()} ({EXP_LABELS[exp]:12s}): gem. thema-verschuiving per schakel = {gem:.3f}")

print(f"\n  Interpretatie:")
print(f"  - Lage verschuiving = thematisch consistente keten")
print(f"  - Hoge verschuiving = thema-accenten wisselen sterk per fase")
print(f"  - Positieve delta = thema groeit in die schakel")
print(f"  - Negatieve delta = thema verdwijnt in die schakel")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# §4.4.5  Thema-aandeel: hoe verschuift de thematische focus?
# ═══════════════════════════════════════════════════════════════════════
#
# §4.4.4 toont absolute thema-scores. Maar de RELATIEVE verdeling
# vertelt een scherper verhaal: als Prijsstrategie 0.38 scoort in
# de analyse, is dat 15% van de totale thema-aandacht of 30%?
#
# Door per case te normaliseren (12 thema-scores → 100%) zie je direct:
#   "In de analyse lag 15% focus op Prijs, maar in de opties
#    verschoof dat naar Menu & aanbod (14%)."
#
# Visualisaties:
#   1. 100% gestapelde staafdiagram per experiment (4 fasen)
#   2. Bump chart: rankverschuivingen van thema's door de keten
# ═══════════════════════════════════════════════════════════════════════

from matplotlib.colors import to_rgba

# ── Kleurenpalet voor 12 thema's (kwalitatief, goed onderscheidbaar) ──
THEMA_KLEUREN = {
    "Prijsstrategie":              "#E74C3C",
    "Groei & expansie":            "#3498DB",
    "Trends & innovatie":          "#2ECC71",
    "Klantervaring":               "#F39C12",
    "Financieel & kosten":         "#9B59B6",
    "Operatie & logistiek":        "#1ABC9C",
    "Concurrentie & marktpositie": "#E67E22",
    "Duurzaamheid":                "#27AE60",
    "Menu & aanbod":               "#C0392B",
    "Personeel & organisatie":     "#2980B9",
    "Marketing & promotie":        "#D35400",
    "Digitalisering & technologie":"#8E44AD",
}

# ── Bereken genormaliseerde thema-aandelen ──
# Per (experiment, case_id, fase): normaliseer 12 thema-scores → 100%
df_norm = df_thema.copy()
totals = df_norm.groupby(["experiment", "case_id", "fase"])["score"].transform("sum")
df_norm["aandeel"] = df_norm["score"] / totals * 100

print("Controle normalisatie (12 thema's per case moeten optellen tot ~100%):")
check = df_norm.groupby(["experiment", "case_id", "fase"])["aandeel"].sum()
print(f"  min={check.min():.1f}%, max={check.max():.1f}%, n_groepen={len(check)}")

# ═══════════════════════════════════════════════════════════════════════
# FIGUUR 1: 100% gestapelde staafdiagram per experiment
# ═══════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(20, 14))

for idx, exp in enumerate(EXPERIMENTS):
    ax = axes[idx // 2][idx % 2]
    sub = df_norm[df_norm["experiment"] == exp]

    # Pivot: thema × fase → gemiddeld aandeel(%) over cases
    pivot = sub.pivot_table(
        index="thema", columns="fase",
        values="aandeel", aggfunc="mean"
    )[fase_order].reindex(THEMA_ORDER)

    # Teken gestapelde balken
    bottom = np.zeros(len(fase_order))
    for thema in THEMA_ORDER:
        vals = pivot.loc[thema].values
        ax.bar(range(len(fase_order)), vals, bottom=bottom,
               color=THEMA_KLEUREN[thema], edgecolor="white",
               linewidth=0.5, label=thema)
        # Label bij segmenten > 6%
        for i, (v, b) in enumerate(zip(vals, bottom)):
            if v > 6:
                ax.text(i, b + v/2, f"{v:.0f}%",
                        ha="center", va="center", fontsize=8,
                        fontweight="bold", color="white")
        bottom += vals

    ax.set_xticks(range(len(fase_order)))
    ax.set_xticklabels([f"{f}\n(stap {'+'.join(str(s) for s in FASEN[f])})" for f in fase_order],
                       fontsize=10)
    ax.set_ylim(0, 105)
    ax.set_ylabel("Thema-aandeel (%)" if idx % 2 == 0 else "")
    ax.set_title(f"{exp.upper()} ({EXP_LABELS[exp]})",
                 fontsize=13, fontweight="bold", color=palette[exp])
    ax.grid(True, alpha=0.15, axis="y")

    # Legenda alleen bij plot B (rechts boven)
    if idx == 1:
        ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left",
                  fontsize=8, framealpha=0.9)

plt.suptitle("§4.4.5  Thematische focus-verdeling per fase (100% gestapeld)",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════════════
# FIGUUR 2: Bump chart — rankverschuiving van thema's door de keten
# ═══════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(20, 14))

for idx, exp in enumerate(EXPERIMENTS):
    ax = axes[idx // 2][idx % 2]
    sub = df_norm[df_norm["experiment"] == exp]

    pivot = sub.pivot_table(
        index="thema", columns="fase",
        values="aandeel", aggfunc="mean"
    )[fase_order]

    # Bereken rank per fase (1 = hoogste aandeel)
    ranks = pivot.rank(ascending=False, method="min")

    x_pos = range(len(fase_order))
    for thema in THEMA_ORDER:
        r = ranks.loc[thema].values
        kleur = THEMA_KLEUREN[thema]

        is_top5 = ranks.loc[thema].min() <= 5
        lw = 2.5 if is_top5 else 1.0
        alpha = 1.0 if is_top5 else 0.35
        ms = 10 if is_top5 else 4

        ax.plot(x_pos, r, "o-", color=kleur, linewidth=lw,
                markersize=ms, alpha=alpha, zorder=3 if is_top5 else 1)

        # Labels rechts
        if ranks.loc[thema].iloc[-1] <= 7:
            ax.text(len(fase_order) - 1 + 0.15, r[-1],
                    f"{thema[:20]}", fontsize=7, va="center",
                    color=kleur, fontweight="bold" if is_top5 else "normal")
        # Labels links
        if ranks.loc[thema].iloc[0] <= 7:
            ax.text(-0.15, r[0], f"{thema[:20]}", fontsize=7,
                    va="center", ha="right", color=kleur,
                    fontweight="bold" if is_top5 else "normal")

    ax.set_xticks(x_pos)
    ax.set_xticklabels(fase_order, fontsize=11)
    ax.set_ylim(12.5, 0.5)
    ax.set_ylabel("Rang (1 = hoogste aandeel)" if idx % 2 == 0 else "")
    ax.set_title(f"{exp.upper()} ({EXP_LABELS[exp]})",
                 fontsize=13, fontweight="bold", color=palette[exp])
    ax.set_xlim(-0.5, len(fase_order) - 0.5)
    ax.grid(True, alpha=0.2)
    ax.axhline(y=5.5, color="gray", linestyle="--", alpha=0.4, linewidth=1)
    ax.text(len(fase_order) - 0.5, 5.7, "top-5 grens",
            fontsize=7, color="gray", ha="right")

plt.suptitle("§4.4.5  Bump chart: rankverschuiving van thema's door de CoT-keten",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════════════
# NUMERIEKE SAMENVATTING
# ═══════════════════════════════════════════════════════════════════════
print("\n" + "═" * 75)
print("  THEMA-AANDEEL: GROOTSTE VERSCHUIVINGEN")
print("═" * 75)

for exp in EXPERIMENTS:
    sub = df_norm[df_norm["experiment"] == exp]
    pivot = sub.pivot_table(
        index="thema", columns="fase",
        values="aandeel", aggfunc="mean"
    )[fase_order]

    print(f"\n  {exp.upper()} ({EXP_LABELS[exp]}):")
    print(f"  {'Thema':30s}  {'Analyse':>8s}  {'Opties':>8s}  {'Advies':>8s}  {'Implem.':>8s}  {'MaxΔ':>6s}")
    print(f"  {'-'*30}  {'-'*8}  {'-'*8}  {'-'*8}  {'-'*8}  {'-'*6}")

    # Bereken max absolute verschuiving per thema
    pivot["max_delta"] = pivot[fase_order].apply(
        lambda r: r.max() - r.min(), axis=1
    )
    pivot_sorted = pivot.sort_values("max_delta", ascending=False)

    for thema, row_v in pivot_sorted.head(5).iterrows():
        vals = [f"{row_v[f]:5.1f}%" for f in fase_order]
        print(f"  {thema:30s}  {'  '.join(vals)}  {row_v['max_delta']:5.1f}pp")

    # Identificeer de grootste single-stap verschuiving
    biggest_shift_val = 0
    biggest_shift_info = ""
    for i in range(len(fase_order) - 1):
        f1, f2 = fase_order[i], fase_order[i+1]
        delta = (pivot[f2] - pivot[f1]).abs()
        max_thema = delta.idxmax()
        if delta[max_thema] > biggest_shift_val:
            biggest_shift_val = delta[max_thema]
            d_signed = pivot.loc[max_thema, f2] - pivot.loc[max_thema, f1]
            biggest_shift_info = (
                f"{max_thema}: {d_signed:+.1f}pp van {f1} naar {f2}"
            )
    print(f"  \u2192 Grootste verschuiving: {biggest_shift_info}")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# §4.4.5b  Bump chart: totaaloverzicht alle experimenten samen
# ═══════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(14, 10))
ax.set_facecolor("#FAFAFA")
fig.patch.set_facecolor("white")

# Gemiddeld aandeel over ALLE experimenten en cases
pivot_all = df_norm.pivot_table(
    index="thema", columns="fase",
    values="aandeel", aggfunc="mean"
)[fase_order]
ranks_all = pivot_all.rank(ascending=False, method="min")

# ── Subtiele horizontale lijnen per rang ──
for rank in range(1, 13):
    ax.axhline(y=rank, color="#E8E8E8", linewidth=0.5, zorder=0)

# ── Top-5 zone ──
ax.axhspan(0.4, 5.5, color="#E8F5E9", alpha=0.2, zorder=0)
ax.axhline(y=5.5, color="#BDBDBD", linestyle="--", alpha=0.5, linewidth=0.8)

# X-posities: ruimte in het midden, labels daarbuiten
x_pos = np.array([0, 1, 2, 3], dtype=float)

# ── Teken lijnen ──
for thema in THEMA_ORDER:
    r = ranks_all.loc[thema].values
    kleur = THEMA_KLEUREN[thema]
    is_top5 = ranks_all.loc[thema].min() <= 5

    lw    = 3.0 if is_top5 else 1.0
    alpha = 1.0 if is_top5 else 0.22
    ms    = 11  if is_top5 else 4

    ax.plot(x_pos, r, "o-", color=kleur, linewidth=lw,
            markersize=ms, alpha=alpha, zorder=3 if is_top5 else 1,
            solid_capstyle="round")

# ── Labels: geplaatst BUITEN de plotrand via fig-transform ──
def _place_labels(ax, ranks_all, side):
    """Plaats labels links of rechts met anti-overlap jitter."""
    fase_col = fase_order[0] if side == "left" else fase_order[-1]
    items = []
    for thema in THEMA_ORDER:
        r = float(ranks_all.loc[thema, fase_col])
        is_top5 = ranks_all.loc[thema].min() <= 5
        items.append((r, thema, is_top5))
    items.sort(key=lambda x: x[0])

    # Greedy anti-overlap: duw conflicten op en neer
    min_gap = 0.58
    positions = [it[0] for it in items]
    # Forward pass: duw naar beneden
    for i in range(1, len(positions)):
        if positions[i] - positions[i-1] < min_gap:
            positions[i] = positions[i-1] + min_gap
    # Backward pass: als we te ver naar beneden zijn geduwd, corrigeer
    for i in range(len(positions)-2, -1, -1):
        if positions[i+1] - positions[i] < min_gap:
            positions[i] = positions[i+1] - min_gap

    x_data = x_pos[0] if side == "left" else x_pos[-1]
    x_txt  = x_data - 0.25 if side == "left" else x_data + 0.25
    ha = "right" if side == "left" else "left"

    for (rank_val, thema, is_top5), y_pos in zip(items, positions):
        kleur = THEMA_KLEUREN[thema]
        r_int = int(rank_val)
        txt = f"#{r_int}  {thema}" if side == "left" else f"{thema}  #{r_int}"
        fs = 10 if is_top5 else 8
        fw = "bold" if is_top5 else "normal"
        al = 1.0 if is_top5 else 0.50

        # Connector lijn van datapunt naar label
        if abs(y_pos - rank_val) > 0.25:
            ax.annotate("", xy=(x_data, rank_val),
                        xytext=(x_txt, y_pos),
                        arrowprops=dict(arrowstyle="-", color=kleur,
                                        alpha=0.15, lw=0.5))

        ax.text(x_txt, y_pos, txt, fontsize=fs, fontweight=fw,
                color=kleur, alpha=al, va="center", ha=ha)

_place_labels(ax, ranks_all, "left")
_place_labels(ax, ranks_all, "right")

# ── Assen: minimaal, geen overlap met labels ──
ax.set_xticks(x_pos)
ax.set_xticklabels(
    [f"{f}\n(stap {'+'.join(str(s) for s in FASEN[f])})" for f in fase_order],
    fontsize=12, fontweight="bold"
)
ax.set_ylim(13.0, 0.2)
ax.set_yticks([])                      # geen y-tick nummers
ax.set_ylabel("")                       # geen y-label
ax.set_xlim(-0.35, 3.35)               # strakke x-limieten
ax.tick_params(axis="y", length=0)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_color("#CCCCCC")

# ── Titel ──
ax.set_title(
    "§4.4.5  Rankverschuiving thema's door de CoT-keten\n"
    "(gemiddeld over alle experimenten)",
    fontsize=15, fontweight="bold", pad=18
)

plt.tight_layout()
plt.subplots_adjust(left=0.28, right=0.72)   # ruime marges voor labels
plt.show()

# ── Numeriek ──
print("\nRangverschuivingen (alle experimenten gecombineerd):")
print(f"  {'Thema':30s}  {'Analyse':>8s}  {'Opties':>8s}  {'Advies':>8s}  {'Implem.':>8s}  {'Δ rang':>6s}")
print(f"  {'-'*30}  {'-'*8}  {'-'*8}  {'-'*8}  {'-'*8}  {'-'*6}")
for thema in THEMA_ORDER:
    r = ranks_all.loc[thema]
    delta = int(r.iloc[0] - r.iloc[-1])
    vals = [f"{int(r[f]):>5d}" for f in fase_order]
    arrow = "↑" if delta > 0 else ("↓" if delta < 0 else "=")
    print(f"  {thema:30s}  {'     '.join(vals)}   {arrow}{abs(delta)}")


In [0]:
# Verbeterde bump chart: movers opvallend, stabiele thema's op achtergrond
import os

FIG_DIR = "/Workspace/Users/rgoedings@hanos.nl/AI_op_het_menu/figures"
os.makedirs(FIG_DIR, exist_ok=True)

# Bereken gemiddeld aandeel per thema per fase
norm_avg = df_norm.groupby(["thema", "fase"])["aandeel"].mean().reset_index()
fase_order_bump = [f for f in ["Analyse", "Opties", "Advies", "Implementatie"] if f in norm_avg["fase"].unique()]

# Bereken rang per fase
rank_data = []
for fase in fase_order_bump:
    fase_df = norm_avg[norm_avg["fase"] == fase].sort_values("aandeel", ascending=False).reset_index(drop=True)
    fase_df["rang"] = range(1, len(fase_df) + 1)
    rank_data.append(fase_df)
df_ranks = pd.concat(rank_data)

# Bereken totale rangverandering per thema (max - min)
thema_volatility = {}
for thema in df_ranks["thema"].unique():
    sub = df_ranks[df_ranks["thema"] == thema]
    ranks = sub["rang"].dropna().values
    thema_volatility[thema] = max(ranks) - min(ranks) if len(ranks) >= 2 else 0

# Threshold: thema's met >=4 rang verschil zijn 'movers'
THRESHOLD = 4
movers = {t for t, v in thema_volatility.items() if v >= THRESHOLD}
stable = {t for t, v in thema_volatility.items() if v < THRESHOLD}

print(f"Movers (≥{THRESHOLD} rangen): {sorted(movers)}")
print(f"Stabiel (<{THRESHOLD} rangen): {sorted(stable)}")

# Plot
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_facecolor("#FAFAFA")
fig.patch.set_facecolor("white")

for rank in range(1, 13):
    ax.axhline(y=rank, color="#EEEEEE", linewidth=0.5, zorder=0)

ax.axhspan(0.4, 5.5, color="#E8F5E9", alpha=0.12, zorder=0)

# Stabiele thema's: grijs, dun, op achtergrond
for thema in sorted(stable):
    sub = df_ranks[df_ranks["thema"] == thema].set_index("fase").reindex(fase_order_bump)
    ax.plot(fase_order_bump, sub["rang"].values, "o-",
            color="#BDBDBD", linewidth=1.0, markersize=4, alpha=0.35, zorder=1)
    if not np.isnan(sub["rang"].values[-1]):
        ax.text(len(fase_order_bump) - 0.85, sub["rang"].values[-1], thema,
                ha="left", va="center", fontsize=7.5, color="#9E9E9E", alpha=0.7)

# Movers: dik, gekleurd, op voorgrond
for thema in sorted(movers):
    sub = df_ranks[df_ranks["thema"] == thema].set_index("fase").reindex(fase_order_bump)
    color = THEMA_KLEUREN.get(thema, "#333333")
    start_rank = sub["rang"].values[0]
    end_rank = sub["rang"].values[-1]
    delta = int(start_rank - end_rank)  # positief = stijger (lagere rang = hoger)
    arrow = "↑" if delta > 0 else "↓"

    ax.plot(fase_order_bump, sub["rang"].values, "o-",
            color=color, linewidth=3.5, markersize=12, alpha=0.95, zorder=3,
            markeredgecolor="white", markeredgewidth=1.5)

    if not np.isnan(sub["rang"].values[0]):
        ax.text(-0.12, sub["rang"].values[0], thema,
                ha="right", va="center", fontsize=10, color=color, fontweight="bold")
    if not np.isnan(sub["rang"].values[-1]):
        ax.text(len(fase_order_bump) - 0.88, sub["rang"].values[-1],
                f"{thema} ({arrow}{abs(delta)})",
                ha="left", va="center", fontsize=10, color=color, fontweight="bold")

ax.set_ylabel("Rang (1 = hoogste aandeel)", fontsize=12)
ax.set_title("Thematische focusverschuiving door de redeneerketen\n"
             "(dikke lijnen = grote rangverschuiving ≥4 posities)", fontsize=14, fontweight="bold")
ax.invert_yaxis()
ax.set_xticks(range(len(fase_order_bump)))
ax.set_xticklabels(fase_order_bump, fontsize=13)
ax.set_ylim(12.5, 0.5)
ax.grid(False)

plt.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_4_4c_bump_chart.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print(f"\n✅ Opgeslagen: {FIG_DIR}/fig_4_4c_bump_chart.png")

In [0]:
# Export §4.4.2 heatmaps naar figures map
import os

FIG_DIR = "/Workspace/Users/rgoedings@hanos.nl/AI_op_het_menu/figures"
os.makedirs(FIG_DIR, exist_ok=True)

# ─── Figuur 1: 2x2 retentie + adjacente heatmaps ───────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 13))

# 1a. Cumulatieve retentie heatmap
ax = axes[0, 0]
ret_pivot = df_retention.pivot_table(
    index="experiment", columns="step_num",
    values="cumulative_retention", aggfunc="mean"
).reindex(EXPERIMENTS)
sns.heatmap(ret_pivot, annot=True, fmt=".2f", cmap="RdYlGn",
            vmin=0.4, vmax=0.9, linewidths=0.5, ax=ax,
            xticklabels=[f"{int(c)}" for c in ret_pivot.columns],
            yticklabels=[f"{e.upper()} ({EXP_LABELS[e]})" for e in EXPERIMENTS])
ax.set_title("Cumulatieve informatieretentie per experiment\n(hoeveel van stap 1..N-1 zit in stap N?)")
ax.set_xlabel("Stap")
ax.set_ylabel("")

# 1b. Adjacente overlap heatmap
ax = axes[0, 1]
adj_pivot = df_flow.pivot_table(
    index="experiment", columns="from_step",
    values="similarity", aggfunc="mean"
).reindex(EXPERIMENTS)
sns.heatmap(adj_pivot, annot=True, fmt=".2f", cmap="RdYlGn",
            vmin=0.3, vmax=0.8, linewidths=0.5, ax=ax,
            xticklabels=[f"{int(c)}→{int(c)+1}" for c in adj_pivot.columns],
            yticklabels=[f"{e.upper()}" for e in EXPERIMENTS])
ax.set_title("Adjacente informatie-overdracht per experiment\n(cosine sim stap N → stap N+1)")
ax.set_xlabel("Overgang")
ax.set_ylabel("")

# 1c. Delta retentie vs gemiddelde
ax = axes[1, 0]
ret_mean_col = ret_pivot.mean(axis=0)
ret_delta = ret_pivot.subtract(ret_mean_col, axis=1)
sns.heatmap(ret_delta, annot=True, fmt="+.3f", cmap="RdYlGn", center=0,
            vmin=-0.06, vmax=0.06, linewidths=0.5, ax=ax,
            xticklabels=[f"{int(c)}" for c in ret_delta.columns],
            yticklabels=[f"{e.upper()} ({EXP_LABELS[e]})" for e in EXPERIMENTS])
ax.set_title("Δ Retentie vs gemiddelde\n(groen = behoudt meer info dan gemiddeld)")
ax.set_xlabel("Stap")
ax.set_ylabel("")

# 1d. Delta adjacente overdracht vs gemiddelde
ax = axes[1, 1]
adj_mean_col = adj_pivot.mean(axis=0)
adj_delta = adj_pivot.subtract(adj_mean_col, axis=1)
sns.heatmap(adj_delta, annot=True, fmt="+.3f", cmap="RdYlGn", center=0,
            vmin=-0.06, vmax=0.06, linewidths=0.5, ax=ax,
            xticklabels=[f"{int(c)}→{int(c)+1}" for c in adj_delta.columns],
            yticklabels=[f"{e.upper()}" for e in EXPERIMENTS])
ax.set_title("Δ Adjacente overdracht vs gemiddelde\n(groen = sterkere koppeling dan gemiddeld)")
ax.set_xlabel("Overgang")
ax.set_ylabel("")

plt.suptitle("§4.4.2 Informatiestroom per experiment: heatmaps",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_4_4d_retentie_heatmaps.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.close(fig)
print("✅ fig_4_4d_retentie_heatmaps.png")

# ─── Figuur 2: Per-experiment 11×11 similarity matrices ────────────────
from utils.steps import STEP_LABELS
step_short = {str(i): STEP_LABELS.get(str(i), f"Stap {i}")[:15] for i in range(1, 12)}
step_labels_11 = [step_short.get(str(i), f"Stap {i}")[:15] for i in range(1, 12)]

fig, axes_m = plt.subplots(2, 2, figsize=(16, 14))

for idx, exp in enumerate(EXPERIMENTS):
    ax = axes_m[idx // 2, idx % 2]
    exp_matrix = np.zeros((11, 11))
    exp_counts = np.zeros((11, 11))

    if exp in full_matrices:
        for step_nums_m, sim_mat in full_matrices[exp]:
            for i, si in enumerate(step_nums_m):
                for j, sj in enumerate(step_nums_m):
                    exp_matrix[si-1, sj-1] += sim_mat[i, j]
                    exp_counts[si-1, sj-1] += 1

    exp_counts[exp_counts == 0] = 1
    exp_matrix /= exp_counts

    im = ax.imshow(exp_matrix, cmap="YlOrRd", vmin=0.3, vmax=0.9, aspect="auto")
    for i in range(11):
        for j in range(11):
            val = exp_matrix[i, j]
            if i != j and val > 0:
                color = "white" if val > 0.7 else "black"
                ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=5.5, color=color)
    ax.set_xticks(range(11))
    ax.set_yticks(range(11))
    ax.set_xticklabels([str(i+1) for i in range(11)], fontsize=7)
    ax.set_yticklabels(step_labels_11, fontsize=6)
    ax.set_title(f"{exp.upper()} ({EXP_LABELS[exp]})", fontsize=12, fontweight="bold",
                 color=palette[exp])

plt.suptitle("§4.4.2 Stap-stap similarity matrices per experiment",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
fig.savefig(f"{FIG_DIR}/fig_4_4e_similarity_matrices.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.close(fig)
print("✅ fig_4_4e_similarity_matrices.png")

print(f"\nOpgeslagen in: {FIG_DIR}")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# §4.4.5c  Diagnose: matcht de thema-beschrijving elke fase even goed?
# ═══════════════════════════════════════════════════════════════════════
#
# PROBLEEM: De thema-beschrijvingen zijn éénmalig gedefinieerd, maar
# thema's manifesteren zich anders per fase:
#   - Analyse: "klanttevredenheid is laag" (diagnostisch)
#   - Opties:  "klantloyaliteitsprogramma overwegen" (strategisch)
#   - Advies:  "investeer in gastvrijheidstraining" (prescriptief)
#   - Implem.: "Q2: mystery guest inzetten, training week 1-4" (operationeel)
#
# Hierdoor kan een thema dat wél aanwezig is, toch laag scoren omdat
# de embedding-match afneemt bij ander taalgebruik.
# ═══════════════════════════════════════════════════════════════════════

# ── Stap 1: Bekijk voorbeeld-teksten per fase voor één case ──
print("=" * 75)
print("  VOORBEELD: tekst per fase (experiment A, case 1)")
print("=" * 75)

row0 = df_run1[df_run1["experiment"] == "a"].iloc[0]
steps = extract_steps(row0["answer"])

for fase_naam, fase_stappen in FASEN.items():
    teksten = []
    for s in fase_stappen:
        if str(s) in steps:
            t = _step_to_text(steps[str(s)])
            if t and len(t.strip()) > 20:
                teksten.append(t)
    if teksten:
        gecombineerd = " ".join(teksten)
        # Toon eerste 300 tekens
        print(f"\n── {fase_naam} (stap {'+'.join(str(s) for s in fase_stappen)}) ──")
        print(f"   {gecombineerd[:300]}...")
        print(f"   [{len(gecombineerd)} tekens totaal]")

# ── Stap 2: Meet hoe goed elke thema-beschrijving matcht per fase ──
# Als de beschrijving fase-neutraal is, verwacht je gelijke match.
# Als niet: analyse-taal matcht beter dan implementatie-taal.
print("\n" + "=" * 75)
print("  THEMA-MATCH CONSISTENTIE: score per fase (gem. over alle cases)")
print("=" * 75)

# Gebruik df_thema (al berekend in §4.4.4)
consistentie = df_thema.pivot_table(
    index="thema", columns="fase",
    values="score", aggfunc="mean"
)[fase_order].reindex(THEMA_ORDER)

print(f"\n  {'Thema':30s}  {'Analyse':>9s}  {'Opties':>9s}  {'Advies':>9s}  {'Implem.':>9s}  {'Range':>7s}  {'Bias?':>6s}")
print(f"  {'-'*30}  {'-'*9}  {'-'*9}  {'-'*9}  {'-'*9}  {'-'*7}  {'-'*6}")

bias_themas = []
for thema in THEMA_ORDER:
    vals = consistentie.loc[thema]
    score_range = vals.max() - vals.min()
    # Bias = score daalt systematisch van Analyse naar Implementatie
    trend = vals.iloc[0] - vals.iloc[-1]  # positief = analyse > implementatie
    bias = "⚠️" if trend > 0.03 else ("✓" if abs(trend) < 0.01 else "~")
    if trend > 0.03:
        bias_themas.append((thema, trend, score_range))
    
    print(f"  {thema:30s}  {vals.iloc[0]:9.3f}  {vals.iloc[1]:9.3f}  {vals.iloc[2]:9.3f}  {vals.iloc[3]:9.3f}  {score_range:7.3f}  {bias}")

print(f"\n  Thema's met analyse-bias (Analyse-score > Implementatie-score + 0.03):")
for thema, trend, rng in sorted(bias_themas, key=lambda x: -x[1]):
    print(f"    ⚠️ {thema:30s}  Δ={trend:+.3f}  (range={rng:.3f})")

if not bias_themas:
    print("    Geen significante bias gevonden.")

# ── Stap 3: Concreet voorbeeld — welke zinnen matchen per fase? ──
# Pak 'Klantervaring' als voorbeeld en toon de best-matchende zinnen
print("\n" + "=" * 75)
print("  DEEP DIVE: 'Klantervaring' — wat matcht er per fase?")
print("=" * 75)

import re

thema_idx = thema_namen.index("Klantervaring")
emb_klant = emb_themas[thema_idx:thema_idx+1]  # (1, 384)

for _, row in df_run1[df_run1["experiment"] == "a"].head(2).iterrows():
    steps = extract_steps(row["answer"])
    if not steps:
        continue
    print(f"\n  Case: {row['case_id']}")
    
    for fase_naam, fase_stappen in FASEN.items():
        teksten = []
        for s in fase_stappen:
            if str(s) in steps:
                t = _step_to_text(steps[str(s)])
                if t and len(t.strip()) > 20:
                    teksten.append(t)
        if not teksten:
            continue
        
        # Split in zinnen, embed elk, meet similarity
        fase_tekst = " ".join(teksten)
        zinnen = [z.strip() for z in re.split(r'[.!?\n]', fase_tekst) if len(z.strip()) > 15]
        if not zinnen:
            continue
        
        emb_zinnen = embedding_model.encode(zinnen, convert_to_numpy=True, normalize_embeddings=True)
        sims = cosine_similarity(emb_klant, emb_zinnen)[0]
        
        # Top-3 best matchende zinnen
        top_idx = sims.argsort()[-3:][::-1]
        print(f"\n    {fase_naam} (best match met 'Klantervaring'):")
        for i in top_idx:
            zin_preview = zinnen[i][:100]
            print(f"      [{sims[i]:.3f}] {zin_preview}")
    break  # alleen eerste case


### 4.4.1 Stap-niveau vergelijking tussen architecturen
> **Adresseert:** *Waar zitten de verschillen in het redeneerproces tussen de vier architecturen?*

De totaalscores (§4.3.1) laten kleine verschillen zien. Deze analyse duikt dieper in door **elke individuele redeneerstap** te vergelijken:

1. **Stap-investering** — Hoeveel tekst besteedt elke architectuur aan elke stap? (meer = dieper uitgewerkt)
2. **Stap-divergentie** — Op welke stappen wijken de architecturen het meest af van elkaar? (cosine similarity)
3. **Self-Refine effect** (alleen D) — Welke stappen verbeteren het meest na de critique-fase?

Deze analyse gebruikt `df_steps` met **{n_steps} individueel geëxtraheerde stappen** uit alle runs.

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# 4.2.3 Stap-niveau vergelijking tussen architecturen
# ═══════════════════════════════════════════════════════════════════════

from utils.steps import COT_STEPS, STEP_LABELS
from itertools import combinations

# Gebruik alleen run 1 voor eerlijke vergelijking (zelfde als judge)
df_steps_r1 = df_steps[(df_steps["run"] == 1) & (df_steps["phase"] == "final")].copy()
step_labels_ordered = [label for _, label in COT_STEPS]

print(f"Stap-analyse: {len(df_steps_r1)} stappen (run 1, final phase)")
print(f"Experimenten: {sorted(df_steps_r1['experiment'].unique())}")
print(f"Stappen per experiment: {df_steps_r1.groupby('experiment')['step_num'].count().to_dict()}")

# ───────────────────────────────────────────────────────────────────────
# 1. STAP-INVESTERING: tekstlengte per stap per experiment
# ───────────────────────────────────────────────────────────────────────
print("\n=== 1. Stap-investering (gemiddelde tekstlengte in karakters) ===")

length_pivot = df_steps_r1.pivot_table(
    index="step_num", columns="experiment",
    values="step_chars", aggfunc="mean"
).reindex(range(1, 12))
length_pivot = length_pivot[EXPERIMENTS]
length_pivot.index = [STEP_LABELS.get(str(i), f"Stap {i}") for i in range(1, 12)]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Heatmap
sns.heatmap(length_pivot, annot=True, fmt=".0f", cmap="YlOrRd",
            linewidths=0.5, ax=axes[0])
axes[0].set_title("Tekstlengte per stap per experiment (karakters)")
axes[0].set_ylabel("")
axes[0].set_xlabel("Experiment")

# Lijn-grafiek: zelfde data maar met duidelijke patronen
for exp in EXPERIMENTS:
    if exp in length_pivot.columns:
        axes[1].plot(range(1, 12), length_pivot[exp].values,
                     marker="o", label=f"Experiment {exp.upper()}",
                     linewidth=2, color=palette.get(exp))
axes[1].set_xlabel("Stap")
axes[1].set_ylabel("Gemiddelde tekstlengte (karakters)")
axes[1].set_title("Investering per redeneerstap")
axes[1].set_xticks(range(1, 12))
axes[1].set_xticklabels([f"{i}" for i in range(1, 12)], fontsize=8)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Welke stappen krijgen het meest/minst aandacht?
print("\nStappen met GROOTSTE verschil tussen experimenten:")
length_pivot["range"] = length_pivot[EXPERIMENTS].max(axis=1) - length_pivot[EXPERIMENTS].min(axis=1)
for _, row in length_pivot.nlargest(3, "range").iterrows():
    vals = {e: f"{row[e]:.0f}" for e in EXPERIMENTS if pd.notna(row[e])}
    print(f"  {row.name}: range={row['range']:.0f} — {vals}")

# ───────────────────────────────────────────────────────────────────────
# 2. STAP-DIVERGENTIE: cosine similarity per stap tussen experimenten
# ───────────────────────────────────────────────────────────────────────
print("\n=== 2. Stap-divergentie (cosine similarity per stap) ===")
print("Hogere similarity = experimenten produceren vergelijkbare tekst voor die stap.")
print("Lagere similarity = experimenten wijken af — hier zit het verschil.\n")

# Bereken per stap, per case: cosine similarity tussen elk paar experimenten
divergence_rows = []

for case_id in sorted(df_steps_r1["case_id"].unique()):
    for step_num in range(1, 12):
        texts_by_exp = {}
        for exp in EXPERIMENTS:
            mask = (
                (df_steps_r1["experiment"] == exp) &
                (df_steps_r1["case_id"] == case_id) &
                (df_steps_r1["step_num"] == step_num)
            )
            rows_found = df_steps_r1[mask]
            if not rows_found.empty:
                texts_by_exp[exp] = rows_found.iloc[0]["step_text"]
        
        if len(texts_by_exp) < 2:
            continue
        
        # Encode alle teksten
        exp_list = sorted(texts_by_exp.keys())
        texts = [texts_by_exp[e] for e in exp_list]
        embs = embedding_model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
        
        # Pairwise similarity
        for i, j in combinations(range(len(exp_list)), 2):
            sim = float(cosine_similarity([embs[i]], [embs[j]])[0][0])
            divergence_rows.append({
                "case_id": case_id,
                "step_num": step_num,
                "step_label": STEP_LABELS.get(str(step_num), f"Stap {step_num}"),
                "pair": f"{exp_list[i].upper()} vs {exp_list[j].upper()}",
                "similarity": sim,
            })

df_divergence = pd.DataFrame(divergence_rows)

# Heatmap: gemiddelde similarity per stap per experimentpaar
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Links: similarity per stap (gemiddeld over alle paren)
step_sim_avg = df_divergence.groupby("step_num")["similarity"].mean().reindex(range(1, 12))
step_labels_short = [STEP_LABELS.get(str(i), f"{i}")[:20] for i in range(1, 12)]

colors_sim = ["#e74c3c" if v < step_sim_avg.mean() else "#2ecc71" for v in step_sim_avg.values]
axes[0].barh(range(1, 12), step_sim_avg.values, color=colors_sim)
axes[0].set_yticks(range(1, 12))
axes[0].set_yticklabels(step_labels_short, fontsize=8)
axes[0].set_xlabel("Gemiddelde cosine similarity")
axes[0].set_title("Waar wijken experimenten het meest af?\n(lager = meer verschil)")
axes[0].axvline(x=step_sim_avg.mean(), color="gray", linestyle="--", alpha=0.5,
                label=f"gemiddeld ({step_sim_avg.mean():.3f})")
axes[0].legend(fontsize=8)
axes[0].invert_yaxis()
for i, v in enumerate(step_sim_avg.values):
    axes[0].text(v + 0.002, i + 1, f"{v:.3f}", va="center", fontsize=8)

# Rechts: heatmap per paar per stap
div_pivot = df_divergence.pivot_table(
    index="step_num", columns="pair", values="similarity", aggfunc="mean"
)
div_pivot.index = [STEP_LABELS.get(str(i), f"Stap {i}")[:20] for i in div_pivot.index]
sns.heatmap(div_pivot, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.7, vmax=1.0,
            linewidths=0.5, ax=axes[1])
axes[1].set_title("Cosine similarity per stap per experimentpaar")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

# Top-3 meest divergente stappen
print("\nMeest divergente stappen (laagste similarity):")
for step_num in step_sim_avg.nsmallest(3).index:
    label = STEP_LABELS.get(str(step_num), f"Stap {step_num}")
    val = step_sim_avg[step_num]
    # Welk paar wijkt het meest af?
    step_pairs = df_divergence[df_divergence["step_num"] == step_num]
    min_pair = step_pairs.loc[step_pairs["similarity"].idxmin()]
    print(f"  Stap {step_num} ({label}): sim={val:.3f} — meest afwijkend: {min_pair['pair']} ({min_pair['similarity']:.3f})")

# ───────────────────────────────────────────────────────────────────────
# 3. SELF-REFINE EFFECT: generate vs refine per stap (alleen D)
# ───────────────────────────────────────────────────────────────────────
print("\n=== 3. Self-Refine effect: wat verandert er na critique? ===")

df_d_gen = df_steps[(df_steps["experiment"] == "d") & (df_steps["phase"] == "generate") & (df_steps["run"] == 1)]
df_d_ref = df_steps[(df_steps["experiment"] == "d") & (df_steps["phase"] == "final") & (df_steps["run"] == 1)]

if not df_d_gen.empty and not df_d_ref.empty:
    refine_rows = []
    for case_id in sorted(df_d_gen["case_id"].unique()):
        for step_num in range(1, 12):
            gen_row = df_d_gen[(df_d_gen["case_id"] == case_id) & (df_d_gen["step_num"] == step_num)]
            ref_row = df_d_ref[(df_d_ref["case_id"] == case_id) & (df_d_ref["step_num"] == step_num)]
            
            if gen_row.empty or ref_row.empty:
                continue
            
            gen_text = gen_row.iloc[0]["step_text"]
            ref_text = ref_row.iloc[0]["step_text"]
            
            # Lengte-verschil
            len_change = len(ref_text) - len(gen_text)
            len_pct = (len_change / max(len(gen_text), 1)) * 100
            
            # Cosine similarity tussen generate en refine
            embs = embedding_model.encode([gen_text, ref_text],
                                          convert_to_numpy=True, normalize_embeddings=True)
            sim = float(cosine_similarity([embs[0]], [embs[1]])[0][0])
            
            refine_rows.append({
                "case_id": case_id,
                "step_num": step_num,
                "step_label": STEP_LABELS.get(str(step_num), f"Stap {step_num}"),
                "gen_chars": len(gen_text),
                "ref_chars": len(ref_text),
                "len_change": len_change,
                "len_change_pct": len_pct,
                "gen_ref_similarity": sim,
            })
    
    df_refine = pd.DataFrame(refine_rows)
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    # Links: lengte-verandering per stap
    len_by_step = df_refine.groupby("step_num")["len_change_pct"].mean()
    colors_len = ["#2ecc71" if v > 0 else "#e74c3c" for v in len_by_step.values]
    axes[0].barh(range(1, 12), len_by_step.reindex(range(1, 12)).fillna(0).values, color=colors_len)
    axes[0].set_yticks(range(1, 12))
    axes[0].set_yticklabels(step_labels_short, fontsize=8)
    axes[0].set_xlabel("Lengte-verandering (%)")
    axes[0].set_title("Self-Refine: lengte-verandering per stap\n(groen=langer, rood=korter)")
    axes[0].axvline(x=0, color="gray", linestyle="-", alpha=0.3)
    axes[0].invert_yaxis()
    
    # Midden: similarity generate vs refine per stap
    sim_by_step = df_refine.groupby("step_num")["gen_ref_similarity"].mean()
    colors_sim_d = ["#e74c3c" if v < sim_by_step.mean() else "#3498db" for v in sim_by_step.values]
    axes[1].barh(range(1, 12), sim_by_step.reindex(range(1, 12)).fillna(0).values, color=colors_sim_d)
    axes[1].set_yticks(range(1, 12))
    axes[1].set_yticklabels(step_labels_short, fontsize=8)
    axes[1].set_xlabel("Cosine similarity (generate → refine)")
    axes[1].set_title("Hoeveel verandert elke stap na critique?\n(lager = meer herschreven)")
    axes[1].set_xlim(0.5, 1.0)
    axes[1].invert_yaxis()
    for i, v in enumerate(sim_by_step.reindex(range(1, 12)).fillna(0).values):
        if v > 0:
            axes[1].text(v + 0.005, i + 1, f"{v:.3f}", va="center", fontsize=8)
    
    # Rechts: heatmap per case
    refine_heatmap = df_refine.pivot_table(
        index="step_num", columns="case_id",
        values="gen_ref_similarity", aggfunc="mean"
    )
    refine_heatmap.index = [STEP_LABELS.get(str(i), f"Stap {i}")[:18] for i in refine_heatmap.index]
    sns.heatmap(refine_heatmap, annot=True, fmt=".3f", cmap="RdYlGn",
                vmin=0.7, vmax=1.0, linewidths=0.5, ax=axes[2])
    axes[2].set_title("Generate → Refine similarity per stap per case")
    axes[2].set_ylabel("")
    
    plt.tight_layout()
    plt.show()
    
    # Meest herschreven stappen
    print("\nMeest herschreven stappen na critique (laagste similarity):")
    for _, row in df_refine.groupby("step_num").agg(
        sim=("gen_ref_similarity", "mean"),
        len_pct=("len_change_pct", "mean")
    ).nsmallest(3, "sim").iterrows():
        label = STEP_LABELS.get(str(_), f"Stap {_}")
        print(f"  Stap {_} ({label}): sim={row['sim']:.3f}, lengte {row['len_pct']:+.0f}%")
else:
    print("Geen generate-fase data beschikbaar voor experiment D.")

## 4.5 Consistentie en diversiteit
> **Adresseert:** *Hoe stabiel/reproduceerbaar zijn de adviezen over meerdere runs?*

De consistentie wordt gemeten met cosine similarity op sentence embeddings (paraphrase-multilingual-MiniLM-L12-v2), toegepast op 5 runs per experiment per case.

**Stabiliteit** = 2/(R(R−1)) × Σ sim(Tᵢ, Tⱼ) — gemiddelde pairwise cosine similarity tussen alle runs.

Een hogere waarde duidt op meer reproduceerbare output. De analyses hieronder tonen:
- Stabiliteitsplot per architectuur (bar chart)
- Run-level consistentiematrices (heatmaps per case)
- Cross-experiment similarity (vergelijking tussen A, B en C)

In [0]:
# Stabiliteit per architectuur — alle experimenten inclusief D

fig, ax = plt.subplots(figsize=(8, 5))

exp_names = stability_summary["experiment"].tolist()
stab_vals = stability_summary["stability"].tolist()
colors = [palette.get(e, "#999") for e in exp_names]

# Teken bars (NaN wordt 0 voor weergave)
bar_vals = [v if pd.notna(v) else 0 for v in stab_vals]
bars = ax.bar(exp_names, bar_vals, color=colors)

# Annotaties: waarde of N/A
for i, (exp, val) in enumerate(zip(exp_names, stab_vals)):
    if pd.isna(val):
        ax.text(i, 0.03, "N/A\n(1 run)", ha="center", fontsize=9,
                color="gray", style="italic")
        bars[i].set_alpha(0.3)  # maak de bar transparant
    else:
        ax.text(i, val + 0.01, f"{val:.3f}", ha="center", fontweight="bold")

ax.set_title("Stabiliteit per architectuur")
ax.set_ylabel("Cosine similarity")
ax.set_xlabel("Experiment")
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

### 4.5.1 Run-level consistentiematrices
Pairwise cosine similarity tussen alle runs per experiment en per case. De heatmaps hieronder tonen hoe vergelijkbaar de gegenereerde adviezen zijn bij herhaalde uitvoering met dezelfde input. Hoge similarity (≥ 0.85) duidt op stabiele, reproduceerbare output.

In [0]:
# Run-level consistentiematrix per case

def run_consistency_matrix_case(experiment, case_id):
    
    df_case = df_runs[
        (df_runs["experiment"] == experiment) &
        (df_runs["case_id"] == case_id)
    ].sort_values("run")

    texts = df_case["answer"].fillna("").tolist()
    runs = df_case["run"].tolist()

    embeddings = embedding_model.encode(
        texts,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    sim = cosine_similarity(embeddings)

    sim_df = pd.DataFrame(
        sim,
        index=[f"run_{r}" for r in runs],
        columns=[f"run_{r}" for r in runs]
    )

    return sim_df

In [0]:
experiments = EXPERIMENTS
cases = sorted(df_runs["case_id"].unique())

for exp in experiments:

    # Bepaal aantal runs voor dit experiment
    n_runs = df_runs[df_runs["experiment"] == exp]["run"].nunique()
    
    if n_runs < 2:
        print(f"\nExperiment {exp.upper()}: slechts {n_runs} run — consistentiematrix niet beschikbaar (minimaal 2 runs nodig).")
        continue

    fig, axes = plt.subplots(1, len(cases), figsize=(20, 4))

    for i, case in enumerate(cases):

        sim_case = run_consistency_matrix_case(exp, case)

        sns.heatmap(
            sim_case,
            annot=True,
            fmt=".3f",
            cmap="coolwarm",
            vmin=0.7,
            vmax=1,
            ax=axes[i]
        )

        axes[i].set_title(case)

    plt.suptitle(f"Run consistentie per case — Experiment {exp.upper()} ({n_runs} runs)")
    plt.tight_layout()
    plt.show()

### 4.6 Efficiëntie en aanvullende consistentievisualisaties
> **Adresseert:** *Hoe efficiënt zijn de architecturen qua tokens en verwerkingstijd?*

De volgende cellen tonen:
- **Latency boxplot** — Spreiding verwerkingstijd per experiment
- **Stabiliteit boxplot & heatmap** — Spreiding consistentie per case en experiment
- **Scatterplot prompt tokens vs. latency** — Relatie tussen contextomvang en snelheid

RAG (B) en web search (C) gebruiken meer prompt tokens door extra context, wat zich vertaalt in hogere latency.

In [0]:
sns.boxplot(
    data=df_runs,
    x="experiment",
    y="latency_seconds"
)
plt.title("Latency per experiment")
plt.show()

In [0]:
# Stabiliteit boxplot — alle experimenten inclusief D
fig, ax = plt.subplots(figsize=(8, 5))

# Zorg dat alle experimenten op de x-as staan, ook als er geen data is
sns.boxplot(
    data=stability_by_case,
    x="experiment",
    y="stability",
    order=EXPERIMENTS,
    ax=ax
)

# Annoteer experimenten zonder data
for i, exp in enumerate(EXPERIMENTS):
    exp_data = stability_by_case[
        (stability_by_case["experiment"] == exp) &
        (stability_by_case["stability"].notna())
    ]
    if exp_data.empty:
        ax.text(i, 0.5, "N/A\n(1 run)", ha="center", va="center",
                fontsize=9, color="gray", style="italic")

ax.set_title("Stabiliteit per experiment")
ax.set_ylabel("Cosine similarity")
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [0]:
# Consistentie heatmap per case en experiment — met NaN-handling
heatmap_df = stability_by_case.pivot(
    index="case_id",
    columns="experiment",
    values="stability"
)

# Zorg dat alle experimenten als kolom bestaan
for exp in EXPERIMENTS:
    if exp not in heatmap_df.columns:
        heatmap_df[exp] = np.nan
heatmap_df = heatmap_df[EXPERIMENTS]  # vaste volgorde

fig, ax = plt.subplots(figsize=(7, 5))

# Teken heatmap zonder annotatie voor NaN
sns.heatmap(
    heatmap_df,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    vmin=0,
    vmax=1,
    ax=ax,
    linewidths=0.5,
)

# Voeg handmatig 'N/A' toe voor NaN-cellen (zichtbare kleur)
for i, row_idx in enumerate(heatmap_df.index):
    for j, col in enumerate(heatmap_df.columns):
        val = heatmap_df.loc[row_idx, col]
        if pd.isna(val):
            ax.text(j + 0.5, i + 0.5, "N/A\n(1 run)",
                    ha="center", va="center",
                    fontsize=8, color="#666666", style="italic")

ax.set_title("Consistentie per case en experiment")
ax.set_xlabel("Experiment")
ax.set_ylabel("Case")
plt.tight_layout()
plt.show()

In [0]:
sns.scatterplot(
    data=df_runs,
    x="prompt_tokens",
    y="latency_seconds",
    hue="experiment"
)
plt.title("Prompt tokens vs latency")
plt.tight_layout()
plt.show()

### 4.7 Semantische vergelijking tússen architecturen
> **Adresseert:** *Hoe sterk verschillen de outputs van A, B en C voor dezelfde case?*

Pairwise cosine similarity tussen de antwoorden van verschillende experimenten voor dezelfde case. Dit meet hoezeer RAG (B) en web search (C) de inhoud van het advies veranderen ten opzichte van de baseline (A).

Een lagere similarity tussen A↔B of A↔C wijst op een groter effect van de retrieval-component.

In [0]:
def semantic_similarity(text_a, text_b):
    
    if not text_a or not text_b:
        return np.nan

    embeddings = embedding_model.encode(
        [text_a, text_b],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    sim = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
    return float(sim)

In [0]:
answers_by_exp = (
    df_runs
    .groupby(["experiment", "case_id"])["answer"]
    .first()
    .reset_index()
)

answers_by_exp.head()

In [0]:
pivot_answers = answers_by_exp.pivot(
    index="case_id",
    columns="experiment",
    values="answer"
)

pivot_answers

In [0]:
from itertools import combinations

rows = []

for case_id, row in pivot_answers.iterrows():
    pair_sims = {}
    for e1, e2 in combinations(EXPERIMENTS, 2):
        key = f"{e1.upper()}_vs_{e2.upper()}"
        pair_sims[key] = semantic_similarity(row[e1], row[e2])
    pair_sims["case_id"] = case_id
    rows.append(pair_sims)

similarity_df = pd.DataFrame(rows)
similarity_df

In [0]:
plt.figure(figsize=(6,4))

sns.heatmap(
    similarity_df.set_index("case_id"),
    annot=True,
    cmap="coolwarm",
    vmin=0,
    vmax=1
)

plt.title("Semantic similarity tussen experimenten")
plt.show()

In [0]:
n = len(EXPERIMENTS)
labels = [e.upper() for e in EXPERIMENTS]
matrix = np.eye(n)

for i, e1 in enumerate(EXPERIMENTS):
    for j, e2 in enumerate(EXPERIMENTS):
        if i < j:
            col = f"{e1.upper()}_vs_{e2.upper()}"
            if col in similarity_df.columns:
                val = similarity_df[col].mean()
                matrix[i][j] = val
                matrix[j][i] = val

sim_matrix = pd.DataFrame(matrix, index=labels, columns=labels)

plt.figure(figsize=(7, 6))

sns.heatmap(
    sim_matrix,
    annot=True,
    cmap="coolwarm",
    vmin=0,
    vmax=1
)

plt.title("Semantic similarity tussen experimentarchitecturen")
plt.xlabel("Experiment")
plt.ylabel("Experiment")
plt.show()

## 4.8 CGR en Hallucinatiegraad
> **Adresseert:** *In hoeverre zijn de adviezen gegrond in de aangeleverde context (geen hallucinaties)?*

Voor alle experimenten wordt de **Claim Grounding Rate (CGR)** berekend:
- Feitelijke claims worden geëxtraheerd uit het advies via LLM
- Per claim wordt gecontroleerd of deze ondersteund wordt door de casecontext
- CGR = (1/N) × Σ gᵢ waarbij gᵢ = 1 als claim ondersteund, 0 als niet
- **Hallucinatiegraad** = 1 − CGR

**Context Precision (CP):**
- Meet hoeveel van de opgehaalde RAG-fragmenten daadwerkelijk relevant zijn voor het antwoord
- CP = relevant_fragments / totaal_fragments
- Alleen van toepassing op experiment B (RAG). Experiment C had geen web search resultaten.

In [0]:
# ── CGR berekening via LLM ─────────────────────────────────────────────
# Check claims tegen de VOLLEDIGE context die aan de LLM is meegegeven
# (case context + RAG chunks voor B, case context + web results voor C)

def extract_claims(answer):
    """Extraheer feitelijke claims uit een antwoord via LLM."""
    prompt = f"""Extraheer de belangrijkste feitelijke claims uit het onderstaande strategisch advies.
Geef maximaal 8 claims. Geef JSON terug: {{"claims": ["claim1", "claim2", ...]}}

ADVIES:
{answer[:3000]}
"""
    try:
        resp = openai_client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0, max_tokens=500,
        )
        parsed = parse_judge_json(resp.choices[0].message.content or "")
        return parsed.get("claims", [])
    except Exception as e:
        print(f"  Extract claims fout: {e}")
        return []

def check_claim_grounding(claim, context):
    """Check of een claim ondersteund wordt door de context."""
    prompt = f"""Wordt de volgende CLAIM ondersteund door de CONTEXT?
Antwoord met JSON: {{"supported": 0 of 1}}

CONTEXT:
{context[:4000]}

CLAIM:
{claim}
"""
    try:
        resp = openai_client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0, max_tokens=50,
        )
        parsed = parse_judge_json(resp.choices[0].message.content or "")
        return int(parsed.get("supported", 0))
    except Exception:
        return 0

# ── Bouw context-map per experiment/case uit de opgeslagen messages ──
def extract_context_from_messages(case_data):
    """Haal de volledige context (voor VRAAG:) uit de user message."""
    user_msg = case_data.get("messages", [{}])[-1].get("content", "")
    # Alles voor VRAAG: is de context
    if "VRAAG:" in user_msg:
        return user_msg[:user_msg.index("VRAAG:")]
    return user_msg[:3000]

context_map = {}  # {(experiment, case_id): context_string}

for exp in EXPERIMENTS:
    exp_dir = os.path.join(RESULTS_DIR, f"experiment_{exp}")
    run_path = os.path.join(exp_dir, "run_1.json")
    if not os.path.exists(run_path):
        continue
    with open(run_path, "r", encoding="utf-8") as f:
        run_data = json.load(f)
    for case_data in run_data.get("cases", []):
        ctx = extract_context_from_messages(case_data)
        context_map[(exp, case_data["case_id"])] = ctx

print(f"Context geladen voor {len(context_map)} experiment/case combinaties")

# ── CGR berekenen voor ALLE experimenten, run 1 ──
df_to_cgr = df_runs[df_runs["run"] == 1].copy()

print(f"CGR te berekenen voor {len(df_to_cgr)} antwoorden...")
print("Dit kan enkele minuten duren...\n")

cgr_rows = []
for _, row in df_to_cgr.iterrows():
    exp, case_id = row["experiment"], row["case_id"]
    ctx = context_map.get((exp, case_id), "")
    print(f"  CGR {exp} / {case_id}...", end=" ")
    
    claims = extract_claims(row["answer"])
    if not claims:
        cgr_val = 1.0  # geen claims = geen hallucinatie
    else:
        groundings = [check_claim_grounding(c, ctx) for c in claims]
        cgr_val = sum(groundings) / len(groundings)
    
    cgr_rows.append({
        "experiment": exp,
        "case_id": case_id,
        "n_claims": len(claims),
        "CGR": cgr_val,
        "hallucination_rate": 1 - cgr_val,
    })
    print(f"claims={len(claims)} | CGR={cgr_val:.2f}")

df_cgr = pd.DataFrame(cgr_rows)
print("\nCGR resultaten:")
df_cgr

In [0]:
# ── Context Precision voor retrieval-experimenten (B + C) ────────────
# Meet hoeveel van de opgehaalde fragmenten/bronnen relevant zijn.
# Experiment B: RAG-chunks (patroon [Bron X | ...])
# Experiment C: Web search bronnen (patroon AANVULLENDE_INTERNET_CONTEXT + Gebruikte bronnen)

def extract_rag_chunks(user_message):
    """Extraheer individuele RAG-chunks uit de user message (experiment B)."""
    chunk_pattern = r'(\[Bron \d+ \|[^\]]+\]\n(?:(?!\[Bron).)*?)(?=\[Bron \d+|VRAAG:|AANVULLENDE_SCENARIO|AANVULLENDE_MENU|$)'
    matches = re.findall(chunk_pattern, user_message, re.DOTALL)
    return [m.strip() for m in matches if len(m.strip()) > 50]

def extract_web_sources(user_message):
    """Extraheer individuele web-bronnen uit de user message (experiment C).
    
    Zoekt naar:
    1. Het AANVULLENDE_INTERNET_CONTEXT blok met samengevatte web-info
    2. Individuele bronnen uit 'Gebruikte bronnen:' sectie
    3. Inline citaties met URLs
    """
    sources = []
    
    # Methode 1: Extraheer per genummerde bron uit 'Gebruikte bronnen:'
    bronnen_pattern = r'\[(\d+)\]\s*(.+?)\s*-\s*(https?://[^\s]+)'
    bronnen_matches = re.findall(bronnen_pattern, user_message)
    
    if bronnen_matches:
        # Zoek ook de samengevatte tekst per bron uit de internet context
        internet_ctx = ""
        if "AANVULLENDE_INTERNET_CONTEXT:" in user_message:
            start = user_message.index("AANVULLENDE_INTERNET_CONTEXT:") + len("AANVULLENDE_INTERNET_CONTEXT:")
            # Zoek einde (volgende sectie of einde)
            end_markers = ["VRAAG:", "INSTRUCTIE:"]
            end = len(user_message)
            for marker in end_markers:
                if marker in user_message[start:]:
                    end = start + user_message[start:].index(marker)
                    break
            internet_ctx = user_message[start:end].strip()
        
        for rank, title, url in bronnen_matches:
            # Probeer relevante tekst rond de URL te vinden in de internet context
            excerpt = ""
            if internet_ctx and url in internet_ctx:
                url_pos = internet_ctx.index(url)
                excerpt = internet_ctx[max(0, url_pos-300):url_pos+100].strip()
            elif internet_ctx:
                # Neem een stuk van de internet context per bron
                chunk_size = len(internet_ctx) // max(len(bronnen_matches), 1)
                idx = int(rank) - 1
                excerpt = internet_ctx[idx*chunk_size:(idx+1)*chunk_size].strip()
            
            sources.append({
                "rank": int(rank),
                "title": title.strip(),
                "url": url.strip(),
                "text": f"{title.strip()}\n{excerpt}" if excerpt else title.strip(),
            })
    
    return sources

def check_fragment_relevance(fragment, answer):
    """Check of een opgehaald fragment/bron relevant is voor het antwoord."""
    prompt = f"""Is dit FRAGMENT relevant voor en gebruikt in het gegeven ANTWOORD?
Antwoord met JSON: {{"relevant": 0 of 1}}

FRAGMENT:
{fragment[:1500]}

ANTWOORD:
{answer[:2000]}
"""
    try:
        resp = openai_client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0, max_tokens=50,
        )
        parsed = parse_judge_json(resp.choices[0].message.content or "")
        return int(parsed.get("relevant", 0))
    except Exception:
        return 0

# ── Context Precision berekenen voor B (RAG) en C (Web search) ──
cp_rows = []

for exp in RETRIEVAL_EXPERIMENTS:
    exp_dir = os.path.join(RESULTS_DIR, f"experiment_{exp}")
    run_path = os.path.join(exp_dir, "run_1.json")
    
    if not os.path.exists(run_path):
        print(f"Experiment {exp}: run_1.json niet gevonden, overgeslagen.")
        continue
    
    with open(run_path, "r", encoding="utf-8") as f:
        run_data = json.load(f)
    
    exp_label = "RAG" if exp == "b" else "Web search"
    print(f"Context Precision voor experiment {exp.upper()} ({exp_label})...")
    
    for case_data in run_data.get("cases", []):
        case_id = case_data["case_id"]
        answer = case_data["answer"]
        user_msg = case_data["messages"][-1]["content"]
        
        # Extract chunks/bronnen afhankelijk van experiment type
        if exp == "b":
            chunks = extract_rag_chunks(user_msg)
            fragments = [ch for ch in chunks]
        else:  # exp == "c"
            web_sources = extract_web_sources(user_msg)
            fragments = [s["text"] for s in web_sources]
        
        k = len(fragments)
        
        if k == 0:
            cp = np.nan
            relevant = 0
        else:
            relevant = sum(check_fragment_relevance(frag, answer) for frag in fragments)
            cp = relevant / k
        
        cp_rows.append({
            "experiment": exp,
            "case_id": case_id,
            "k": k,
            "relevant_fragments": relevant,
            "context_precision": cp,
        })
        print(f"  CP {exp}/{case_id}: {relevant}/{k} relevant = {cp:.2f}" if k > 0 else f"  CP {exp}/{case_id}: geen fragmenten")

df_cp = pd.DataFrame(cp_rows) if cp_rows else pd.DataFrame(
    columns=["experiment", "case_id", "k", "relevant_fragments", "context_precision"]
)
print("\nContext Precision resultaten:")
df_cp

In [0]:
# ── CGR en Context Precision samenvatting ──────────────────────────────────

if not df_cgr.empty:
    cgr_summary = df_cgr.groupby("experiment").agg(
        avg_CGR=("CGR", "mean"),
        avg_hallucination=("hallucination_rate", "mean"),
        avg_claims=("n_claims", "mean"),
    ).round(3)
    print("CGR per experiment:")
    display(cgr_summary)
    
    # Visualisatie CGR
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    cgr_plot = df_cgr.copy()
    sns.barplot(data=cgr_plot, x="experiment", y="CGR", ax=axes[0])
    axes[0].set_title("Claim Grounding Rate per experiment")
    axes[0].set_ylim(0, 1.1)
    axes[0].set_ylabel("CGR")
    
    sns.barplot(data=cgr_plot, x="experiment", y="hallucination_rate", ax=axes[1])
    axes[1].set_title("Hallucinatiegraad per experiment")
    axes[1].set_ylim(0, 1.1)
    axes[1].set_ylabel("1 - CGR")
    
    plt.tight_layout()
    plt.show()
else:
    print("Geen CGR data beschikbaar.")
    cgr_summary = pd.DataFrame()

if not df_cp.empty:
    cp_summary = df_cp.groupby("experiment").agg(
        avg_context_precision=("context_precision", "mean"),
    ).round(3)
    print("\nContext Precision per experiment (alleen B, C had geen web results):")
    display(cp_summary)
else:
    print("Geen Context Precision data beschikbaar.")
    cp_summary = pd.DataFrame()

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# 4.8.1 CGR lengtenormalisatie
# ═══════════════════════════════════════════════════════════════════════
# Experiment D (Self-Refine) genereert 3× meer tekst dan A/B/C.
# Dit vergroot het absolute aantal claims, waardoor de ruwe CGR
# D bevoordeelt. Hier normaliseren we voor antwoordlengte.
#
# Metric: claims_per_1k_words = (n_claims / n_words) × 1000
# Interpretatie: claimdichtheid — hoeveel feitelijke beweringen
#               per 1000 woorden?

if 'df_cgr' in dir() and not df_cgr.empty:
    # Merge met antwoordlengte (run 1)
    df_cgr_len = df_cgr.merge(
        df_runs[df_runs["run"] == 1][["experiment", "case_id", "answer"]],
        on=["experiment", "case_id"], how="left"
    )
    df_cgr_len["n_words"] = df_cgr_len["answer"].str.split().str.len()
    df_cgr_len["claims_per_1k_words"] = (df_cgr_len["n_claims"] / df_cgr_len["n_words"]) * 1000
    df_cgr_len["grounded_per_1k_words"] = (df_cgr_len["n_claims"] * df_cgr_len["CGR"] / df_cgr_len["n_words"]) * 1000

    # Samenvatting
    print("CGR LENGTENORMALISATIE")
    print("=" * 70)
    norm_summary = df_cgr_len.groupby("experiment").agg(
        gem_woorden=("n_words", "mean"),
        gem_claims=("n_claims", "mean"),
        claims_per_1k=("claims_per_1k_words", "mean"),
        CGR_raw=("CGR", "mean"),
        grounded_per_1k=("grounded_per_1k_words", "mean"),
    ).round(2).reindex(EXPERIMENTS)
    display(norm_summary)

    # Visualisatie: ruwe CGR vs genormaliseerde claims
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1. Claims per 1k woorden (claimdichtheid)
    for i, exp in enumerate(EXPERIMENTS):
        vals = df_cgr_len[df_cgr_len["experiment"] == exp]["claims_per_1k_words"]
        axes[0].bar(i, vals.mean(), color=palette[exp], yerr=vals.std(), capsize=5)
        axes[0].text(i, vals.mean() + vals.std() + 0.2, f"{vals.mean():.1f}",
                     ha="center", fontweight="bold", fontsize=10)
    axes[0].set_xticks(range(len(EXPERIMENTS)))
    axes[0].set_xticklabels([e.upper() for e in EXPERIMENTS])
    axes[0].set_title("Claimdichtheid\n(claims per 1000 woorden)")
    axes[0].set_ylabel("Claims / 1k woorden")

    # 2. Ruwe CGR vs grounded_per_1k
    x = np.arange(len(EXPERIMENTS))
    w = 0.35
    cgr_raw = [norm_summary.loc[e, "CGR_raw"] if e in norm_summary.index else 0 for e in EXPERIMENTS]
    grounded_1k = [norm_summary.loc[e, "grounded_per_1k"] if e in norm_summary.index else 0 for e in EXPERIMENTS]
    
    bars1 = axes[1].bar(x - w/2, cgr_raw, w, label="CGR (ruwe proportie)",
                         color=[palette[e] for e in EXPERIMENTS], alpha=0.8, edgecolor="black")
    ax2 = axes[1].twinx()
    bars2 = ax2.bar(x + w/2, grounded_1k, w, label="Gegronde claims / 1k woorden",
                     color=[palette[e] for e in EXPERIMENTS], alpha=0.4, edgecolor="black", hatch="//")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([e.upper() for e in EXPERIMENTS])
    axes[1].set_ylabel("CGR (0–1)")
    ax2.set_ylabel("Gegronde claims / 1k woorden")
    axes[1].set_title("CGR: ruw vs lengte-genormaliseerd")
    axes[1].legend(loc="upper left", fontsize=8)
    ax2.legend(loc="upper right", fontsize=8)

    # 3. Scatter: antwoordlengte vs n_claims (verbosity → meer claims?)
    for exp in EXPERIMENTS:
        sub = df_cgr_len[df_cgr_len["experiment"] == exp]
        axes[2].scatter(sub["n_words"], sub["n_claims"], color=palette[exp],
                        s=100, label=f"{exp.upper()}", edgecolors="white", alpha=0.8)
    # Trendlijn
    from numpy.polynomial.polynomial import polyfit
    all_w = df_cgr_len["n_words"].dropna().values
    all_c = df_cgr_len["n_claims"].dropna().values
    if len(all_w) > 2:
        b, m = polyfit(all_w, all_c, 1)
        x_line = np.linspace(all_w.min(), all_w.max(), 100)
        axes[2].plot(x_line, b + m * x_line, "--", color="gray", alpha=0.5)
        r = np.corrcoef(all_w, all_c)[0, 1]
        axes[2].set_title(f"Antwoordlengte vs aantal claims\n(r={r:.2f})")
    else:
        axes[2].set_title("Antwoordlengte vs aantal claims")
    axes[2].set_xlabel("Aantal woorden")
    axes[2].set_ylabel("Aantal claims")
    axes[2].legend(fontsize=9)

    plt.suptitle("§4.8.1 CGR lengtenormalisatie", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

    # Conclusie
    print("\nCONCLUSIE:")
    if len(all_w) > 2 and r > 0.5:
        print(f"  ⚠️ Sterke correlatie lengte ↔ aantal claims (r={r:.2f})")
        print("  → Langere antwoorden produceren meer claims; ruwe CGR is misleidend")
        print("  → Gebruik 'gegronde claims per 1k woorden' als robuustere maat")
    else:
        print(f"  Correlatie lengte ↔ claims: r={r:.2f}")
        print("  → Claimdichtheid verschilt niet sterk; ruwe CGR is betrouwbaar")
else:
    print("⚠️ df_cgr niet beschikbaar — draai eerst §4.8 CGR berekening")

### ⚠️ Bevinding: Experiment C web search leverde geen resultaten — OPGELOST

**Root cause:** DuckDuckGo blokkeert requests vanuit Databricks serverless compute (datacenter IP-detectie).

**Bewijs:**
- Alle 5 cases × 4 pogingen (logs 10 maart + 19 maart 2026) → `results: []`
- DuckDuckGo retourneert HTTP 202 + JS-challenge i.p.v. zoekresultaten
- `duckduckgo_search` library faalt *silently* bij bot-detectie
- B↔C similarity van 0.95 bevestigt: C gedroeg zich als baseline

**Oplossing geïmplementeerd:** `web_search.py` is uitgebreid met `retrieve_web_context_openai()` dat de **OpenAI Responses API** met `web_search_preview` tool gebruikt. Dit werkt betrouwbaar vanuit elke omgeving (gewone API-call, geen scraping).

**Testresultaten na fix:**
| Case | Bronnen | Samenvatting | Tokens | Augmented context |
|------|---------|-------------|--------|-------------------|
| case_1 | 5 | 3573 chars | 2141 | 7328 chars |
| case_2 | 5 | 3279 chars | 2221 | 7313 chars |
| case_3 | 2 | 3004 chars | 2058 | 6576 chars |
| case_4 | 5 | 4485 chars | 2625 | 8877 chars |
| case_5 | 5 | 2524 chars | 2203 | 7322 chars |

**Aanpassingen:**
- `utils/web_search.py` — nieuwe `retrieve_web_context_openai()` functie (OpenAI als primary, DuckDuckGo als legacy fallback)
- `experiments/c_experiment.py` — `backend: "openai"` als default in `EXPERIMENT["web_search"]`

**Volgende stap:** Re-run experiment C met verrijkte case data + OpenAI web search.

## 4.9 Scenariokwaliteit: diversiteit × passendheid
> **Adresseert:** *Hoe divers én contextrelevant zijn de scenario’s binnen één gegenereerd advies?*

Elk advies formuleert scenario’s als onderdeel van de strategische analyse. Goede scenario’s voldoen aan twee voorwaarden:

1. **Diversiteit** — De scenario’s moeten inhoudelijk voldoende van elkaar verschillen om relevante onzekerheden te verkennen
2. **Passendheid** — Elk scenario moet passen bij het type restaurant, de doelgroep, het budget en de marktcontext

Deze twee dimensies staan in spanning: hoge diversiteit vergroot het risico dat scenario’s te ver van de context afdwalen, terwijl hoge passendheid het risico van te uniforme scenario’s vergroot.

**Meetmethode:**
- **Semantische diversiteit** = 1 − gem. cosine similarity (embedding-based)
- **Contextpassendheid** = LLM-beoordeling per scenario tegen de casedata (0–100)
- **Plausibiliteit** = LLM-beoordeling: is het scenario realistisch gegeven budget, capaciteit, doelgroep? (0–100)
- **Scenariokwaliteit** = harmonisch gemiddelde van diversiteit (genorm.) × passendheid × plausibiliteit

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# 4.9.1 Intra-advies scenario diversiteit — ALLE runs
# ═══════════════════════════════════════════════════════════════════════
# Meet hoe divers de 3 scenario's BINNEN één gegenereerd advies zijn,
# over alle 5 runs heen. Puur embedding-based (geen API calls).
#
# Metric: sem_diversity = 1 − gem. pairwise cosine similarity
#   0.0 = alle scenario's identiek
#   1.0 = maximaal verschillend

import re

# ── Stap 1: Scenario-extractie functie (ontbrak in het notebook) ──
def extract_scenarios_from_answer(answer: str) -> list:
    """Extraheer individuele scenario's uit een LLM-gegenereerd strategisch advies.
    
    Zoekt naar patronen als 'Scenario 1:', 'Scenario A:', of genummerde scenario-blokken
    in stap 5 (Scenario-analyse) van het 11-stappen CoT-format.
    """
    if not answer or not isinstance(answer, str):
        return []
    
    scenarios = []
    
    # Patroon 1: "Scenario 1: ...", "Scenario 2: ...", etc.
    pattern = r'(?:^|\n)\s*\**\s*[Ss]cenario\s*[\d\w][.:)\-]\s*\**\s*(.+?)(?=\n\s*\**\s*[Ss]cenario\s*[\d\w][.:)\-]|\n\s*(?:#{1,4}|\d+\.|\*\*\s*(?:Stap|Step))\s|$)'
    matches = re.findall(pattern, answer, re.DOTALL)
    
    if matches and len(matches) >= 2:
        scenarios = [m.strip() for m in matches if len(m.strip()) > 50]
    
    # Patroon 2: Genummerde items binnen de scenario-sectie
    if len(scenarios) < 2:
        # Zoek de scenario-analyse sectie
        sec_pattern = r'(?:[Ss]cenario[\-\s]*analyse|[Ss]cenario.{0,20}verkenning).*?(?=\n\s*(?:#{1,4}|\*\*\s*(?:Stap|Step)\s*(?:6|7))\s|$)'
        sec_match = re.search(sec_pattern, answer, re.DOTALL | re.IGNORECASE)
        if sec_match:
            section = sec_match.group(0)
            # Zoek genummerde of gebulleteerde items
            items = re.split(r'\n\s*(?:\d+[.)\-]|[\-\*\u2022])\s+\**', section)
            scenarios = [i.strip() for i in items if len(i.strip()) > 80]
    
    return scenarios[:5]  # max 5 scenario's

print(f"extract_scenarios_from_answer() gedefinieerd")

# ── Stap 2: Bereken diversiteit voor ALLE runs ──
print(f"\nScenario-diversiteit berekenen voor {len(df_runs)} antwoorden...")

div_rows = []
for _, row in df_runs.iterrows():
    exp, run, case_id, answer = row["experiment"], row["run"], row["case_id"], row["answer"]
    
    scenarios = extract_scenarios_from_answer(answer)
    n_scen = len(scenarios)
    
    if n_scen >= 2:
        embs = embedding_model.encode(scenarios[:3], normalize_embeddings=True)
        sim = cosine_similarity(embs)
        iu = np.triu_indices_from(sim, k=1)
        pairwise_sims = sim[iu]
        sem_diversity = float(1.0 - pairwise_sims.mean())
        min_sim = float(pairwise_sims.min())  # meest verschillende paar
        max_sim = float(pairwise_sims.max())  # meest gelijke paar
    else:
        sem_diversity = np.nan
        min_sim = np.nan
        max_sim = np.nan
    
    div_rows.append({
        "experiment": exp, "run": run, "case_id": case_id,
        "n_scenarios": n_scen,
        "diversity": round(sem_diversity, 4) if not np.isnan(sem_diversity) else np.nan,
        "max_pairwise_sim": round(max_sim, 4) if not np.isnan(max_sim) else np.nan,
        "min_pairwise_sim": round(min_sim, 4) if not np.isnan(min_sim) else np.nan,
    })

df_div_allruns = pd.DataFrame(div_rows)

# Statistieken
print(f"\nResultaat: {len(df_div_allruns)} antwoorden geanalyseerd")
for exp in EXPERIMENTS:
    sub = df_div_allruns[df_div_allruns["experiment"] == exp]
    valid = sub["diversity"].dropna()
    no_scen = sub["n_scenarios"].apply(lambda x: x < 2).sum()
    print(f"  {exp.upper()} ({EXP_LABELS[exp]}): {len(valid)}/{len(sub)} met ≥2 scenario's | "
          f"gem. diversiteit = {valid.mean():.3f} (±{valid.std():.3f})" if len(valid) > 0 else
          f"  {exp.upper()} ({EXP_LABELS[exp]}): geen scenario's gevonden")

# ── Stap 3: Visualisatie ──
df_plot = df_div_allruns.dropna(subset=["diversity"])

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 3a. Boxplot: diversiteitsverdeling per experiment
ax = axes[0, 0]
sns.boxplot(data=df_plot, x="experiment", y="diversity", order=EXPERIMENTS,
            hue="experiment", palette=palette, legend=False, ax=ax,
            showmeans=True, meanprops={"marker": "D", "markerfacecolor": "white", "markersize": 8})
ax.set_title("Intra-advies scenario diversiteit per experiment\n(alle runs)")
ax.set_ylabel("Diversiteit (1 − gem. cosine sim)\n↑ hoger = meer divers")
ax.set_xlabel("Experiment")
ax.set_ylim(0, max(0.6, df_plot["diversity"].max() * 1.1))
ax.axhline(y=df_plot["diversity"].median(), color="gray", ls="--", alpha=0.3, label="mediaan totaal")
for i, exp in enumerate(EXPERIMENTS):
    vals = df_plot[df_plot["experiment"] == exp]["diversity"]
    if len(vals) > 0:
        ax.text(i, vals.mean() + 0.008, f"μ={vals.mean():.3f}", ha="center", fontsize=9, fontweight="bold")

# 3b. Violin plot: dichtheid van de verdeling
ax = axes[0, 1]
sns.violinplot(data=df_plot, x="experiment", y="diversity", order=EXPERIMENTS,
               hue="experiment", palette=palette, legend=False, ax=ax,
               inner="quartile", cut=0)
ax.set_title("Dichtheidsverdeling scenario diversiteit\n(violin plot — breedte = frequentie)")
ax.set_ylabel("Diversiteit")
ax.set_xlabel("Experiment")
ax.set_ylim(0, max(0.6, df_plot["diversity"].max() * 1.1))

# 3c. Spreiding per case: hoe consistent is de diversiteit?
ax = axes[1, 0]
for exp in EXPERIMENTS:
    sub = df_plot[df_plot["experiment"] == exp]
    case_means = sub.groupby("case_id")["diversity"].agg(["mean", "std"]).reset_index()
    x_pos = EXPERIMENTS.index(exp)
    for _, r in case_means.iterrows():
        ax.errorbar(x_pos + np.random.uniform(-0.15, 0.15), r["mean"], yerr=r["std"],
                    fmt="o", color=palette[exp], markersize=8, capsize=4, alpha=0.7)
ax.set_xticks(range(len(EXPERIMENTS)))
ax.set_xticklabels([f"{e.upper()}" for e in EXPERIMENTS])
ax.set_title("Diversiteit per case (gem. ± std over runs)\n(elke stip = 1 case, error bar = spreiding over runs)")
ax.set_ylabel("Diversiteit")
ax.set_xlabel("Experiment")
ax.set_ylim(0, max(0.6, df_plot["diversity"].max() * 1.1))

# 3d. Heatmap: experiment × case
ax = axes[1, 1]
heatmap_data = df_plot.pivot_table(index="case_id", columns="experiment",
                                    values="diversity", aggfunc="mean")
heatmap_data = heatmap_data.reindex(columns=EXPERIMENTS)
sns.heatmap(heatmap_data, annot=True, fmt=".3f", cmap="RdYlGn",
            vmin=0, vmax=heatmap_data.values.max() * 1.2,
            linewidths=0.5, ax=ax)
ax.set_title("Gem. diversiteit per case × experiment\n(donkerder groen = meer divers)")
ax.set_ylabel("Case")
ax.set_xlabel("Experiment")

plt.suptitle("§4.9.1 Intra-advies scenario diversiteit (alle runs)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# ── Stap 4: Samenvattingstabel + statistische toets ──
from scipy.stats import kruskal, mannwhitneyu

print("\n" + "=" * 75)
print("INTRA-ADVIES SCENARIO DIVERSITEIT — SAMENVATTING (alle runs)")
print("=" * 75)

summary_rows = []
for exp in EXPERIMENTS:
    vals = df_plot[df_plot["experiment"] == exp]["diversity"]
    summary_rows.append({
        "Experiment": f"{exp.upper()} ({EXP_LABELS[exp]})",
        "N": len(vals),
        "Gemiddelde": f"{vals.mean():.4f}" if len(vals) > 0 else "—",
        "Std": f"{vals.std():.4f}" if len(vals) > 0 else "—",
        "Min": f"{vals.min():.4f}" if len(vals) > 0 else "—",
        "Mediaan": f"{vals.median():.4f}" if len(vals) > 0 else "—",
        "Max": f"{vals.max():.4f}" if len(vals) > 0 else "—",
        "IQR": f"{vals.quantile(0.75) - vals.quantile(0.25):.4f}" if len(vals) > 0 else "—",
    })

df_summary = pd.DataFrame(summary_rows)
display(df_summary)

# Kruskal-Wallis test (non-parametrisch alternatief voor ANOVA)
groups = [df_plot[df_plot["experiment"] == exp]["diversity"].dropna().values
          for exp in EXPERIMENTS]
groups_valid = [g for g in groups if len(g) >= 2]

if len(groups_valid) >= 2:
    stat, p = kruskal(*groups_valid)
    print(f"\nKruskal-Wallis test: H={stat:.3f}, p={p:.4f}")
    if p < 0.05:
        print("→ Significante verschillen in diversiteit tussen experimenten (p<0.05)")
        # Post-hoc: pairwise Mann-Whitney
        print("\nPost-hoc pairwise vergelijkingen (Mann-Whitney U):")
        from itertools import combinations
        for e1, e2 in combinations(EXPERIMENTS, 2):
            g1 = df_plot[df_plot["experiment"] == e1]["diversity"].dropna()
            g2 = df_plot[df_plot["experiment"] == e2]["diversity"].dropna()
            if len(g1) >= 2 and len(g2) >= 2:
                u_stat, u_p = mannwhitneyu(g1, g2, alternative="two-sided")
                sig = "✅" if u_p < 0.05 else "⚠️" if u_p < 0.10 else "❌"
                diff = g1.mean() - g2.mean()
                print(f"  {e1.upper()} vs {e2.upper()}: U={u_stat:.0f}, p={u_p:.4f} {sig} | "
                      f"Δ={diff:+.4f} ({'{'}{e1.upper()} diverser{'}'}" if diff > 0 else
                      f"  {e1.upper()} vs {e2.upper()}: U={u_stat:.0f}, p={u_p:.4f} {sig} | "
                      f"Δ={diff:+.4f} ({'{'}{e2.upper()} diverser{'}'})")
    else:
        print("→ Geen significante verschillen in diversiteit tussen experimenten")
else:
    print("\nTe weinig data voor statistische toets")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# 4.7 Scenariokwaliteit: diversiteit × passendheid × plausibiliteit
# ═══════════════════════════════════════════════════════════════════════

# ── Stap 1: Laad case context ──
case_contexts = {}
CASE_DIR = os.path.join(PROJECT_ROOT, "data", "case_data")
for fname in sorted(os.listdir(CASE_DIR)):
    if fname.endswith(".json"):
        with open(os.path.join(CASE_DIR, fname)) as f:
            case = json.load(f)
        case_id = case.get("metadata", {}).get("case_id", fname.replace(".json", ""))
        
        # Bouw compacte context-samenvatting voor de LLM
        profiel = case.get("restaurantprofiel", {})
        context_summary = (
            f"RESTAURANT: {profiel.get('naam', '?')}\n"
            f"TYPE: {profiel.get('type', '?')}\n"
            f"LOCATIE: {profiel.get('locatie', '?')}\n"
            f"CAPACITEIT: {profiel.get('capaciteit', '?')}\n"
            f"GEM. BESTEDING: {profiel.get('gemiddelde_besteding_per_gast', '?')}\n"
            f"PERSONEEL: {profiel.get('personeel', '?')}\n"
            f"POSITIONERING: {case.get('positionering', '?')}\n"
            f"DOELGROEP: {case.get('doelgroep', {}).get('primair', '?')} / {case.get('doelgroep', {}).get('kenmerken', '?')}\n"
            f"MARKTCONTEXT: {case.get('marktcontext', {}).get('omgeving', '?')[:200]}\n"
            f"UITDAGING: {case.get('marktcontext', {}).get('impact', '?')[:200]}\n"
            f"MENUKAART: {', '.join(g.get('naam','') for g in case.get('menukaart',{}).get('hoofdgerechten',[])[:5])}"
        )
        case_contexts[case_id] = context_summary

print(f"Case contexts geladen: {list(case_contexts.keys())}")

# ── Stap 2: LLM scenario-evaluatie prompt ──
SCENARIO_EVAL_PROMPT = """Je bent een strenge beoordelaar van strategische scenario's voor restaurants.

Hieronder staan de CONTEXT van het restaurant en de SCENARIO'S uit een strategisch advies.
Beoordeel ELK scenario individueel op twee criteria, en beoordeel de SET als geheel op diversiteit.

RESTAURANT CONTEXT:
{context}

GEËXTRAHEERDE SCENARIO'S:
{scenarios_text}

BEOORDEEL:

1. PER SCENARIO (0–100 elk):
   a) PASSENDHEID: Past dit scenario bij het type restaurant, de doelgroep, het budget en de marktcontext?
      - 0–20: Totaal ongeschikt (bijv. dure kreeft voorstellen bij een studentencafé)
      - 21–40: Slecht passend, negeert kernidentiteit van het restaurant
      - 41–55: Enigszins passend maar mist belangrijke beperkingen
      - 56–70: Redelijk passend, houdt rekening met de meeste contextfactoren
      - 71–85: Goed passend, concreet afgestemd op dit specifieke restaurant
      - 86–100: Uitstekend, scenario is maatwerk voor deze context
   
   b) PLAUSIBILITEIT: Is dit scenario realistisch uitvoerbaar gegeven budget, capaciteit en personeel?
      - 0–20: Volledig onrealistisch
      - 21–40: Zeer onwaarschijnlijk, vereist middelen die het restaurant niet heeft
      - 41–55: Mogelijk maar met grote aanpassingen
      - 56–70: Redelijk realistisch met enige inspanning
      - 71–85: Goed uitvoerbaar binnen bestaande mogelijkheden
      - 86–100: Direct implementeerbaar

2. DIVERSITEIT VAN DE SET (0–100):
   - 0–20: Scenario's zijn vrijwel identiek, verkennen geen alternatieven
   - 21–40: Scenario's lijken sterk op elkaar, zelfde strategische richting
   - 41–55: Enig verschil maar overlap in kernstrategie
   - 56–70: Duidelijk verschillende richtingen maar deels overlappend
   - 71–85: Goed onderscheidend, elk scenario verkent een andere strategische as
   - 86–100: Maximaal divers, scenario's dekken tegengestelde strategieën

BELANGRIJK:
- Gebruik het HELE spectrum 0–100. Wees STRENG.
- Een fast-food restaurant dat een fine-dining scenario krijgt → passendheid < 30.
- Scenario's die alleen in prijsstelling verschillen maar dezelfde strategie volgen → diversiteit < 40.

Geef ALLEEN valid JSON terug:
{{
  "scenario_1": {{"passendheid": <0-100>, "plausibiliteit": <0-100>, "toelichting": "<1 zin>"}},
  "scenario_2": {{"passendheid": <0-100>, "plausibiliteit": <0-100>, "toelichting": "<1 zin>"}},
  "scenario_3": {{"passendheid": <0-100>, "plausibiliteit": <0-100>, "toelichting": "<1 zin>"}},
  "diversiteit": {{"score": <0-100>, "toelichting": "<1 zin>"}}
}}

Als er minder dan 3 scenario's zijn, beoordeel alleen de aanwezige.
"""

def evaluate_scenarios(scenarios: list, case_context: str) -> dict:
    """Beoordeel scenario's op passendheid, plausibiliteit en diversiteit."""
    scenarios_text = ""
    for i, s in enumerate(scenarios[:3], 1):
        scenarios_text += f"\nSCENARIO {i}:\n{s[:800]}\n"
    
    prompt = SCENARIO_EVAL_PROMPT.format(
        context=case_context,
        scenarios_text=scenarios_text
    )
    
    try:
        resp = openai_client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0, max_tokens=600,
        )
        parsed = parse_judge_json(resp.choices[0].message.content or "")
    except Exception as e:
        print(f"FOUT: {e}")
        parsed = {}
    
    return parsed

# ── Stap 3: Evalueer alle scenario's ──
print(f"\nScenario-evaluatie: {len(df_runs[df_runs['run']==1])} antwoorden...")
print("(LLM beoordeelt passendheid + plausibiliteit + diversiteit per case)\n")

scenario_rows = []
for _, row in df_runs[df_runs["run"] == 1].iterrows():
    exp, case_id, answer = row["experiment"], row["case_id"], row["answer"]
    context = case_contexts.get(case_id, "Geen context beschikbaar")
    
    # Extraheer scenario's (bestaande functie)
    scenarios = extract_scenarios_from_answer(answer)
    n_scenarios = len(scenarios)
    
    # Semantische diversiteit (bestaande meting)
    if n_scenarios >= 2:
        embs = embedding_model.encode(scenarios[:3], normalize_embeddings=True)
        sim = cosine_similarity(embs)
        iu = np.triu_indices_from(sim, k=1)
        sem_diversity = float(1.0 - sim[iu].mean())
    else:
        sem_diversity = np.nan
    
    # LLM beoordeling
    if n_scenarios >= 2:
        eval_result = evaluate_scenarios(scenarios[:3], context)
    else:
        eval_result = {}
    
    # Parse resultaten
    passendheid_scores = []
    plausibiliteit_scores = []
    for i in range(1, 4):
        s_data = eval_result.get(f"scenario_{i}", {})
        if isinstance(s_data, dict):
            p = s_data.get("passendheid", np.nan)
            pl = s_data.get("plausibiliteit", np.nan)
            if isinstance(p, (int, float)): passendheid_scores.append(p)
            if isinstance(pl, (int, float)): plausibiliteit_scores.append(pl)
    
    div_data = eval_result.get("diversiteit", {})
    llm_diversity = div_data.get("score", np.nan) if isinstance(div_data, dict) else np.nan
    div_note = div_data.get("toelichting", "") if isinstance(div_data, dict) else ""
    
    avg_passendheid = np.mean(passendheid_scores) if passendheid_scores else np.nan
    avg_plausibiliteit = np.mean(plausibiliteit_scores) if plausibiliteit_scores else np.nan
    
    # Scenariokwaliteit = harmonisch gemiddelde van 3 dimensies (alle 0-1 schaal)
    if not np.isnan(avg_passendheid) and not np.isnan(avg_plausibiliteit) and not np.isnan(llm_diversity):
        d = llm_diversity / 100
        p = avg_passendheid / 100
        pl = avg_plausibiliteit / 100
        # Harmonisch gemiddelde (straft zwakste dimensie zwaar af)
        if d > 0 and p > 0 and pl > 0:
            scenario_quality = 3 / (1/d + 1/p + 1/pl)
        else:
            scenario_quality = 0
    else:
        scenario_quality = np.nan
    
    scenario_rows.append({
        "experiment": exp, "case_id": case_id,
        "n_scenarios": n_scenarios,
        "sem_diversity": round(sem_diversity, 3) if not np.isnan(sem_diversity) else np.nan,
        "llm_diversity": llm_diversity,
        "passendheid": round(avg_passendheid, 1) if not np.isnan(avg_passendheid) else np.nan,
        "plausibiliteit": round(avg_plausibiliteit, 1) if not np.isnan(avg_plausibiliteit) else np.nan,
        "scenario_quality": round(scenario_quality, 3) if not np.isnan(scenario_quality) else np.nan,
        "div_toelichting": div_note,
    })
    
    print(f"  {exp}/{case_id}: {n_scenarios} scen. | "
          f"sem_div={sem_diversity:.2f} | llm_div={llm_diversity} | "
          f"passend={avg_passendheid:.0f} | plausibel={avg_plausibiliteit:.0f} | "
          f"kwaliteit={scenario_quality:.3f}" if not np.isnan(scenario_quality) else 
          f"  {exp}/{case_id}: {n_scenarios} scen. | GEEN data")

df_scenario = pd.DataFrame(scenario_rows)

# ── Stap 4: Visualisatie ──
palette = {"a": "#4C72B0", "b": "#55A868", "c": "#C44E52", "d": "#8172B2"}

fig, axes = plt.subplots(2, 3, figsize=(20, 10))

# 1. Passendheid per experiment
sns.barplot(data=df_scenario, x="experiment", y="passendheid",
            order=EXPERIMENTS, hue="experiment", palette=palette, legend=False, ax=axes[0,0])
axes[0,0].set_title("Passendheid per experiment (0-100)")
axes[0,0].set_ylim(0, 105)
axes[0,0].set_ylabel("Passendheid (0-100)")
axes[0,0].axhline(y=55, color="gray", ls="--", alpha=0.3)
for i, exp in enumerate(EXPERIMENTS):
    v = df_scenario[df_scenario["experiment"]==exp]["passendheid"].mean()
    if not np.isnan(v): axes[0,0].text(i, v+1.5, f"{v:.0f}", ha="center", fontweight="bold")

# 2. Plausibiliteit per experiment
sns.barplot(data=df_scenario, x="experiment", y="plausibiliteit",
            order=EXPERIMENTS, hue="experiment", palette=palette, legend=False, ax=axes[0,1])
axes[0,1].set_title("Plausibiliteit per experiment (0-100)")
axes[0,1].set_ylim(0, 105)
axes[0,1].set_ylabel("Plausibiliteit (0-100)")
axes[0,1].axhline(y=55, color="gray", ls="--", alpha=0.3)
for i, exp in enumerate(EXPERIMENTS):
    v = df_scenario[df_scenario["experiment"]==exp]["plausibiliteit"].mean()
    if not np.isnan(v): axes[0,1].text(i, v+1.5, f"{v:.0f}", ha="center", fontweight="bold")

# 3. LLM diversiteit per experiment
sns.barplot(data=df_scenario, x="experiment", y="llm_diversity",
            order=EXPERIMENTS, hue="experiment", palette=palette, legend=False, ax=axes[0,2])
axes[0,2].set_title("LLM-diversiteit per experiment (0-100)")
axes[0,2].set_ylim(0, 105)
axes[0,2].set_ylabel("Diversiteit (0-100)")
axes[0,2].axhline(y=55, color="gray", ls="--", alpha=0.3)
for i, exp in enumerate(EXPERIMENTS):
    v = df_scenario[df_scenario["experiment"]==exp]["llm_diversity"].mean()
    if not np.isnan(v): axes[0,2].text(i, v+1.5, f"{v:.0f}", ha="center", fontweight="bold")

# 4. Scenariokwaliteit (harmonisch gemiddelde)
sns.barplot(data=df_scenario, x="experiment", y="scenario_quality",
            order=EXPERIMENTS, hue="experiment", palette=palette, legend=False, ax=axes[1,0])
axes[1,0].set_title("Scenariokwaliteit (harm. gem. D×P×Pl)")
axes[1,0].set_ylim(0, 1.05)
axes[1,0].set_ylabel("Kwaliteit (0-1)")
for i, exp in enumerate(EXPERIMENTS):
    v = df_scenario[df_scenario["experiment"]==exp]["scenario_quality"].mean()
    if not np.isnan(v): axes[1,0].text(i, v+0.02, f"{v:.3f}", ha="center", fontweight="bold")

# 5. Diversiteit vs Passendheid scatter (de trade-off!)
for exp in EXPERIMENTS:
    sub = df_scenario[df_scenario["experiment"]==exp]
    axes[1,1].scatter(sub["llm_diversity"], sub["passendheid"],
                      color=palette[exp], s=150, label=exp.upper(),
                      edgecolors="black", linewidth=0.5, zorder=3)
axes[1,1].set_xlabel("Diversiteit (0-100) →")
axes[1,1].set_ylabel("Passendheid (0-100) →")
axes[1,1].set_title("Trade-off: Diversiteit vs Passendheid")
axes[1,1].legend()
axes[1,1].set_xlim(0, 100)
axes[1,1].set_ylim(0, 100)
# Ideale zone markeren (rechtsboven)
from matplotlib.patches import Rectangle
rect = Rectangle((60, 60), 40, 40, linewidth=1, edgecolor='green', facecolor='green', alpha=0.08)
axes[1,1].add_patch(rect)
axes[1,1].text(80, 62, "ideale zone", ha="center", fontsize=8, color="green", alpha=0.7)

# 6. Heatmap: alle dimensies per experiment
heatmap_data = df_scenario.groupby("experiment")[
    ["llm_diversity", "passendheid", "plausibiliteit"]
].mean().reindex(EXPERIMENTS)
heatmap_data.columns = ["Diversiteit", "Passendheid", "Plausibiliteit"]
sns.heatmap(heatmap_data.T, annot=True, fmt=".1f", cmap="RdYlGn",
            vmin=0, vmax=100, linewidths=0.5, ax=axes[1,2])
axes[1,2].set_title("Scenario-evaluatie per experiment (0-100)")

plt.tight_layout()
plt.show()

# ── Samenvatting ──
print("\n=== Scenario-evaluatie samenvatting ===")
scen_summary = df_scenario.groupby("experiment").agg(
    gem_passendheid=("passendheid", "mean"),
    gem_plausibiliteit=("plausibiliteit", "mean"),
    gem_llm_diversiteit=("llm_diversity", "mean"),
    gem_sem_diversiteit=("sem_diversity", "mean"),
    gem_kwaliteit=("scenario_quality", "mean"),
    gem_n_scenarios=("n_scenarios", "mean"),
).round(2).reindex(EXPERIMENTS)
print(scen_summary)

# Update diversity_summary voor integrale vergelijking
diversity_summary = df_scenario.groupby("experiment").agg(
    avg_diversity=("scenario_quality", "mean"),  # gebruik nu scenario_quality i.p.v. simpele cosine
).round(3)
print(f"\n→ diversity_summary (voor integrale vergelijking) bijgewerkt met scenario_quality")
print(f"  (was: pure cosine diversity, nu: harmonisch gem. van diversiteit × passendheid × plausibiliteit)")

# Toelichtingen
print("\n=== Diversiteits-toelichtingen per experiment/case ===")
for _, row in df_scenario.iterrows():
    if row["div_toelichting"]:
        print(f"  {row['experiment']}/{row['case_id']}: {row['div_toelichting']}")

## 4.10 Integrale vergelijking van de architecturen
> **Adresseert:** *Welke LLM-architectuur genereert de beste strategische adviezen op basis van alle evaluatiedimensies?*

Per experiment worden twee scores berekend:

**Basisscore** (uit documentatie) = w₁×S + w₂×CGR + w₃×(1−H) + w₄×CP

**Uitgebreide score** (incl. kwaliteit en diversiteit):

| Gewicht | Metric | Schaal |
|---------|--------|--------|
| 0.20 | Stabiliteit (S) | 0–1 |
| 0.25 | Judge kwaliteit (Q) | 0–100 → genormaliseerd naar 0–1 |
| 0.20 | Claim Grounding Rate (CGR) | 0–1 |
| 0.10 | 1 − Hallucinatiegraad | 0–1 |
| 0.10 | Context Precision (CP) | 0–1 |
| 0.15 | Diversiteit (D) | 0–1 |

Voor experimenten A en D (geen retrieval) worden CGR, H en CP niet berekend — de overige metrics (stabiliteit, judge kwaliteit, diversiteit) worden proportioneel herwogen.

In [0]:
# ── Integrale vergelijkingstabel ─────────────────────────────────────────────────────────────

# Start met stability
final = stability_summary[["experiment", "stability"]].copy()

# Voeg judge scores toe (0–100 per criterium)
if 'judge_summary' in dir() and not judge_summary.empty:
    if "totaal" in judge_summary.columns:
        judge_avg = judge_summary[["totaal"]].reset_index()
        judge_avg.columns = ["experiment", "judge_score"]
    else:
        judge_avg = judge_summary[[judge_summary.columns[-1]]].reset_index()
        judge_avg.columns = ["experiment", "judge_score"]
    final = final.merge(judge_avg, on="experiment", how="left")
    
    # Voeg individuele criteria toe voor radar chart
    for crit in SCORE_KEYS:
        if crit in judge_summary.columns:
            crit_avg = judge_summary[[crit]].reset_index()
            crit_avg.columns = ["experiment", f"judge_{crit}"]
            final = final.merge(crit_avg, on="experiment", how="left")
else:
    final["judge_score"] = np.nan
    for crit in SCORE_KEYS:
        final[f"judge_{crit}"] = np.nan

# Voeg CoT kwaliteit toe
if 'df_cot' in dir() and not df_cot.empty:
    cot_avg = df_cot.groupby("experiment").agg(
        cot_quality=("quality", "mean"),
        cot_coherence=("coherence", "mean"),
    ).round(3).reset_index()
    final = final.merge(cot_avg, on="experiment", how="left")
else:
    final["cot_quality"] = np.nan
    final["cot_coherence"] = np.nan

# Voeg CGR en hallucinatie toe
if 'cgr_summary' in dir() and not cgr_summary.empty:
    cgr_reset = cgr_summary.reset_index()
    final = final.merge(
        cgr_reset[["experiment", "avg_CGR", "avg_hallucination"]],
        on="experiment", how="left"
    )
else:
    final["avg_CGR"] = np.nan
    final["avg_hallucination"] = np.nan

# Voeg context precision toe
if 'cp_summary' in dir() and not cp_summary.empty:
    cp_reset = cp_summary.reset_index()
    final = final.merge(
        cp_reset[["experiment", "avg_context_precision"]],
        on="experiment", how="left"
    )
else:
    final["avg_context_precision"] = np.nan

# Voeg diversiteit toe
if 'diversity_summary' in dir() and not diversity_summary.empty:
    div_reset = diversity_summary.reset_index()
    final = final.merge(
        div_reset[["experiment", "avg_diversity"]],
        on="experiment", how="left"
    )
else:
    final["avg_diversity"] = np.nan

# Voeg efficiëntie toe
eff = df_runs.groupby("experiment").agg(
    avg_latency=("latency_seconds", "mean"),
    avg_tokens=("total_tokens", "mean"),
).round(1).reset_index()
final = final.merge(eff, on="experiment", how="left")

# ── Efficiëntie-scores voor radar (lager = beter → inversie) ──
for col, out_col in [("avg_latency", "latency_score"), ("avg_tokens", "token_score")]:
    vals = final[col].dropna()
    if len(vals) > 0 and vals.max() > 0:
        final[out_col] = 1 - (final[col] - vals.min()) / (vals.max() - vals.min()) if vals.max() != vals.min() else 1.0
        final[out_col] = 0.2 + 0.8 * final[out_col]
    else:
        final[out_col] = np.nan

# ── Basisscore (originele formule uit documentatie) ──
def calc_base_score(row):
    s = row["stability"] if pd.notna(row["stability"]) else 0
    cgr = row["avg_CGR"] if pd.notna(row["avg_CGR"]) else 0
    h_inv = (1 - row["avg_hallucination"]) if pd.notna(row["avg_hallucination"]) else 0
    cp = row["avg_context_precision"] if pd.notna(row["avg_context_precision"]) else 0
    
    if row["experiment"] in RETRIEVAL_EXPERIMENTS:
        return (
            WEIGHTS["stability"] * s +
            WEIGHTS["cgr"] * cgr +
            WEIGHTS["hallucination_inverse"] * h_inv +
            WEIGHTS["context_precision"] * cp
        )
    else:
        return s

final["basisscore"] = final.apply(calc_base_score, axis=1).round(3)

# ── Uitgebreide score (incl. judge kwaliteit en diversiteit) ──
def calc_extended_score(row):
    w = WEIGHTS_EXTENDED
    
    s = row["stability"] if pd.notna(row["stability"]) else 0
    q = row["judge_score"] / 100 if pd.notna(row["judge_score"]) else 0
    cgr = row["avg_CGR"] if pd.notna(row["avg_CGR"]) else 0
    h_inv = (1 - row["avg_hallucination"]) if pd.notna(row["avg_hallucination"]) else 0
    cp = row["avg_context_precision"] if pd.notna(row["avg_context_precision"]) else 0
    d = row["avg_diversity"] if pd.notna(row["avg_diversity"]) else 0
    
    available_weights = {}
    available_weights["stability"] = w["stability"]
    available_weights["judge_score"] = w["judge_score"]
    available_weights["diversity"] = w["diversity"]
    
    if row["experiment"] in RETRIEVAL_EXPERIMENTS:
        available_weights["cgr"] = w["cgr"]
        available_weights["hallucination_inverse"] = w["hallucination_inverse"]
        available_weights["context_precision"] = w["context_precision"]
    
    total_w = sum(available_weights.values())
    
    score = 0
    score += (available_weights["stability"] / total_w) * s
    score += (available_weights["judge_score"] / total_w) * q
    score += (available_weights["diversity"] / total_w) * d
    
    if row["experiment"] in RETRIEVAL_EXPERIMENTS:
        score += (available_weights["cgr"] / total_w) * cgr
        score += (available_weights["hallucination_inverse"] / total_w) * h_inv
        score += (available_weights["context_precision"] / total_w) * cp
    
    return score

final["uitgebreide_score"] = final.apply(calc_extended_score, axis=1).round(3)
final = final.sort_values("uitgebreide_score", ascending=False)

print("INTEGRALE VERGELIJKING VAN DE ARCHITECTUREN")
print("=" * 70)
display(final)

# ── Visualisatie: beide scores naast elkaar ──
palette = {"a": "#4C72B0", "b": "#55A868", "c": "#C44E52", "d": "#8172B2"}
final_sorted = final.sort_values("experiment")
exp_colors = [palette.get(e, "#999999") for e in final_sorted["experiment"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(final_sorted["experiment"], final_sorted["basisscore"], color=exp_colors)
axes[0].set_title("Basisscore (S + CGR + H + CP)")
axes[0].set_ylabel("Score")
axes[0].set_ylim(0, 1.1)
for i, v in enumerate(final_sorted["basisscore"]):
    axes[0].text(i, v + 0.02, f"{v:.3f}", ha="center", fontweight="bold")

axes[1].bar(final_sorted["experiment"], final_sorted["uitgebreide_score"], color=exp_colors)
axes[1].set_title("Uitgebreide score (+ Judge + Diversiteit)")
axes[1].set_ylabel("Score")
axes[1].set_ylim(0, 1.1)
for i, v in enumerate(final_sorted["uitgebreide_score"]):
    axes[1].text(i, v + 0.02, f"{v:.3f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

# ── Radar chart: alle evaluatiedimensies uitgesplitst ──
# Judge-criteria individueel (0–100 → 0–1) + overige metrics + efficiëntie

radar_metrics = [
    ("stability",               "Stabiliteit",          "direct"),
    ("judge_probleemdefinitie", "Probleem-\ndefinitie",  "div100"),
    ("judge_analyse",           "Analyse",              "div100"),
    ("judge_scenarios",         "Scenario's",           "div100"),
    ("judge_advies",            "Advies",               "div100"),
    ("judge_cot_samenhang",     "CoT\nsamenhang",       "div100"),
    ("avg_CGR",                 "CGR",                  "direct"),
    ("avg_diversity",           "Diversiteit",          "direct"),
    ("latency_score",           "Snelheid\n(↑=sneller)", "direct"),
    ("token_score",             "Token-\nefficiëntie\n(↑=zuiniger)", "direct"),
]

metric_keys   = [m[0] for m in radar_metrics]
metric_labels = [m[1] for m in radar_metrics]
metric_norms  = [m[2] for m in radar_metrics]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
angles = np.linspace(0, 2 * np.pi, len(metric_keys), endpoint=False).tolist()
angles += angles[:1]

for exp in EXPERIMENTS:
    exp_rows = final[final["experiment"] == exp]
    if exp_rows.empty:
        continue
    row = exp_rows.iloc[0]
    values = []
    for key, norm in zip(metric_keys, metric_norms):
        v = row.get(key, np.nan)
        if pd.isna(v):
            values.append(0)
        elif norm == "div100":
            values.append(v / 100)   # 0–100 → 0–1
        else:
            values.append(float(v))  # al 0–1
    values += values[:1]
    ax.plot(angles, values, "o-", linewidth=2,
            label=f"Experiment {exp.upper()}", color=palette.get(exp))
    ax.fill(angles, values, alpha=0.08, color=palette.get(exp))

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metric_labels, fontsize=8)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=7, color="grey")
ax.set_title("Vergelijking per evaluatiedimensie (alle criteria + efficiëntie)", pad=25, fontsize=12)
ax.legend(loc="upper right", bbox_to_anchor=(1.45, 1.1))

fig.text(0.02, 0.02,
         "Snelheid & Token-efficiëntie: geïnverteerd geschaald (↑ = beter).\n"
         "Laagste latency/tokengebruik → 1.0, hoogste → 0.2",
         fontsize=7, color="grey", style="italic")
plt.tight_layout()
plt.show()

## 4.11 Statistische toetsing
> **Adresseert:** *Zijn de gevonden verschillen tussen architecturen statistisch significant?*

Met N=5 cases per experiment zijn de steekproeven klein. Drie methoden worden gecombineerd:

1. **Wilcoxon signed-rank test** — niet-parametrische paarsgewijze toets die geen normaliteitaanname vereist
2. **Bootstrap 95% CI** — herhaald resampling (10.000×) van het verschil in gemiddelden
3. **Cliff’s delta** — non-parametrische effectgrootte (verwaarloosbaar / klein / middelgroot / groot)

**Multiple testing correctie:** Holm-Bonferroni per metriek (controleert FWER, minder conservatief dan Bonferroni)

| Cel | Analyse |
|-----|---------|
| 4.11 | Wilcoxon + Bootstrap CI per metriek |
| **4.11a** | **Holm-correctie + Cliff’s delta effectgroottes** |

*p < 0.05 = significant, p < 0.10 = marginaal significant*

In [0]:
# ── Statistische toetsing van experimenten ─────────────────────────────
from scipy.stats import wilcoxon, mannwhitneyu
from itertools import combinations

np.random.seed(42)
N_BOOTSTRAP = 10_000

def bootstrap_ci(values_a, values_b, n_boot=N_BOOTSTRAP, ci=0.95):
    """Bootstrap 95% CI voor het verschil in gemiddelden (A - B)."""
    a = np.array(values_a, dtype=float)
    b = np.array(values_b, dtype=float)
    # Verwijder NaN paren
    mask = ~(np.isnan(a) | np.isnan(b))
    a, b = a[mask], b[mask]
    if len(a) < 2:
        return np.nan, np.nan, np.nan
    diffs = []
    for _ in range(n_boot):
        idx = np.random.randint(0, len(a), size=len(a))
        diffs.append(a[idx].mean() - b[idx].mean())
    diffs = np.array(diffs)
    alpha = (1 - ci) / 2
    return np.mean(diffs), np.percentile(diffs, alpha * 100), np.percentile(diffs, (1 - alpha) * 100)

def safe_wilcoxon(a, b):
    """Wilcoxon test met NaN-handling en minimum sample check."""
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    mask = ~(np.isnan(a) | np.isnan(b))
    a, b = a[mask], b[mask]
    diff = a - b
    # Verwijder nullen (Wilcoxon vereist niet-nul verschillen)
    diff = diff[diff != 0]
    if len(diff) < 3:
        return np.nan  # te weinig observaties
    try:
        _, p = wilcoxon(diff)
        return p
    except Exception:
        return np.nan

# ── Bereid case-level data voor per metric ──
print("STATISTISCHE TOETSING")
print("=" * 70)

# 1. Judge totaalscore per case (van df_judge)
metric_dfs = {}

if 'df_judge' in dir() and not df_judge.empty:
    judge_by_case = df_judge.pivot_table(
        index="case_id", columns="experiment", values="totaal"
    )
    metric_dfs["Judge totaal (0-100)"] = judge_by_case

# 2. Stabiliteit per case (van stability_by_case)
if 'stability_by_case' in dir() and not stability_by_case.empty:
    stab_by_case = stability_by_case.pivot_table(
        index="case_id", columns="experiment", values="stability"
    )
    metric_dfs["Stabiliteit"] = stab_by_case

# 3. CGR per case
if 'df_cgr' in dir() and not df_cgr.empty:
    cgr_by_case = df_cgr.pivot_table(
        index="case_id", columns="experiment", values="CGR"
    )
    metric_dfs["CGR"] = cgr_by_case

# 4. Diversiteit per case
if 'df_diversity' in dir() and not df_diversity.empty:
    div_by_case = df_diversity.pivot_table(
        index="case_id", columns="experiment", values="diversity"
    )
    metric_dfs["Diversiteit"] = div_by_case

# 5. Pairwise ranking per case (gemiddeld over criteria)
if 'df_pairwise' in dir() and not df_pairwise.empty:
    rank_by_case = df_pairwise.groupby(["case_id", "experiment"])["rank"].mean().unstack()
    metric_dfs["Pairwise rank (lager=beter)"] = rank_by_case

# ── Voer toetsen uit voor alle metric/experiment-paren ──
results_rows = []

for metric_name, df_metric in metric_dfs.items():
    available = [e for e in EXPERIMENTS if e in df_metric.columns]
    
    for e1, e2 in combinations(available, 2):
        vals_1 = df_metric[e1].values
        vals_2 = df_metric[e2].values
        
        # Wilcoxon
        p_val = safe_wilcoxon(vals_1, vals_2)
        
        # Bootstrap CI
        mean_diff, ci_lo, ci_hi = bootstrap_ci(vals_1, vals_2)
        
        # Significantie
        if pd.notna(p_val):
            if p_val < 0.05:
                sig = "✅ p<0.05"
            elif p_val < 0.10:
                sig = "⚠️ p<0.10"
            else:
                sig = "❌ n.s."
        else:
            sig = "— n.v.t."
        
        # CI bevat nul?
        ci_sig = ""
        if pd.notna(ci_lo) and pd.notna(ci_hi):
            if ci_lo > 0 or ci_hi < 0:
                ci_sig = " (CI excl. 0)"
            else:
                ci_sig = " (CI incl. 0)"
        
        results_rows.append({
            "Metric": metric_name,
            "Vergelijking": f"{e1.upper()} vs {e2.upper()}",
            "Gem. verschil": round(mean_diff, 3) if pd.notna(mean_diff) else np.nan,
            "95% CI": f"[{ci_lo:.3f}, {ci_hi:.3f}]" if pd.notna(ci_lo) else "—",
            "Wilcoxon p": round(p_val, 4) if pd.notna(p_val) else np.nan,
            "Significantie": sig + ci_sig,
        })

df_stats = pd.DataFrame(results_rows)

# ── Toon resultaten ──
for metric_name in metric_dfs:
    subset = df_stats[df_stats["Metric"] == metric_name]
    if not subset.empty:
        print(f"\n{metric_name}:")
        display(subset[["Vergelijking", "Gem. verschil", "95% CI", "Wilcoxon p", "Significantie"]].reset_index(drop=True))

# ── Samenvatting: significante verschillen ──
sig_results = df_stats[df_stats["Significantie"].str.contains("✅|⚠️", na=False)]
print(f"\n{'=' * 70}")
print(f"SAMENVATTING: {len(sig_results)} van {len(df_stats)} vergelijkingen zijn (marginaal) significant")
if not sig_results.empty:
    print("\nSignificante verschillen:")
    for _, r in sig_results.iterrows():
        print(f"  {r['Metric']} | {r['Vergelijking']}: verschil={r['Gem. verschil']}, p={r['Wilcoxon p']}, {r['Significantie']}")
else:
    print("\nGeen significante verschillen gevonden.")
    print("NB: Met N=5 cases is de statistische power zeer beperkt.")
    print("Overweeg meer cases toe te voegen voor sterkere conclusies.")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# 4.11a Correctie voor multiple testing + effectgroottes
# ═══════════════════════════════════════════════════════════════════════
# Voegt Holm-Bonferroni correctie en Cliff's delta (non-parametrische
# effectgrootte) toe aan de statistische toetsing uit §4.11.
#
# Cliff's delta interpretatie:
#   |d| < 0.147 = verwaarloosbaar
#   |d| < 0.33  = klein
#   |d| < 0.474 = middelgroot
#   |d| ≥ 0.474 = groot

from statsmodels.stats.multitest import multipletests

def cliffs_delta(x, y):
    """Cliff's delta: non-parametrische effectgrootte.
    Vergelijkt alle paren (xi, yj) en telt dominantie.
    Bereik: -1 tot 1, 0 = geen verschil."""
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    x = x[~np.isnan(x)]
    y = y[~np.isnan(y)]
    if len(x) == 0 or len(y) == 0:
        return np.nan
    n_more = sum(1 for xi in x for yj in y if xi > yj)
    n_less = sum(1 for xi in x for yj in y if xi < yj)
    return (n_more - n_less) / (len(x) * len(y))

def interpret_cliffs_delta(d):
    """Interpreteer Cliff's delta volgens Vargha & Delaney (2000)."""
    if np.isnan(d):
        return "—"
    ad = abs(d)
    if ad < 0.147:
        return "verwaarloosbaar"
    elif ad < 0.33:
        return "klein"
    elif ad < 0.474:
        return "middelgroot"
    else:
        return "groot"

# ── 1. Bereken Cliff's delta per vergelijking ──
print("EFFECTGROOTTES EN CORRECTIE VOOR MULTIPLE TESTING")
print("=" * 75)

effect_rows = []

for metric_name, df_metric in metric_dfs.items():
    available = [e for e in EXPERIMENTS if e in df_metric.columns]
    
    for e1, e2 in combinations(available, 2):
        vals_1 = df_metric[e1].dropna().values
        vals_2 = df_metric[e2].dropna().values
        
        d = cliffs_delta(vals_1, vals_2)
        interp = interpret_cliffs_delta(d)
        
        # Haal originele p-waarde op uit df_stats
        mask = (df_stats["Metric"] == metric_name) & \
               (df_stats["Vergelijking"] == f"{e1.upper()} vs {e2.upper()}")
        orig_p = df_stats.loc[mask, "Wilcoxon p"].values
        orig_p = orig_p[0] if len(orig_p) > 0 else np.nan
        
        effect_rows.append({
            "Metric": metric_name,
            "Vergelijking": f"{e1.upper()} vs {e2.upper()}",
            "Cliff's delta": round(d, 3) if not np.isnan(d) else np.nan,
            "Effectgrootte": interp,
            "Wilcoxon p (origineel)": orig_p,
        })

df_effects = pd.DataFrame(effect_rows)

# ── 2. Holm-Bonferroni correctie per metriek ──
all_corrected = []

for metric_name in metric_dfs:
    sub = df_effects[df_effects["Metric"] == metric_name].copy()
    p_vals = sub["Wilcoxon p (origineel)"].values
    
    # Vervang NaN door 1.0 voor multipletests
    p_clean = np.where(np.isnan(p_vals), 1.0, p_vals)
    
    if len(p_clean) > 1:
        reject, p_corrected, _, _ = multipletests(p_clean, method="holm")
    else:
        p_corrected = p_clean
        reject = p_clean < 0.05
    
    sub["p (Holm-corrected)"] = [round(p, 4) if not np.isnan(orig) else np.nan 
                                  for p, orig in zip(p_corrected, p_vals)]
    sub["Significant (Holm)"] = ["✅" if r and not np.isnan(orig) else "❌" 
                                  for r, orig in zip(reject, p_vals)]
    all_corrected.append(sub)

df_corrected = pd.concat(all_corrected, ignore_index=True)

# ── 3. Resultaten per metriek ──
for metric_name in metric_dfs:
    sub = df_corrected[df_corrected["Metric"] == metric_name]
    if not sub.empty:
        print(f"\n{metric_name}:")
        display(sub[["Vergelijking", "Cliff's delta", "Effectgrootte", 
                     "Wilcoxon p (origineel)", "p (Holm-corrected)", "Significant (Holm)"]]
                .reset_index(drop=True))

# ── 4. Samenvatting ──
print("\n" + "=" * 75)
print("SAMENVATTING STATISTISCHE ROBUUSTHEID")
print("=" * 75)

# Hoeveel p-waarden overleven Holm-correctie?
n_orig_sig = df_corrected["Wilcoxon p (origineel)"].dropna().apply(lambda x: x < 0.05).sum()
n_holm_sig = df_corrected["Significant (Holm)"].apply(lambda x: x == "✅").sum()
n_total = df_corrected["Wilcoxon p (origineel)"].notna().sum()

print(f"\nTotaal vergelijkingen: {n_total}")
print(f"Significant (p<0.05, ongecorrigeerd): {n_orig_sig}")
print(f"Significant (Holm-Bonferroni): {n_holm_sig}")
if n_orig_sig > n_holm_sig:
    print(f"→ {n_orig_sig - n_holm_sig} resultaten overleven Holm-correctie NIET")

# Effectgroottes samenvatten
print("\nVerdeling effectgroottes (Cliff's delta):")
for size in ["groot", "middelgroot", "klein", "verwaarloosbaar"]:
    n = (df_corrected["Effectgrootte"] == size).sum()
    if n > 0:
        examples = df_corrected[df_corrected["Effectgrootte"] == size][
            ["Metric", "Vergelijking", "Cliff's delta"]
        ].head(3)
        cd_col = "Cliff's delta"
        ex_parts = [f"{r['Vergelijking']}({r[cd_col]:.2f})" for _, r in examples.iterrows()]
        ex_str = "; ".join(ex_parts)
        print(f"  {size:>16s}: {n}x  (bijv. {ex_str})")

print("\nMethodologische noot:")
print("  Holm-Bonferroni is minder conservatief dan Bonferroni maar")
print("  controleert nog steeds de family-wise error rate (FWER).")
print("  Met N=5 cases is de power laag; grote effecten (Δ≥0.474) kunnen")
print("  toch niet-significant zijn door beperkte steekproefgrootte.")

## 4.12 Verbosity bias: is langer = beter?
> **Adresseert:** *Beoordeelt de LLM-judge langere antwoorden systematisch hoger?*

Experiment D (Self-Refine) genereert 3× meer tokens dan de baseline. Als de judge langere antwoorden hoger beoordeelt (verbosity bias), dan is D’s hogere rubric-score een artefact van lengte, niet van kwaliteit.

Deze analyse meet:
1. **Antwoordlengte per experiment** (in characters en tokens)
2. **Correlatie** lengte ↔ judge totaalscore
3. **Lengte-gecorrigeerde score** — resteert het verschil na correctie voor lengte?

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# 4.10 Verbosity bias analyse
# ═══════════════════════════════════════════════════════════════════════

# 1. Antwoordlengte per experiment
df_length = df_runs[df_runs["run"] == 1].copy()
df_length["answer_chars"] = df_length["answer"].str.len()
df_length["answer_words"] = df_length["answer"].str.split().str.len()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1a. Boxplot woordaantal
sns.boxplot(data=df_length, x="experiment", y="answer_words",
            order=EXPERIMENTS, palette=palette, ax=axes[0])
axes[0].set_title("Antwoordlengte per experiment")
axes[0].set_ylabel("Aantal woorden")
for i, exp in enumerate(EXPERIMENTS):
    med = df_length[df_length["experiment"]==exp]["answer_words"].median()
    axes[0].text(i, med + 50, f"{med:.0f}", ha="center", fontweight="bold", fontsize=9)

# 2. Correlatie lengte ↔ judge score
df_corr = df_length.merge(df_judge[["experiment", "case_id", "totaal"]], on=["experiment", "case_id"])

sns.scatterplot(data=df_corr, x="answer_words", y="totaal",
                hue="experiment", style="experiment",
                palette=palette, s=120, ax=axes[1])

# Trendlijn
from numpy.polynomial.polynomial import polyfit
x_all = df_corr["answer_words"].values
y_all = df_corr["totaal"].values
mask = ~(np.isnan(x_all) | np.isnan(y_all))
if mask.sum() > 2:
    b, m = polyfit(x_all[mask], y_all[mask], 1)
    x_line = np.linspace(x_all[mask].min(), x_all[mask].max(), 100)
    axes[1].plot(x_line, b + m * x_line, "--", color="gray", alpha=0.7)
    r = np.corrcoef(x_all[mask], y_all[mask])[0, 1]
    axes[1].set_title(f"Lengte vs Judge score (r = {r:.3f})")
else:
    axes[1].set_title("Lengte vs Judge score")
axes[1].set_xlabel("Aantal woorden")
axes[1].set_ylabel("Judge totaalscore (0-100)")

# 3. Lengte-gecorrigeerd: residuen na lineaire regressie
if mask.sum() > 2:
    predicted = b + m * df_corr["answer_words"].values
    df_corr["residual"] = df_corr["totaal"] - predicted
    
    resid_by_exp = df_corr.groupby("experiment")["residual"].mean().reindex(EXPERIMENTS)
    colors = [palette[e] for e in EXPERIMENTS]
    bars = axes[2].bar(EXPERIMENTS, resid_by_exp.values, color=colors)
    axes[2].axhline(y=0, color="gray", linestyle="--", alpha=0.5)
    axes[2].set_title("Lengte-gecorrigeerde kwaliteit\n(residuelen na correctie voor woordaantal)")
    axes[2].set_ylabel("Residu (+ = beter dan verwacht o.b.v. lengte)")
    axes[2].set_xlabel("Experiment")
    for i, v in enumerate(resid_by_exp.values):
        axes[2].text(i, v + 0.1 * np.sign(v), f"{v:+.1f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

# Samenvatting
print("\n=== Verbosity analyse ===")
length_summary = df_length.groupby("experiment").agg(
    gem_woorden=("answer_words", "mean"),
    gem_chars=("answer_chars", "mean"),
    gem_tokens=("total_tokens", "mean"),
).round(0).reindex(EXPERIMENTS)
print(length_summary)
print(f"\nPearson correlatie (woorden ↔ judge score): r = {r:.3f}")
if abs(r) < 0.3:
    print("→ Zwakke correlatie: verbosity bias is BEPERKT")
elif abs(r) < 0.6:
    print("→ Matige correlatie: enige verbosity bias aanwezig")
else:
    print("→ Sterke correlatie: SIGNIFICANTE verbosity bias — langere antwoorden scoren hoger")

if mask.sum() > 2:
    print(f"\nNa lengte-correctie (residuelen):")
    for exp in EXPERIMENTS:
        v = resid_by_exp[exp]
        print(f"  {exp.upper()}: {v:+.1f} {'(beter dan verwacht)' if v > 0 else '(slechter dan verwacht)'}")


## 4.13 De Judge-Ranking contradictie
> **Kernbevinding:** *Experiment D scoort het hoogst op absolute rubric-beoordeling, maar eindigt consistent laatst in pairwise vergelijking.*

Dit is de belangrijkste methodologische bevinding van dit onderzoek. Twee evaluatiemethoden — rubric scoring en pairwise ranking — geven tegengestelde conclusies over dezelfde adviezen.

Mogelijke verklaringen:
1. **Verbosity bias** — De judge beloont langere, gedetailleerdere antwoorden op individuele criteria, maar bij directe vergelijking wint een beknopter, coherenter advies
2. **Coherentie vs. detail** — D’s self-refine voegt detail toe per stap, maar verliest overzicht en leesbaarheid
3. **Methodologische implicatie** — Absolute LLM-as-judge is onvoldoende; pairwise ranking is betrouwbaarder voor vergelijkend onderzoek

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# 4.11 Judge-Ranking contradictie
# ═══════════════════════════════════════════════════════════════════════

# Combineer judge scores en pairwise ranks per experiment/case
df_contradiction = df_judge[["experiment", "case_id", "totaal"]].copy()
df_contradiction.columns = ["experiment", "case_id", "judge_score"]

# Gemiddelde pairwise rank per experiment/case (over criteria)
pw_avg = df_pairwise.groupby(["experiment", "case_id"])["rank"].mean().reset_index()
pw_avg.columns = ["experiment", "case_id", "pairwise_rank"]

df_contradiction = df_contradiction.merge(pw_avg, on=["experiment", "case_id"], how="inner")

palette = {"a": "#4C72B0", "b": "#55A868", "c": "#C44E52", "d": "#8172B2"}

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Scatter: judge score vs pairwise rank (per case)
for exp in EXPERIMENTS:
    subset = df_contradiction[df_contradiction["experiment"] == exp]
    axes[0].scatter(subset["judge_score"], subset["pairwise_rank"],
                    color=palette[exp], s=150, label=f"{exp.upper()}",
                    edgecolors="black", linewidth=0.5, zorder=3)

axes[0].set_xlabel("Judge totaalscore (0-100) →")
axes[0].set_ylabel("← Pairwise rank (1=best)")
axes[0].set_title("Judge score vs Pairwise rank per case")
axes[0].invert_yaxis()  # lager rank = beter = bovenaan
axes[0].legend()
axes[0].axhline(y=2.5, color="gray", linestyle="--", alpha=0.3)
# Annoteer D-cluster
d_data = df_contradiction[df_contradiction["experiment"] == "d"]
if not d_data.empty:
    axes[0].annotate("D: hoge score\nmaar slechte rank",
                     xy=(d_data["judge_score"].mean(), d_data["pairwise_rank"].mean()),
                     xytext=(d_data["judge_score"].mean()-5, 1.5),
                     arrowprops=dict(arrowstyle="->", color="#8172B2"),
                     fontsize=9, color="#8172B2", fontweight="bold")

# 2. Directe vergelijking: bars naast elkaar
exp_summary = df_contradiction.groupby("experiment").agg(
    judge=('judge_score', 'mean'),
    rank=('pairwise_rank', 'mean')
).reindex(EXPERIMENTS)

x = np.arange(len(EXPERIMENTS))
w = 0.35

# Normaliseer naar zelfde schaal voor vergelijking
judge_norm = (exp_summary["judge"] - exp_summary["judge"].min()) / (exp_summary["judge"].max() - exp_summary["judge"].min())
rank_norm = 1 - (exp_summary["rank"] - 1) / (exp_summary["rank"].max() - 1)  # inversie: lager = beter

bars1 = axes[1].bar(x - w/2, judge_norm, w, label="Judge score (genorm.)",
                     color=[palette[e] for e in EXPERIMENTS], alpha=0.7, edgecolor="black")
bars2 = axes[1].bar(x + w/2, rank_norm, w, label="Pairwise rank (genorm., ↑=beter)",
                     color=[palette[e] for e in EXPERIMENTS], alpha=0.4, edgecolor="black", hatch="//")
axes[1].set_xticks(x)
axes[1].set_xticklabels([e.upper() for e in EXPERIMENTS])
axes[1].set_ylabel("Genormaliseerde score (0-1)")
axes[1].set_title("Contradictie: Judge (vol) vs Ranking (gearceerd)")
axes[1].legend()
axes[1].set_ylim(0, 1.15)

# 3. Contradictie-index per experiment
# = verschil in genormaliseerde rank: hoe hoger, hoe groter de contradictie
exp_summary["contradictie"] = judge_norm.values - rank_norm.values
colors_c = ["#e74c3c" if v > 0.2 else "#f39c12" if v > 0 else "#2ecc71" for v in exp_summary["contradictie"]]
bars3 = axes[2].bar(EXPERIMENTS, exp_summary["contradictie"], color=colors_c, edgecolor="black")
axes[2].axhline(y=0, color="gray", linestyle="--")
axes[2].set_title("Contradictie-index\n(+ = judge overschat t.o.v. ranking)")
axes[2].set_ylabel("Judge (norm.) − Rank (norm.)")
axes[2].set_xlabel("Experiment")
for i, v in enumerate(exp_summary["contradictie"]):
    axes[2].text(i, v + 0.02 * np.sign(v), f"{v:+.2f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

# Samenvatting
print("\n=== Judge-Ranking Contradictie ===")
print(f"{'Exp':>4} | {'Judge':>8} | {'Rank':>8} | {'Contradictie':>12}")
print("-" * 45)
for exp in EXPERIMENTS:
    j = exp_summary.loc[exp, 'judge']
    r = exp_summary.loc[exp, 'rank']
    c = exp_summary.loc[exp, 'contradictie']
    flag = " ⚠️ HOOG" if abs(c) > 0.3 else ""
    print(f"{exp.upper():>4} | {j:>8.1f} | {r:>8.2f} | {c:>+12.2f}{flag}")

print(f"\nCorrelatie judge_score ↔ pairwise_rank: "
      f"r = {df_contradiction['judge_score'].corr(df_contradiction['pairwise_rank']):.3f}")
print("(Negatieve r = hogere judge score → slechtere rank = contradictie)")

## 4.14 Kosten-baten analyse
> **Adresseert:** *Is de extra rekentijd en tokenverbruik van complexere architecturen gerechtvaardigd door betere kwaliteit?*

Voor praktische toepasbaarheid is niet alleen de kwaliteit relevant, maar ook de **efficiëntie**: hoeveel kwaliteitsverbetering levert elke extra seconde of token op?

Metrics:
- **Kwaliteit per 1000 tokens** = judge_score / (tokens / 1000)
- **Kwaliteit per seconde** = judge_score / latency
- **Pareto-optimaliteit** — welk experiment biedt de beste trade-off?

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# 4.12 Kosten-baten analyse
# ═══════════════════════════════════════════════════════════════════════

palette = {"a": "#4C72B0", "b": "#55A868", "c": "#C44E52", "d": "#8172B2"}

# Bereken per experiment
cost_benefit = final[["experiment", "judge_score", "avg_latency", "avg_tokens"]].copy()
cost_benefit["quality_per_1k_tokens"] = cost_benefit["judge_score"] / (cost_benefit["avg_tokens"] / 1000)
cost_benefit["quality_per_second"] = cost_benefit["judge_score"] / cost_benefit["avg_latency"]
cost_benefit = cost_benefit.sort_values("experiment")

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Scatter: tokens vs kwaliteit (met Pareto annotatie)
for _, row in cost_benefit.iterrows():
    exp = row["experiment"]
    axes[0].scatter(row["avg_tokens"], row["judge_score"],
                    color=palette[exp], s=250, zorder=3,
                    edgecolors="black", linewidth=1)
    axes[0].annotate(f" {exp.upper()}", (row["avg_tokens"], row["judge_score"]),
                     fontsize=12, fontweight="bold", color=palette[exp])

axes[0].set_xlabel("Gemiddeld tokenverbruik →")
axes[0].set_ylabel("Judge totaalscore (0-100) →")
axes[0].set_title("Kwaliteit vs Tokens")
# Pareto frontier (bovenlinkse hoek is optimaal)
# Sorteer op tokens, markeer Pareto-optimale punten
sorted_cb = cost_benefit.sort_values("avg_tokens")
pareto_mask = []
best_quality = -1
for _, row in sorted_cb.iterrows():
    if row["judge_score"] > best_quality:
        best_quality = row["judge_score"]
        pareto_mask.append(True)
    else:
        pareto_mask.append(False)
pareto_exps = sorted_cb[pareto_mask]
if len(pareto_exps) > 1:
    axes[0].plot(pareto_exps["avg_tokens"], pareto_exps["judge_score"],
                 "--", color="gray", alpha=0.5, label="Pareto frontier")
    axes[0].legend(fontsize=9)

# 2. Efficiëntie-barplot
x = np.arange(len(EXPERIMENTS))
w = 0.35

bars1 = axes[1].bar(x - w/2, cost_benefit["quality_per_1k_tokens"].values, w,
                     label="Score / 1k tokens",
                     color=[palette[e] for e in EXPERIMENTS], alpha=0.8, edgecolor="black")
ax2 = axes[1].twinx()
bars2 = ax2.bar(x + w/2, cost_benefit["quality_per_second"].values, w,
                label="Score / seconde",
                color=[palette[e] for e in EXPERIMENTS], alpha=0.4, edgecolor="black", hatch="//")
axes[1].set_xticks(x)
axes[1].set_xticklabels([e.upper() for e in EXPERIMENTS])
axes[1].set_ylabel("Kwaliteit per 1000 tokens")
ax2.set_ylabel("Kwaliteit per seconde")
axes[1].set_title("Efficiëntie: kwaliteit per eenheid kost")
axes[1].legend(loc="upper left", fontsize=8)
ax2.legend(loc="upper right", fontsize=8)

# Annoteer bars
for i, v in enumerate(cost_benefit["quality_per_1k_tokens"].values):
    axes[1].text(i - w/2, v + 0.3, f"{v:.1f}", ha="center", fontsize=9, fontweight="bold")
for i, v in enumerate(cost_benefit["quality_per_second"].values):
    ax2.text(i + w/2, v + 0.05, f"{v:.2f}", ha="center", fontsize=9, fontweight="bold")

# 3. Latency vs kwaliteit scatter
for _, row in cost_benefit.iterrows():
    exp = row["experiment"]
    axes[2].scatter(row["avg_latency"], row["judge_score"],
                    color=palette[exp], s=250, zorder=3,
                    edgecolors="black", linewidth=1)
    axes[2].annotate(f" {exp.upper()}", (row["avg_latency"], row["judge_score"]),
                     fontsize=12, fontweight="bold", color=palette[exp])

axes[2].set_xlabel("Gemiddelde latency (seconden) →")
axes[2].set_ylabel("Judge totaalscore (0-100) →")
axes[2].set_title("Kwaliteit vs Snelheid")

plt.tight_layout()
plt.show()

# Samenvattingstabel
print("\n=== Kosten-baten samenvatting ===")
print(f"{'Exp':>4} | {'Score':>6} | {'Tokens':>7} | {'Latency':>8} | {'Score/1kT':>10} | {'Score/sec':>10}")
print("-" * 65)
for _, row in cost_benefit.iterrows():
    print(f"{row['experiment'].upper():>4} | {row['judge_score']:>6.1f} | {row['avg_tokens']:>7.0f} | {row['avg_latency']:>7.1f}s | "
          f"{row['quality_per_1k_tokens']:>10.1f} | {row['quality_per_second']:>10.2f}")

# Conclusie
best_eff = cost_benefit.loc[cost_benefit["quality_per_1k_tokens"].idxmax()]
print(f"\nMeest token-efficiënt: {best_eff['experiment'].upper()} "
      f"({best_eff['quality_per_1k_tokens']:.1f} score per 1k tokens)")
best_speed = cost_benefit.loc[cost_benefit["quality_per_second"].idxmax()]
print(f"Snelste kwaliteit: {best_speed['experiment'].upper()} "
      f"({best_speed['quality_per_second']:.2f} score per seconde)")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# 4.12b Kosten-baten per individuele run
# ═══════════════════════════════════════════════════════════════════════
# Elke run als apart bolletje, kleur = experiment

from scipy import stats

# --- 1. Per-run token data uit cases ---
run_token_data = []
for _, row in df_runs[df_runs['run'].notna()].iterrows():
    exp = row['experiment']
    run_num = int(row['run'])
    cases = row['cases']
    if not isinstance(cases, list):
        continue
    completion_tokens = [c['usage']['completion_tokens'] for c in cases if 'usage' in c]
    run_token_data.append({
        'experiment': exp,
        'run': run_num,
        'avg_completion_tokens': np.mean(completion_tokens),
    })
df_run_tokens = pd.DataFrame(run_token_data)

# --- 2. Per-run judge scores ---
# df_judge heeft per antwoord (experiment × case × run) een score
if 'run' in df_judge.columns:
    df_run_scores = df_judge.groupby(['experiment', 'run'])['totaal'].mean().reset_index()
    df_run_scores.rename(columns={'totaal': 'rubric_score'}, inplace=True)
else:
    # Fallback: judge is alleen per experiment/case → gebruik experiment-gemiddelde per run
    # (dan zijn alle runs binnen een experiment gelijk — minder informatief)
    print("⚠️ df_judge heeft geen 'run' kolom — per-run scores niet beschikbaar.")
    print("   Herbereken judge scores per individuele run (zie cel hieronder).")
    df_run_scores = df_run_tokens.copy()
    exp_scores = cost_benefit.set_index('experiment')['judge_score']
    df_run_scores['rubric_score'] = df_run_scores['experiment'].map(exp_scores)

# --- 3. Merge ---
df_plot = df_run_tokens.merge(df_run_scores[['experiment', 'run', 'rubric_score']],
                              on=['experiment', 'run'])

# --- 4. Plot ---
fig, ax = plt.subplots(figsize=(10, 7))

for exp in EXPERIMENTS:
    sub = df_plot[df_plot['experiment'] == exp]
    label = f"{exp.upper()} — {EXP_LABELS[exp]}"
    ax.scatter(sub['avg_completion_tokens'], sub['rubric_score'],
               color=palette[exp], s=150, zorder=3,
               edgecolors='black', linewidth=0.8, alpha=0.85,
               label=label)

# Trendlijn
x_all = df_plot['avg_completion_tokens'].values
y_all = df_plot['rubric_score'].values
slope, intercept, r_value, p_value, std_err = stats.linregress(x_all, y_all)

x_line = np.linspace(x_all.min() - 100, x_all.max() + 100, 100)
y_line = slope * x_line + intercept
ax.plot(x_line, y_line, '--', color='gray', alpha=0.7, linewidth=1.5,
        label=f'Trend (r = {r_value:.2f})')

ax.set_xlabel('Gemiddeld tokengebruik (completion tokens)', fontsize=11)
ax.set_ylabel('Gemiddelde rubric score (0-100)', fontsize=11)
ax.set_title(f'Tokengebruik vs. rubric score\nr = {r_value:.2f} — meer tokens = hogere beoordeling',
             fontsize=13)
ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig_4_9_kosten_baten_v3_per_run.png'),
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(f"\nCorrelatie: r = {r_value:.3f}, p = {p_value:.4f}")
print(f"N = {len(df_plot)} runs (5 per experiment)")

## 4.15 Convergentie van strategische opties
> **Adresseert:** *Hoe dicht bij elkaar of ver van elkaar liggen de gegenereerde strategische opties tussen architecturen?*

De voorgaande analyses vergelijken volledige antwoorden (∼900 woorden). Maar het kernproduct van de LLM is **stap 6: Strategische opties** — de concrete strategieën die aan de restauranthouder worden voorgelegd.

Deze analyse isoleert de strategische opties en meet:

1. **Optie-extractie** — Individuele strategieën worden geëxtraheerd uit stap 6 per experiment/case
2. **Semantische afstand** — Embedding-gebaseerde pairwise similarity tússen architecturen, per case
3. **Thematische classificatie** — LLM labelt elke optie met een strategisch thema (pricing, assortiment, marketing, etc.)
4. **Convergentie-index** — In hoeverre convergeren de architecturen op dezelfde strategieën?
5. **2D-visualisatie** — UMAP-projectie van de opties-ruimte, gekleurd per experiment

**Hypothese:** RAG (B) en Web search (C) zouden systematisch andere opties moeten genereren dan de baseline (A), omdat zij extra contextuele informatie meekrijgen. Self-Refine (D) zou op dezelfde opties moeten uitkomen als A, maar dieper uitgewerkt.

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# 4.14a  Strategische opties extractie + embedding
# ═══════════════════════════════════════════════════════════════════════
import sys
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from utils.steps import extract_steps

# ── Stap 1: Extraheer stap 6 (Strategische opties) per experiment/case ──
df_run1 = df_runs[df_runs["run"] == 1].copy()

option_rows = []
for _, row in df_run1.iterrows():
    steps = extract_steps(row["answer"])
    step6 = steps.get("6", {}).get("text", "")
    if not step6:
        print(f"  ⚠️ Geen stap 6 gevonden voor {row['experiment']}/{row['case_id']}")
        continue

    # Split individuele opties (zoek patronen als "Optie 1:", "**Optie A:**", genummerde items)
    options = re.split(
        r'(?:^|\n)\s*(?:\*\*)?(?:Optie|Strategie|Option)\s*[\dA-Dabcd]+[.:)]',
        step6, flags=re.IGNORECASE
    )
    # Filter lege strings en header
    options = [o.strip() for o in options if o.strip() and len(o.strip()) > 50]

    if not options:
        # Fallback: splits op dubbele newlines of genummerde lijsten
        options = re.split(r'\n\s*(?:\d+\.\s|[-•]\s\*\*)', step6)
        options = [o.strip() for o in options if o.strip() and len(o.strip()) > 50]

    if not options:
        # Laatste fallback: gebruik hele stap 6 als één optie
        options = [step6]

    for i, opt_text in enumerate(options):
        option_rows.append({
            "experiment": row["experiment"],
            "case_id": row["case_id"],
            "option_idx": i + 1,
            "option_text": opt_text[:2000],  # cap lengte
            "option_chars": len(opt_text),
        })

df_options = pd.DataFrame(option_rows)
print(f"Geëxtraheerd: {len(df_options)} strategische opties uit {len(df_run1)} antwoorden")
print(f"\nAantal opties per experiment:")
print(df_options.groupby("experiment")["option_idx"].count().to_string())
print(f"\nAantal opties per experiment/case:")
print(df_options.groupby(["experiment", "case_id"]).size().unstack(fill_value=0).to_string())

# ── Stap 2: Embed alle opties ──
print("\nEmbedding berekenen...")
option_embeddings = embedding_model.encode(
    df_options["option_text"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)
df_options["embedding_idx"] = range(len(df_options))
print(f"Embeddings shape: {option_embeddings.shape}")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# 4.14b  Semantische afstand + convergentie-analyse
# ═══════════════════════════════════════════════════════════════════════
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
from itertools import combinations

palette = {"a": "#4C72B0", "b": "#55A868", "c": "#C44E52", "d": "#8172B2"}
EXP_LABELS = {"a": "Baseline", "b": "RAG", "c": "Web search", "d": "Self-Refine"}

# ── 1. Pairwise similarity BINNEN elke case, TUSSEN experimenten ──
# Methode: voor elke case, bereken max-match similarity tussen opties van exp_i en exp_j
# (best-match = per optie van exp_i, neem de maximale similarity met enige optie van exp_j)

def option_set_similarity(idxs_a, idxs_b, embeddings):
    """Gemiddelde best-match cosine similarity tussen twee sets opties.
    Symmetrisch: gemiddelde van A→B en B→A best-match."""
    if len(idxs_a) == 0 or len(idxs_b) == 0:
        return np.nan
    emb_a = embeddings[idxs_a]
    emb_b = embeddings[idxs_b]
    sim = cos_sim(emb_a, emb_b)  # (|A|, |B|)
    # Best match A→B: voor elke optie in A, de max similarity met B
    best_a_to_b = sim.max(axis=1).mean()
    # Best match B→A: voor elke optie in B, de max similarity met A
    best_b_to_a = sim.max(axis=0).mean()
    return (best_a_to_b + best_b_to_a) / 2

# Bereken per case en per experiment-paar
conv_rows = []
cases = sorted(df_options["case_id"].unique())

for case in cases:
    case_opts = df_options[df_options["case_id"] == case]
    for e1, e2 in combinations(EXPERIMENTS, 2):
        idxs_1 = case_opts[case_opts["experiment"] == e1]["embedding_idx"].values
        idxs_2 = case_opts[case_opts["experiment"] == e2]["embedding_idx"].values
        sim = option_set_similarity(idxs_1, idxs_2, option_embeddings)
        conv_rows.append({
            "case_id": case,
            "pair": f"{e1.upper()} vs {e2.upper()}",
            "exp_1": e1, "exp_2": e2,
            "similarity": round(sim, 4) if not np.isnan(sim) else np.nan,
        })

df_conv = pd.DataFrame(conv_rows)

# ── 2. Visualisatie: heatmaps + convergentie-index ──
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 2a. Heatmap: gemiddelde optie-similarity per experiment-paar
pivot_sim = df_conv.pivot_table(index="exp_1", columns="exp_2", values="similarity", aggfunc="mean")
# Maak symmetrische matrix
full_matrix = np.eye(len(EXPERIMENTS))
for i, e1 in enumerate(EXPERIMENTS):
    for j, e2 in enumerate(EXPERIMENTS):
        if i < j:
            val = pivot_sim.loc[e1, e2] if (e1 in pivot_sim.index and e2 in pivot_sim.columns) else np.nan
            full_matrix[i, j] = val
            full_matrix[j, i] = val

sns.heatmap(
    pd.DataFrame(full_matrix, index=[e.upper() for e in EXPERIMENTS],
                 columns=[e.upper() for e in EXPERIMENTS]),
    annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.5, vmax=1.0,
    ax=axes[0], linewidths=0.5
)
axes[0].set_title("Gem. optie-similarity tussen architecturen\n(best-match op stap 6)")

# 2b. Similarity per case (grouped bar)
pivot_by_case = df_conv.pivot_table(index="pair", columns="case_id", values="similarity")
pivot_by_case.plot(kind="bar", ax=axes[1], colormap="Set2", edgecolor="black")
axes[1].set_ylabel("Best-match similarity")
axes[1].set_title("Optie-convergentie per case")
axes[1].set_ylim(0.4, 1.0)
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(fontsize=8, ncol=2)

# 2c. Vergelijking: volledige antwoord similarity vs optie-similarity
# (toont of stap 6 meer/minder varieert dan het geheel)
full_answer_sim = {}  # uit §4.5
for _, row_sim in similarity_df.iterrows():
    for col in similarity_df.columns:
        if col != "case_id" and "_vs_" in col:
            key = (row_sim["case_id"], col.replace("_", " "))
            full_answer_sim[key] = row_sim[col]

# Match met optie-level sim
comparison_rows = []
for _, row in df_conv.iterrows():
    pair_key = f"{row['exp_1'].upper()} vs {row['exp_2'].upper()}"
    full_key = (row["case_id"], pair_key)
    full_sim = full_answer_sim.get(full_key, np.nan)
    comparison_rows.append({
        "pair": pair_key,
        "case_id": row["case_id"],
        "Volledig antwoord": full_sim,
        "Strategische opties (stap 6)": row["similarity"],
    })

df_compare = pd.DataFrame(comparison_rows)
df_compare_avg = df_compare.groupby("pair")[["Volledig antwoord", "Strategische opties (stap 6)"]].mean()

df_compare_avg.plot(kind="bar", ax=axes[2], color=["#95a5a6", "#e74c3c"],
                    edgecolor="black", width=0.7)
axes[2].set_ylabel("Cosine similarity")
axes[2].set_title("Volledig antwoord vs Strategische opties\n(lagere stap-6 sim = meer diversificatie)")
axes[2].set_ylim(0.4, 1.0)
axes[2].tick_params(axis='x', rotation=45)
axes[2].axhline(y=0.8, color="gray", linestyle="--", alpha=0.3)
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

# ── Convergentie-index samenvatting ──
print("\n=== Convergentie-analyse strategische opties ===")
print(f"\nGemiddelde best-match similarity per experiment-paar:")
for _, row in df_conv.groupby("pair")["similarity"].agg(["mean", "std"]).iterrows():
    sim = row["mean"]
    label = "✅ Hoge convergentie" if sim > 0.85 else "⚠️ Matige convergentie" if sim > 0.7 else "❌ Lage convergentie"
    print(f"  {_:12s}: {sim:.3f} (±{row['std']:.3f}) {label}")

print(f"\nVergelijking met volledig antwoord:")
for pair in df_compare_avg.index:
    full = df_compare_avg.loc[pair, "Volledig antwoord"]
    opts = df_compare_avg.loc[pair, "Strategische opties (stap 6)"]
    delta = opts - full
    direction = "↓ minder gelijk" if delta < -0.02 else "↑ meer gelijk" if delta > 0.02 else "≈ vergelijkbaar"
    print(f"  {pair:12s}: volledig={full:.3f}, opties={opts:.3f}, Δ={delta:+.3f} {direction}")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# 4.14c  Thematische classificatie (keyword) + t-SNE visualisatie
# ═══════════════════════════════════════════════════════════════════════
from sklearn.manifold import TSNE

# ── 1. Keyword-gebaseerde thematische classificatie ──
# (geen LLM nodig — robuust en reproduceerbaar)
THEME_KEYWORDS = {
    "menukaart_optimalisatie": [
        "menu", "menukaart", "pricing", "prijs", "prijsstelling", "portie",
        "menu-engineering", "food cost", "marge", "winstmarge", "gerecht",
        "kaart", "menustructuur", "upsell", "cross-sell",
    ],
    "assortiment_innovatie": [
        "seizoen", "seizoensgebonden", "nieuw", "innovatie", "variatie",
        "plant-based", "vegan", "vegetarisch", "trend", "fusion",
        "specials", "dagmenu", "weekmenu", "assortiment", "aanbod",
    ],
    "marketing_positionering": [
        "marketing", "merk", "branding", "social media", "instagram",
        "communicatie", "doelgroep", "segment", "positionering", "imago",
        "zichtbaarheid", "online", "recensie", "review", "reputatie",
        "storytelling", "campagne", "promotie",
    ],
    "operationele_efficientie": [
        "kosten", "inkoop", "verspilling", "efficiënt", "operationeel",
        "leverancier", "voorraad", "personeel", "planning", "proces",
        "kostenbesparing", "waste", "foodwaste", "bezetting",
    ],
    "klantbeleving": [
        "beleving", "ervaring", "service", "ambiance", "gastvrijheid",
        "loyalty", "klantbinding", "tevredenheid", "terugkeer", "sfeer",
        "interieur", "gastheer", "hospitality", "klant",
    ],
    "digitalisering": [
        "digitaal", "app", "bestelsysteem", "delivery", "bezorg",
        "technologie", "qr", "reservering", "online bestellen",
        "platform", "thuisbezorg", "takeaway",
    ],
    "duurzaamheid": [
        "duurzaam", "lokaal", "biologisch", "milieu", "circulair",
        "footprint", "seizoensgroenten", "streekproduct", "fair",
        "verantwoord", "co2", "klimaat",
    ],
}

def classify_by_keywords(text):
    """Classificeer optie op basis van keyword-matching (case-insensitive)."""
    text_lower = text.lower()
    scores = {}
    for theme, keywords in THEME_KEYWORDS.items():
        scores[theme] = sum(1 for kw in keywords if kw in text_lower)
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "overig"

print("Thematische classificatie (keyword-based)...")
df_options["theme"] = df_options["option_text"].apply(classify_by_keywords)

for _, row in df_options.iterrows():
    print(f"  {row['experiment'].upper()}/{row['case_id']}/opt{row['option_idx']}: {row['theme']}")

# ── 2. Thematische overlap-analyse ──
print("\n=== Thematische verdeling per experiment ===")
theme_dist = pd.crosstab(df_options["experiment"], df_options["theme"], margins=True)
print(theme_dist.to_string())

# Jaccard similarity: welke thema's delen twee experimenten?
def jaccard_themes(exp_a, exp_b):
    themes_a = set(df_options[df_options["experiment"] == exp_a]["theme"])
    themes_b = set(df_options[df_options["experiment"] == exp_b]["theme"])
    if not themes_a and not themes_b:
        return np.nan
    return len(themes_a & themes_b) / len(themes_a | themes_b)

print("\nJaccard similarity (thematische overlap):")
for e1, e2 in combinations(EXPERIMENTS, 2):
    j = jaccard_themes(e1, e2)
    print(f"  {e1.upper()} vs {e2.upper()}: {j:.3f}")

# ── 3. t-SNE visualisatie van de optie-ruimte ──
perplexity = min(8, len(df_options) - 1)
tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42, n_iter=1000)
coords_2d = tsne.fit_transform(option_embeddings)
df_options["x"] = coords_2d[:, 0]
df_options["y"] = coords_2d[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# 3a. t-SNE gekleurd per experiment
for exp in EXPERIMENTS:
    subset = df_options[df_options["experiment"] == exp]
    axes[0].scatter(subset["x"], subset["y"], c=palette[exp], s=150,
                    label=f"{exp.upper()} ({EXP_LABELS[exp]})",
                    edgecolors="black", linewidth=0.5, zorder=3)
    for _, r in subset.iterrows():
        axes[0].annotate(f"{r['case_id'][-1]}",
                         (r["x"], r["y"]), fontsize=7, ha="center", va="bottom",
                         xytext=(0, 5), textcoords="offset points")
axes[0].set_title("t-SNE: strategische opties per experiment")
axes[0].legend(fontsize=9)
axes[0].set_xlabel("t-SNE dim 1")
axes[0].set_ylabel("t-SNE dim 2")

# 3b. t-SNE gekleurd per case (vorm = experiment)
case_colors = {"case_1": "#e74c3c", "case_2": "#3498db", "case_3": "#2ecc71",
               "case_4": "#f39c12", "case_5": "#9b59b6"}
exp_markers = {"a": "o", "b": "s", "c": "D", "d": "^"}
for case in cases:
    for exp in EXPERIMENTS:
        subset = df_options[(df_options["case_id"] == case) & (df_options["experiment"] == exp)]
        if subset.empty:
            continue
        axes[1].scatter(subset["x"], subset["y"], c=case_colors.get(case, "gray"),
                        marker=exp_markers[exp], s=150,
                        edgecolors="black", linewidth=0.5, zorder=3)
from matplotlib.lines import Line2D
case_handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=c,
                       markersize=10, label=k) for k, c in case_colors.items()]
exp_handles = [Line2D([0], [0], marker=m, color="w", markerfacecolor="gray",
                      markersize=10, label=f"{e.upper()}") for e, m in exp_markers.items()]
axes[1].legend(handles=case_handles + exp_handles, fontsize=8, ncol=2, loc="best")
axes[1].set_title("t-SNE: opties per case (kleur) + experiment (vorm)\n(cluster = case-gebonden, niet experiment-gebonden)")
axes[1].set_xlabel("t-SNE dim 1")
axes[1].set_ylabel("t-SNE dim 2")

# 3c. Thematische verdeling per experiment (stacked bar)
theme_pivot = pd.crosstab(df_options["experiment"], df_options["theme"], normalize="index")
theme_pivot.reindex(EXPERIMENTS).plot(
    kind="bar", stacked=True, ax=axes[2], colormap="Set3", edgecolor="black"
)
axes[2].set_title("Thematische verdeling per experiment\n(proportioneel)")
axes[2].set_ylabel("Aandeel")
axes[2].set_xlabel("Experiment")
axes[2].legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
axes[2].set_ylim(0, 1.05)
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

# ── Unieke vs gedeelde thema's per experiment ──
print("\n=== Unieke thema's per experiment ===")
for exp in EXPERIMENTS:
    exp_themes = set(df_options[df_options["experiment"] == exp]["theme"])
    other_themes = set(df_options[df_options["experiment"] != exp]["theme"])
    unique = exp_themes - other_themes
    shared = exp_themes & other_themes
    print(f"  {exp.upper()} ({EXP_LABELS[exp]}): {len(exp_themes)} thema's | "
          f"uniek: {unique if unique else 'geen'} | gedeeld: {shared}")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# 4.14d  Convergentie-conclusie + case-level deep dive
# ═══════════════════════════════════════════════════════════════════════

# ── 1. Case-level convergentie heatmap ──
fig, axes = plt.subplots(1, len(cases), figsize=(4 * len(cases), 3.5),
                          sharey=True)
if len(cases) == 1:
    axes = [axes]

for idx, case in enumerate(cases):
    case_opts = df_options[df_options["case_id"] == case]
    exps_in_case = [e for e in EXPERIMENTS if e in case_opts["experiment"].values]
    n = len(exps_in_case)
    sim_matrix = np.eye(n)
    
    for i, e1 in enumerate(exps_in_case):
        for j, e2 in enumerate(exps_in_case):
            if i < j:
                idxs_1 = case_opts[case_opts["experiment"] == e1]["embedding_idx"].values
                idxs_2 = case_opts[case_opts["experiment"] == e2]["embedding_idx"].values
                sim = option_set_similarity(idxs_1, idxs_2, option_embeddings)
                sim_matrix[i, j] = sim
                sim_matrix[j, i] = sim
    
    sns.heatmap(
        pd.DataFrame(sim_matrix, index=[e.upper() for e in exps_in_case],
                     columns=[e.upper() for e in exps_in_case]),
        annot=True, fmt=".2f", cmap="RdYlGn", vmin=0.5, vmax=1.0,
        ax=axes[idx], linewidths=0.5, square=True,
        cbar=idx == len(cases) - 1  # alleen rechts een colorbar
    )
    axes[idx].set_title(f"{case}", fontsize=11)

plt.suptitle("Optie-convergentie per case (stap 6: Strategische opties)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# ── 2. Convergentie-score per experiment (gem. similarity met alle anderen) ──
print("\n" + "=" * 70)
print("CONVERGENTIE VAN STRATEGISCHE OPTIES — SAMENVATTING")
print("=" * 70)

# Per experiment: gemiddelde similarity met alle andere experimenten
for exp in EXPERIMENTS:
    sims = df_conv[(df_conv["exp_1"] == exp) | (df_conv["exp_2"] == exp)]["similarity"]
    print(f"  {exp.upper()} ({EXP_LABELS[exp]}): gem. convergentie = {sims.mean():.3f} (±{sims.std():.3f})")

# Totaal
overall_conv = df_conv["similarity"].mean()
print(f"\n  OVERALL convergentie: {overall_conv:.3f}")

if overall_conv > 0.85:
    print("  → Architecturen genereren STERK OVERLAPPENDE strategieën")
    print("    De keuze voor RAG/web search/self-refine verandert de kern-adviezen nauwelijks.")
elif overall_conv > 0.70:
    print("  → Architecturen tonen MATIGE DIVERGENTIE in strategische opties")
    print("    Er zijn herkenbare overeenkomsten, maar ook duidelijke unieke elementen.")
else:
    print("  → Architecturen genereren STERK UITEENLOPENDE strategieën")
    print("    De retrieval-component heeft een groot effect op de inhoud van het advies.")

# ── 3. Conclusietabel ──
print("\n\n=== Conclusietabel ===")
print(f"{'Meting':<45s} {'Waarde':<20s} {'Interpretatie'}")
print("-" * 90)

# Gemiddelde inter-experiment similarity (volledig antwoord, uit §4.5)
full_sim_cols = [c for c in similarity_df.columns if "_vs_" in c]
full_avg = similarity_df[full_sim_cols].mean().mean()
opt_avg = df_conv["similarity"].mean()

print(f"{'Similarity volledig antwoord (§4.5)':<45s} {full_avg:<20.3f} {'Hoog: antwoorden lijken globaal op elkaar'}")
print(f"{'Similarity strategische opties (§4.14)':<45s} {opt_avg:<20.3f} {'Lager/Hoger dan volledig?'}")
print(f"{'Delta (opties vs volledig)':<45s} {opt_avg - full_avg:<+20.3f} {'< 0 = opties variëren méér dan het geheel'}")

# Thematische overlap
jacc_vals = []
for e1, e2 in combinations(EXPERIMENTS, 2):
    jacc_vals.append(jaccard_themes(e1, e2))
mean_jacc = np.mean(jacc_vals)
print(f"{'Gem. Jaccard thematische overlap':<45s} {mean_jacc:<20.3f} {'1.0 = identieke thema\'s, 0 = geen overlap'}")

n_unique = sum(1 for exp in EXPERIMENTS 
               if (set(df_options[df_options['experiment'] == exp]['theme']) - 
                   set(df_options[df_options['experiment'] != exp]['theme'])))
print(f"{'Experimenten met unieke thema\'s':<45s} {str(n_unique) + f"/{len(EXPERIMENTS)}":<20s} {'Meer = grotere strategische diversiteit'}")
print(f"{'Aantal geëxtraheerde opties (totaal)':<45s} {len(df_options):<20d} {''}")
print(f"{'Opties per experiment (gem.)':<45s} {len(df_options)/len(EXPERIMENTS):<20.1f} {''}")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# §4.16  Case Difficulty Analyse
# ═══════════════════════════════════════════════════════════════════════
#
# KERNVRAAG: Welke restaurantcases zijn "moeilijk" voor ALLE architecturen?
#            Als een case consistent lager scoort, zegt dat iets over de
#            complexiteit van de casus — niet over de architectuur.
#
# Moeilijkheidsdimensies (6 metrics per case, gemiddeld over experimenten):
#   1. Stabiliteit          — inter-run cosine similarity (hogere = makkelijker)
#   2. Antwoordlengte       — completion tokens (meer tokens = meer te zeggen)
#   3. Informatiestroom     — analyse→advies retentie (hogere = coherenter)
#   4. Thematische spreiding — entropie van thema-verdeling (hogere = diffuser)
#   5. Thematische volatiliteit — verschuiving door fasen (hogere = instabieler)
#   6. Inter-experiment convergentie — overeenstemming architecturen (hogere = makkelijker)
# ═══════════════════════════════════════════════════════════════════════

from scipy.stats import entropy
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

cases = sorted(df_run1["case_id"].unique())
print(f"Cases: {cases}")

# ── Stap 0: Wat is elke case? ──
print("\n" + "═" * 75)
print("  CASE-OMSCHRIJVINGEN (stap 1: Probleemdefinitie, experiment A)")
print("═" * 75)

case_beschrijvingen = {}
for case_id in cases:
    row = df_run1[(df_run1["case_id"] == case_id) & (df_run1["experiment"] == "a")]
    if row.empty:
        continue
    steps = extract_steps(row.iloc[0]["answer"])
    if steps and "1" in steps:
        t = _step_to_text(steps["1"])
        # Eerste 2 zinnen als samenvatting
        zinnen = [z.strip() for z in re.split(r'[.!?]', t) if len(z.strip()) > 10][:2]
        samenvatting = ". ".join(zinnen) + "."
        case_beschrijvingen[case_id] = samenvatting
        print(f"\n  {case_id}: {samenvatting[:200]}")

# ═══════════════════════════════════════════════════════════════════════
# METRIC 1: Stabiliteit (inter-run cosine similarity)
# ═══════════════════════════════════════════════════════════════════════
print("\n" + "─" * 75)
print("  METRIC 1: Stabiliteit (herberekend)")
print("─" * 75)

def mean_pairwise_cosine(texts, model):
    texts = [t.strip() for t in texts if isinstance(t, str) and t.strip()]
    if len(texts) < 2:
        return np.nan
    emb = model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
    sim = cos_sim(emb)
    iu = np.triu_indices_from(sim, k=1)
    return float(sim[iu].mean())

stab_rows = []
for exp in EXPERIMENTS:
    for case_id in cases:
        sub = df_runs[(df_runs["experiment"] == exp) & (df_runs["case_id"] == case_id)]
        texts = sub["answer"].tolist()
        s = mean_pairwise_cosine(texts, embedding_model)
        stab_rows.append({"experiment": exp, "case_id": case_id, "stability": s})

stability_by_case = pd.DataFrame(stab_rows)
stab_per_case = stability_by_case.groupby("case_id")["stability"].mean()
print(stab_per_case.round(3).to_string())

# ═══════════════════════════════════════════════════════════════════════
# METRIC 2: Antwoordlengte (completion tokens)
# ═══════════════════════════════════════════════════════════════════════
len_per_case = df_runs.groupby("case_id")["completion_tokens"].mean()

# ═══════════════════════════════════════════════════════════════════════
# METRIC 3: Informatiestroom (analyse→advies retentie)
# ═══════════════════════════════════════════════════════════════════════
print("\n" + "─" * 75)
print("  METRIC 3: Analyse→Advies informatiestroom")
print("─" * 75)

flow_rows = []
for _, row in df_run1.iterrows():
    exp, case_id = row["experiment"], row["case_id"]
    steps = extract_steps(row["answer"])
    if not steps:
        continue
    # Analyse tekst (stappen 1-4)
    analyse_teksten = []
    for s in ["1", "2", "3", "4"]:
        if s in steps:
            t = _step_to_text(steps[s])
            if t and len(t.strip()) > 20:
                analyse_teksten.append(t)
    # Advies tekst (stap 8)
    advies_tekst = _step_to_text(steps.get("8", "")) if "8" in steps else ""
    
    if analyse_teksten and advies_tekst and len(advies_tekst.strip()) > 20:
        emb_analyse = embedding_model.encode(
            [" ".join(analyse_teksten)], convert_to_numpy=True, normalize_embeddings=True
        )
        emb_advies = embedding_model.encode(
            [advies_tekst], convert_to_numpy=True, normalize_embeddings=True
        )
        sim = float(cos_sim(emb_analyse, emb_advies)[0, 0])
        flow_rows.append({"experiment": exp, "case_id": case_id, "flow": sim})

df_case_flow = pd.DataFrame(flow_rows)
flow_per_case = df_case_flow.groupby("case_id")["flow"].mean()
print(flow_per_case.round(3).to_string())

# ═══════════════════════════════════════════════════════════════════════
# METRIC 4: Thematische spreiding (entropie)
# ═══════════════════════════════════════════════════════════════════════
# Hogere entropie = meer gelijkmatig verdeelde thema's = diffuser probleem
entropy_rows = []
for (exp, case_id), grp in df_norm.groupby(["experiment", "case_id"]):
    # Gemiddeld over alle fasen
    thema_scores = grp.groupby("thema")["aandeel"].mean().values
    thema_probs = thema_scores / thema_scores.sum()
    h = entropy(thema_probs)  # Shannon entropy
    entropy_rows.append({"experiment": exp, "case_id": case_id, "entropy": h})

df_entropy = pd.DataFrame(entropy_rows)
entropy_per_case = df_entropy.groupby("case_id")["entropy"].mean()

# ═══════════════════════════════════════════════════════════════════════
# METRIC 5: Thematische volatiliteit
# ═══════════════════════════════════════════════════════════════════════
# Gem. absolute verschuiving van thema-aandelen door de fasen
vol_rows = []
for (exp, case_id), grp in df_norm.groupby(["experiment", "case_id"]):
    pivot = grp.pivot_table(index="thema", columns="fase", values="aandeel", aggfunc="mean")[fase_order]
    # Gemiddelde absolute verschuiving per opeenvolgende fasen
    total_shift = 0
    for i in range(len(fase_order) - 1):
        total_shift += (pivot[fase_order[i+1]] - pivot[fase_order[i]]).abs().mean()
    avg_shift = total_shift / (len(fase_order) - 1)
    vol_rows.append({"experiment": exp, "case_id": case_id, "volatility": avg_shift})

df_vol = pd.DataFrame(vol_rows)
vol_per_case = df_vol.groupby("case_id")["volatility"].mean()

# ═══════════════════════════════════════════════════════════════════════
# METRIC 6: Inter-experiment convergentie
# ═══════════════════════════════════════════════════════════════════════
# Cosine similarity tussen architecturen voor dezelfde case
print("\n" + "─" * 75)
print("  METRIC 6: Inter-experiment convergentie")
print("─" * 75)

conv_rows = []
for case_id in cases:
    sub = df_run1[df_run1["case_id"] == case_id]
    if len(sub) < 2:
        continue
    texts = sub["answer"].tolist()
    emb = embedding_model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
    sim = cos_sim(emb)
    iu = np.triu_indices_from(sim, k=1)
    conv_rows.append({"case_id": case_id, "convergence": float(sim[iu].mean())})

df_conv_case = pd.DataFrame(conv_rows).set_index("case_id")
conv_per_case = df_conv_case["convergence"]
print(conv_per_case.round(3).to_string())

# ═══════════════════════════════════════════════════════════════════════
# SAMENVATTINGSTABEL
# ═══════════════════════════════════════════════════════════════════════
df_difficulty = pd.DataFrame({
    "Stabiliteit": stab_per_case,
    "Antwoordlengte": len_per_case,
    "Informatiestroom": flow_per_case,
    "Thema-entropie": entropy_per_case,
    "Thema-volatiliteit": vol_per_case,
    "Convergentie": conv_per_case,
}).reindex(cases)

# Voeg case-namen toe
df_difficulty.index.name = "case_id"

print("\n" + "═" * 75)
print("  CASE DIFFICULTY PROFIEL")
print("═" * 75)
display(df_difficulty.round(3))

# ── Normaliseer alles naar 0-1 schaal voor vergelijking ──
# Hogere score = "makkelijker" voor alle metrics behalve volatiliteit en entropie
df_norm_diff = df_difficulty.copy()
for col in df_norm_diff.columns:
    vals = df_norm_diff[col]
    mn, mx = vals.min(), vals.max()
    if mx > mn:
        df_norm_diff[col] = (vals - mn) / (mx - mn)
    else:
        df_norm_diff[col] = 0.5

# Inverteer metrics waar hoger = moeilijker
df_norm_diff["Thema-entropie"] = 1 - df_norm_diff["Thema-entropie"]
df_norm_diff["Thema-volatiliteit"] = 1 - df_norm_diff["Thema-volatiliteit"]

# Moeilijkheidsindex: gemiddelde van alle genormaliseerde metrics
# Lager = moeilijker
df_norm_diff["Difficulty Index"] = df_norm_diff.mean(axis=1)
df_norm_diff = df_norm_diff.sort_values("Difficulty Index")

print("\nGenormaliseerde scores (0=moeilijkst, 1=makkelijkst):")
display(df_norm_diff.round(3))

# ═══════════════════════════════════════════════════════════════════════
# FIGUUR 1: Radar chart per case
# ═══════════════════════════════════════════════════════════════════════
from matplotlib.patches import FancyBboxPatch

metrics = ["Stabiliteit", "Antwoordlengte", "Informatiestroom",
           "Thema-entropie", "Thema-volatiliteit", "Convergentie"]
n_metrics = len(metrics)
angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
angles += angles[:1]  # sluit de cirkel

case_kleuren = {
    "case_1": "#E74C3C", "case_2": "#3498DB", "case_3": "#2ECC71",
    "case_4": "#F39C12", "case_5": "#9B59B6",
}

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

for case_id in df_norm_diff.index:
    vals = df_norm_diff.loc[case_id, metrics].values.tolist()
    vals += vals[:1]
    kleur = case_kleuren.get(case_id, "#999")
    idx_val = df_norm_diff.loc[case_id, "Difficulty Index"]
    
    ax.plot(angles, vals, "o-", color=kleur, linewidth=2.5, markersize=8,
            label=f"{case_id} (idx={idx_val:.2f})", alpha=0.85)
    ax.fill(angles, vals, color=kleur, alpha=0.08)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=11, fontweight="bold")
ax.set_ylim(0, 1.1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["0.25", "0.50", "0.75", "1.00"], fontsize=9, color="#666")
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=11)
ax.set_title("§4.16  Case Difficulty Profiel\n(0=moeilijkst, 1=makkelijkst)",
             fontsize=15, fontweight="bold", pad=25)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════════════
# FIGUUR 2: Barplot difficulty index + breakdown
# ═══════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(18, 7), gridspec_kw={"width_ratios": [1, 2]})

# Links: Difficulty Index ranking
ax = axes[0]
sorted_cases = df_norm_diff.sort_values("Difficulty Index")
colors = [case_kleuren.get(c, "#999") for c in sorted_cases.index]
ax.barh(range(len(sorted_cases)), sorted_cases["Difficulty Index"], color=colors,
        edgecolor="white", linewidth=1.5)
for i, (case_id, val) in enumerate(sorted_cases["Difficulty Index"].items()):
    label = "MOEILIJKST" if i == 0 else ("MAKKELIJKST" if i == len(sorted_cases) - 1 else "")
    ax.text(val + 0.01, i, f"{val:.2f}  {label}", va="center", fontsize=11,
            fontweight="bold" if label else "normal")
ax.set_yticks(range(len(sorted_cases)))
ax.set_yticklabels(sorted_cases.index, fontsize=12, fontweight="bold")
ax.set_xlabel("Difficulty Index (0=moeilijk, 1=makkelijk)", fontsize=11)
ax.set_title("Case ranking", fontsize=13, fontweight="bold")
ax.set_xlim(0, 1.15)
ax.grid(True, alpha=0.2, axis="x")
ax.spines[["top", "right"]].set_visible(False)

# Rechts: Heatmap met alle metrics
ax = axes[1]
heat_data = df_norm_diff.loc[sorted_cases.index, metrics]
sns.heatmap(heat_data, annot=True, fmt=".2f", cmap="RdYlGn",
            vmin=0, vmax=1, linewidths=1.5, ax=ax,
            cbar_kws={"label": "Score (0=moeilijk, 1=makkelijk)", "shrink": 0.8})
ax.set_title("Metric breakdown per case", fontsize=13, fontweight="bold")
ax.set_ylabel("")

plt.suptitle("§4.16  Welke restaurantcases zijn het moeilijkst?",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════════════
# FIGUUR 3: Per-experiment variatie per case (stability heatmap)
# ═══════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 5))
heat_stab = stability_by_case.pivot_table(
    index="case_id", columns="experiment", values="stability"
).reindex(sorted_cases.index)
heat_stab.columns = [f"{e.upper()} ({EXP_LABELS[e]})" for e in heat_stab.columns]

sns.heatmap(heat_stab, annot=True, fmt=".3f", cmap="RdYlGn",
            vmin=0.7, vmax=1.0, linewidths=1.5, ax=ax,
            cbar_kws={"label": "Stabiliteit (cosine sim)"})
ax.set_title("§4.16  Stabiliteit per case × experiment\n(geordend op moeilijkheid)",
             fontsize=13, fontweight="bold")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════════════
# NUMERIEKE CONCLUSIE
# ═══════════════════════════════════════════════════════════════════════
print("\n" + "═" * 75)
print("  CASE DIFFICULTY — CONCLUSIE")
print("═" * 75)

easiest = df_norm_diff["Difficulty Index"].idxmax()
hardest = df_norm_diff["Difficulty Index"].idxmin()

print(f"\n  Makkelijkste case: {easiest} (idx={df_norm_diff.loc[easiest, 'Difficulty Index']:.3f})")
if easiest in case_beschrijvingen:
    print(f"    → {case_beschrijvingen[easiest][:150]}")

print(f"\n  Moeilijkste case:  {hardest} (idx={df_norm_diff.loc[hardest, 'Difficulty Index']:.3f})")
if hardest in case_beschrijvingen:
    print(f"    → {case_beschrijvingen[hardest][:150]}")

print(f"\n  Spreiding difficulty index: {df_norm_diff['Difficulty Index'].max() - df_norm_diff['Difficulty Index'].min():.3f}")

# Per metric: welke case is het zwakst?
print(f"\n  Per metric, moeilijkste case:")
for m in metrics:
    worst = df_norm_diff[m].idxmin()
    print(f"    {m:25s}  → {worst} ({df_norm_diff.loc[worst, m]:.3f})")

# Correlatie tussen metrics
print(f"\n  Correlatie tussen difficulty-dimensies:")
corr = df_norm_diff[metrics].corr()
for i, m1 in enumerate(metrics):
    for j, m2 in enumerate(metrics):
        if i < j and abs(corr.loc[m1, m2]) > 0.5:
            print(f"    {m1} × {m2}: r={corr.loc[m1, m2]:.2f}")


# 5. Conclusie en synthese

## Kernbevindingen

| Dimensie | Beste architectuur | Bevinding |
|----------|-------------------|----------|
| **Stabiliteit (S)** | A (Baseline) | Hoogste reproduceerbaarheid over 5 runs; D beperkt door 1 run |
| **Kwaliteit (Judge)** | D (Self-Refine) | Hoogste rubric-scores, maar beïnvloed door verbosity bias |
| **Pairwise ranking** | A (Baseline) | Consistent preferentie boven andere architecturen |
| **CGR** | B (RAG) | Beste grounding door externe kenniscontext |
| **Scenario diversiteit** | A (Baseline) | Iets diverser maar niet significant; case-effecten domineren |
| **Kosten-efficiëntie** | A (Baseline) | Beste score-per-token en score-per-seconde ratio |

## Methodologische contradictie

De **Judge-Ranking contradictie** (§4.13) is de belangrijkste methodologische bevinding:
- **Rubric scoring** (0–100) beloont volledigheid en detail → D wint
- **Pairwise vergelijking** beloont precisie en relevantie → A wint
- **Oorzaak**: Verbosity bias — D genereert 3× meer tokens, wat rubric-scores opdrijft zonder proportionele kwaliteitswinst

## Temperatuuranalyse (§4.2)

- De experiment-ranking is **grotendeels robuust** over temperaturen 0.3–1.0
- Temperatuur heeft **geen significant effect** op intra-advies scenario diversiteit
- Architectuurkeuze is belangrijker dan parametertuning

## Beperkingen

1. **Kleine steekproef** — N=5 cases beperkt de statistische power; grote effecten kunnen niet-significant zijn
2. **Één model** — Alle resultaten gelden voor gpt-4o-mini; generaliseerbaarheid naar andere LLMs onbekend
3. **Experiment D** — Slechts 1 run voor de hoofdexperimenten; stabiliteitsschatting beperkt
4. **Web search (C)** — DuckDuckGo blokkeerde requests vanuit Databricks compute; C gedroeg zich als baseline
5. **Judge bias** — LLM-as-judge met hetzelfde model (gpt-4o-mini) kan systematische voorkeuren hebben
6. **CGR en lengte** — Ruwe CGR is gevoelig voor antwoordlengte; lengte-genormaliseerde CGR (§4.8.1) geeft robuuster beeld

## Aanbevelingen

- Voor **productie-inzet**: Experiment A (Baseline) biedt de beste balans tussen kwaliteit, stabiliteit en efficiëntie
- Voor **maximale inhoudelijke diepgang**: Experiment B (RAG) met domeinspecifieke kennisbank
- **Self-Refine (D)** wordt afgeraden door hoge kosten en verbosity bias zonder proportionele kwaliteitswinst
- Verhoog N (meer cases/runs) voor krachtigere statistische conclusies

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# EXPORT: Sla alle belangrijke grafieken op als PNG
# ═══════════════════════════════════════════════════════════════════════

import os
import matplotlib
matplotlib.use("Agg")  # non-interactive backend voor savefig
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

FIG_DIR = "/Workspace/Users/rgoedings@hanos.nl/AI_op_het_menu/figures"
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(fig, name, dpi=150):
    path = os.path.join(FIG_DIR, f"{name}.png")
    fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  ✅ {name}.png")

print("Exporteren van figuren...\n")

# ──────────────────────────────────────────────────────────────────────
# 1. Stabiliteit barplot (§4.1 / §4.5)
# ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
exp_names = stability_summary["experiment"].tolist()
stab_vals = stability_summary["stability"].tolist()
colors_list = [palette.get(e, "#999") for e in exp_names]
bars = ax.bar(exp_names, [v if pd.notna(v) else 0 for v in stab_vals], color=colors_list)
for i, (exp, val) in enumerate(zip(exp_names, stab_vals)):
    if pd.notna(val):
        ax.text(i, val + 0.01, f"{val:.3f}", ha="center", fontweight="bold")
ax.set_title("Stabiliteit per architectuur", fontsize=14)
ax.set_ylabel("Cosine similarity")
ax.set_xlabel("Experiment")
ax.set_ylim(0, 1.05)
plt.tight_layout()
savefig(fig, "fig_4_1_stabiliteit_barplot")

# ──────────────────────────────────────────────────────────────────────
# 2. Temperatuur × scenario diversiteit (§4.2)
# ──────────────────────────────────────────────────────────────────────
if 'df_ht_valid' in dir() and not df_ht_valid.empty:
    temps_sorted = sorted(df_ht_valid["temperature"].unique())
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    sns.boxplot(data=df_ht_valid, x="temperature", y="diversity", ax=axes[0, 0],
                color="#4C72B0", showmeans=True,
                meanprops={"marker": "D", "markerfacecolor": "white", "markersize": 8})
    for i, t in enumerate(temps_sorted):
        vals = df_ht_valid[df_ht_valid["temperature"] == t]["diversity"]
        if len(vals) > 0:
            axes[0, 0].text(i, vals.mean() + 0.008, f"μ={vals.mean():.3f}", ha="center", fontsize=9, fontweight="bold")
    axes[0, 0].set_title("Scenario diversiteit per temperatuur")
    axes[0, 0].set_ylabel("Diversiteit (1 − gem. cosine sim)")
    axes[0, 0].set_xlabel("Temperature")
    axes[0, 0].grid(True, alpha=0.3)

    for exp in EXPERIMENTS:
        sub = df_ht_valid[df_ht_valid["experiment"] == exp]
        means = sub.groupby("temperature")["diversity"].mean()
        stds = sub.groupby("temperature")["diversity"].std()
        axes[0, 1].plot(means.index, means.values, "o-", color=palette[exp],
                        label=f"{exp.upper()} ({EXP_LABELS[exp]})", linewidth=2)
        axes[0, 1].fill_between(means.index, means - stds, means + stds, alpha=0.15, color=palette[exp])
    axes[0, 1].set_xlabel("Temperature")
    axes[0, 1].set_ylabel("Gem. diversiteit")
    axes[0, 1].set_title("Diversiteit per experiment × temperatuur")
    axes[0, 1].legend(fontsize=9)
    axes[0, 1].grid(True, alpha=0.3)

    hm_data = df_ht_valid.pivot_table(index="experiment", columns="temperature",
                                       values="diversity", aggfunc="mean")
    hm_data = hm_data.reindex(EXPERIMENTS)
    sns.heatmap(hm_data, annot=True, fmt=".3f", cmap="RdYlGn",
                vmin=0, vmax=hm_data.values.max() * 1.2,
                linewidths=0.5, ax=axes[1, 0])
    axes[1, 0].set_title("Diversiteit heatmap (experiment × temperature)")
    axes[1, 0].set_ylabel("Experiment")
    axes[1, 0].set_xlabel("Temperature")

    axes[1, 1].text(0.5, 0.5, "Stabiliteit vs diversiteit\n(zie notebook)",
                    ha="center", va="center", fontsize=12, transform=axes[1, 1].transAxes)
    axes[1, 1].set_title("Trade-off: stabiliteit vs diversiteit")

    plt.suptitle("§4.2 Temperatuur × scenario diversiteit", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    savefig(fig, "fig_4_2_temperatuur_diversiteit")
else:
    print("  ⚠️ df_ht_valid niet beschikbaar — §4.2 figuur overgeslagen")

# ──────────────────────────────────────────────────────────────────────
# 3. Judge scores barplot + heatmap (§4.3)
# ──────────────────────────────────────────────────────────────────────
if 'df_judge' in dir() and not df_judge.empty:
    SCORE_KEYS = ["probleemdefinitie", "analyse", "scenarios", "advies", "cot_samenhang"]
    judge_melted = df_judge.melt(
        id_vars=["experiment", "case_id"],
        value_vars=SCORE_KEYS,
        var_name="criterium",
        value_name="score"
    )

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(data=judge_melted, x="criterium", y="score", hue="experiment",
                order=SCORE_KEYS, hue_order=EXPERIMENTS, ax=ax)
    ax.set_ylim(0, 105)
    ax.set_title("Score per criterium per experiment (0–100)", fontsize=14)
    ax.set_ylabel("Score (0–100)")
    ax.set_xlabel("Criterium")
    ax.axhline(y=55, color="gray", linestyle="--", alpha=0.5, label="voldoende (55)")
    plt.xticks(rotation=15)
    plt.legend(title="Experiment")
    plt.tight_layout()
    savefig(fig, "fig_4_3a_judge_scores_barplot")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.barplot(data=df_judge, x="experiment", y="totaal", hue="experiment",
                order=EXPERIMENTS, hue_order=EXPERIMENTS,
                palette=palette, legend=False, ax=axes[0])
    axes[0].set_title("Gemiddelde totaalscore per experiment (0–100)")
    axes[0].set_ylabel("Score (0–100)")
    axes[0].set_ylim(0, 105)

    heatmap_data = df_judge.groupby("experiment")[SCORE_KEYS].mean()
    heatmap_data = heatmap_data.reindex(EXPERIMENTS)
    sns.heatmap(heatmap_data.T, annot=True, fmt=".1f", cmap="RdYlGn", vmin=0, vmax=100,
                linewidths=0.5, ax=axes[1])
    axes[1].set_title("Gemiddelde score per criterium per experiment")
    axes[1].set_ylabel("Criterium")
    axes[1].set_xlabel("Experiment")
    plt.tight_layout()
    savefig(fig, "fig_4_3b_judge_heatmap")
else:
    print("  ⚠️ df_judge niet beschikbaar — §4.3 figuren overgeslagen (vereist API calls)")

# ──────────────────────────────────────────────────────────────────────
# 4. Informatiestroom CoT (§4.4.1)
# ──────────────────────────────────────────────────────────────────────
if 'df_flow' in dir() and not df_flow.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Adjacente overlap per experiment
    flow_by_exp = df_flow.groupby(["experiment", "transition"])["similarity"].mean().reset_index()
    for exp in EXPERIMENTS:
        sub = flow_by_exp[flow_by_exp["experiment"] == exp].sort_values("transition")
        axes[0].plot(sub["transition"], sub["similarity"], "o-",
                     color=palette[exp], label=f"{exp.upper()} ({EXP_LABELS[exp]})", linewidth=2)
    axes[0].set_xlabel("Overgang (stap N → N+1)")
    axes[0].set_ylabel("Cosine similarity")
    axes[0].set_title("Informatie-overdracht tussen opeenvolgende stappen")
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.3)
    axes[0].tick_params(axis='x', rotation=45)

    # Cumulatieve retentie per experiment
    if 'df_retention' in dir() and not df_retention.empty:
        # Gebruik de juiste kolomnaam
        ret_col = "cumulative_retention" if "cumulative_retention" in df_retention.columns else "retention"
        step_col = "step_num" if "step_num" in df_retention.columns else "step"
        ret_by_exp = df_retention.groupby(["experiment", step_col])[ret_col].mean().reset_index()
        for exp in EXPERIMENTS:
            sub = ret_by_exp[ret_by_exp["experiment"] == exp].sort_values(step_col)
            axes[1].plot(sub[step_col], sub[ret_col], "o-",
                         color=palette[exp], label=f"{exp.upper()}", linewidth=2)
        axes[1].set_xlabel("Stap")
        axes[1].set_ylabel("Cumulatieve retentie")
        axes[1].set_title("Cumulatieve retentie (hoeveel info uit eerdere stappen terugkomt)")
        axes[1].legend(fontsize=9)
        axes[1].grid(True, alpha=0.3)

    plt.suptitle("§4.4 Informatiestroom door de CoT-keten", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    savefig(fig, "fig_4_4a_informatiestroom")
else:
    print("  ⚠️ df_flow niet beschikbaar — §4.4 figuur overgeslagen")

# ──────────────────────────────────────────────────────────────────────
# 5. Thematische traceerbaarheid (§4.4.4)
# ──────────────────────────────────────────────────────────────────────
if 'df_thema' in dir() and not df_thema.empty:
    thema_pivot = df_thema.groupby(["thema", "fase"])["score"].mean().reset_index()
    thema_wide = thema_pivot.pivot(index="thema", columns="fase", values="score")
    fase_order = [f for f in ["Analyse", "Opties", "Advies", "Implementatie"] if f in thema_wide.columns]
    thema_wide = thema_wide[fase_order]

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(thema_wide, annot=True, fmt=".3f", cmap="YlOrRd",
                linewidths=0.5, ax=ax)
    ax.set_title("Thematische aanwezigheid per fase (gemiddeld over alle experimenten)", fontsize=13)
    ax.set_ylabel("Thema")
    ax.set_xlabel("Fase")
    plt.tight_layout()
    savefig(fig, "fig_4_4b_thema_heatmap")
else:
    print("  ⚠️ df_thema niet beschikbaar — thema heatmap overgeslagen")

# ──────────────────────────────────────────────────────────────────────
# 6. Bump chart thematische focus (§4.4.5b)
# ──────────────────────────────────────────────────────────────────────
if 'df_norm' in dir() and not df_norm.empty:
    norm_avg = df_norm.groupby(["thema", "fase"])["aandeel"].mean().reset_index()
    fase_order_bump = [f for f in ["Analyse", "Opties", "Advies", "Implementatie"] if f in norm_avg["fase"].unique()]

    rank_data = []
    for fase in fase_order_bump:
        fase_df = norm_avg[norm_avg["fase"] == fase].sort_values("aandeel", ascending=False).reset_index(drop=True)
        fase_df["rang"] = range(1, len(fase_df) + 1)
        rank_data.append(fase_df)
    df_ranks = pd.concat(rank_data)

    fig, ax = plt.subplots(figsize=(14, 10))
    themas = df_ranks["thema"].unique()
    if 'THEMA_KLEUREN' in dir():
        colors_thema = THEMA_KLEUREN
    else:
        cmap = plt.cm.get_cmap("tab20", len(themas))
        colors_thema = {t: cmap(i) for i, t in enumerate(themas)}

    for thema in themas:
        sub = df_ranks[df_ranks["thema"] == thema]
        sub = sub.set_index("fase").reindex(fase_order_bump)
        color = colors_thema.get(thema, "gray")
        ax.plot(fase_order_bump, sub["rang"].values, "o-", color=color, linewidth=2.5, markersize=10, label=thema)
        if not np.isnan(sub["rang"].values[0]):
            ax.text(-0.15, sub["rang"].values[0], thema, ha="right", va="center", fontsize=9, color=color, fontweight="bold")
        if not np.isnan(sub["rang"].values[-1]):
            ax.text(len(fase_order_bump) - 0.85, sub["rang"].values[-1], thema, ha="left", va="center", fontsize=9, color=color, fontweight="bold")

    ax.set_ylabel("Rang (1 = hoogste aandeel)")
    ax.set_title("Bump chart: verschuiving thematische focus door de redeneerketen", fontsize=14)
    ax.invert_yaxis()
    ax.set_xticks(range(len(fase_order_bump)))
    ax.set_xticklabels(fase_order_bump, fontsize=12)
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    savefig(fig, "fig_4_4c_bump_chart")
else:
    print("  ⚠️ df_norm niet beschikbaar — bump chart overgeslagen")

# ──────────────────────────────────────────────────────────────────────
# 7. Consistentiematrices (§4.5.1)
# ──────────────────────────────────────────────────────────────────────
cases = sorted(df_runs["case_id"].unique())
for exp in EXPERIMENTS:
    n_runs = df_runs[df_runs["experiment"] == exp]["run"].nunique()
    if n_runs < 2:
        continue
    fig, axes_cm = plt.subplots(1, len(cases), figsize=(20, 4))
    for i, case in enumerate(cases):
        sim_case = run_consistency_matrix_case(exp, case)
        sns.heatmap(sim_case, annot=True, fmt=".3f", cmap="coolwarm",
                    vmin=0.7, vmax=1, ax=axes_cm[i])
        axes_cm[i].set_title(case)
    plt.suptitle(f"Run consistentie — Experiment {exp.upper()} ({n_runs} runs)", fontsize=13)
    plt.tight_layout()
    savefig(fig, f"fig_4_5_consistentie_{exp}")

# ──────────────────────────────────────────────────────────────────────
# 8. Latency boxplot (§4.6)
# ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df_runs, x="experiment", y="latency_seconds", ax=ax,
            order=EXPERIMENTS, palette=palette)
ax.set_title("Latency per experiment", fontsize=14)
ax.set_ylabel("Seconden")
ax.set_xlabel("Experiment")
plt.tight_layout()
savefig(fig, "fig_4_6a_latency_boxplot")

# ──────────────────────────────────────────────────────────────────────
# 9. Stabiliteit boxplot per case (§4.6)
# ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=stability_by_case, x="experiment", y="stability",
            order=EXPERIMENTS, ax=ax, palette=palette)
ax.set_title("Stabiliteit spreiding per case", fontsize=14)
ax.set_ylabel("Cosine similarity")
ax.set_xlabel("Experiment")
plt.tight_layout()
savefig(fig, "fig_4_6b_stabiliteit_boxplot")

# ──────────────────────────────────────────────────────────────────────
# 10. Semantic similarity heatmap (§4.7)
# ──────────────────────────────────────────────────────────────────────
if 'similarity_df' in dir() and not similarity_df.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(similarity_df.set_index("case_id"), annot=True, fmt=".2f",
                cmap="coolwarm", vmin=0.7, vmax=1, ax=ax)
    ax.set_title("Semantische similarity tussen experimenten per case", fontsize=13)
    plt.tight_layout()
    savefig(fig, "fig_4_7_similarity_heatmap")
else:
    print("  ⚠️ similarity_df niet beschikbaar")

# ──────────────────────────────────────────────────────────────────────
# 11. CGR barplot (§4.8)
# ──────────────────────────────────────────────────────────────────────
if 'df_cgr' in dir() and not df_cgr.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sns.barplot(data=df_cgr, x="experiment", y="CGR", ax=axes[0],
                order=EXPERIMENTS, palette=palette)
    axes[0].set_title("Claim Grounding Rate per experiment", fontsize=13)
    axes[0].set_ylim(0, 1.1)
    axes[0].set_ylabel("CGR")

    sns.barplot(data=df_cgr, x="experiment", y="hallucination_rate", ax=axes[1],
                order=EXPERIMENTS, palette=palette)
    axes[1].set_title("Hallucinatiegraad per experiment", fontsize=13)
    axes[1].set_ylim(0, 0.3)
    axes[1].set_ylabel("1 − CGR")
    plt.tight_layout()
    savefig(fig, "fig_4_8_cgr_hallucinatie")
else:
    print("  ⚠️ df_cgr niet beschikbaar — §4.8 figuren overgeslagen (vereist API calls)")

# ──────────────────────────────────────────────────────────────────────
# 12. Kosten-baten (§4.9 / §4.14)
# ──────────────────────────────────────────────────────────────────────
if 'cost_benefit' in dir() and not cost_benefit.empty:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    for _, row in cost_benefit.iterrows():
        exp = row["experiment"]
        axes[0].scatter(row["avg_tokens"], row["judge_score"],
                        color=palette[exp], s=250, zorder=3, edgecolors="black", linewidth=1)
        axes[0].annotate(f" {exp.upper()}", (row["avg_tokens"], row["judge_score"]),
                         fontsize=12, fontweight="bold", color=palette[exp])
    axes[0].set_xlabel("Gemiddeld tokenverbruik →")
    axes[0].set_ylabel("Judge totaalscore (0-100) →")
    axes[0].set_title("Kwaliteit vs Tokens")

    x = np.arange(len(EXPERIMENTS))
    w = 0.35
    axes[1].bar(x - w/2, cost_benefit["quality_per_1k_tokens"].values, w,
                color=[palette[e] for e in EXPERIMENTS], alpha=0.8, edgecolor="black")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([e.upper() for e in EXPERIMENTS])
    axes[1].set_ylabel("Kwaliteit per 1000 tokens")
    axes[1].set_title("Efficiëntie: kwaliteit per token")
    for i, v in enumerate(cost_benefit["quality_per_1k_tokens"].values):
        axes[1].text(i - w/2, v + 0.3, f"{v:.1f}", ha="center", fontsize=9, fontweight="bold")

    for _, row in cost_benefit.iterrows():
        exp = row["experiment"]
        axes[2].scatter(row["avg_latency"], row["judge_score"],
                        color=palette[exp], s=250, zorder=3, edgecolors="black", linewidth=1)
        axes[2].annotate(f" {exp.upper()}", (row["avg_latency"], row["judge_score"]),
                         fontsize=12, fontweight="bold", color=palette[exp])
    axes[2].set_xlabel("Gemiddelde latency (seconden) →")
    axes[2].set_ylabel("Judge totaalscore (0-100) →")
    axes[2].set_title("Kwaliteit vs Snelheid")

    plt.suptitle("§4.9 / §4.14 Kosten-baten analyse", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    savefig(fig, "fig_4_9_kosten_baten")
else:
    print("  ⚠️ cost_benefit niet beschikbaar — §4.9 figuren overgeslagen (vereist judge data)")

# ──────────────────────────────────────────────────────────────────────
# 13. t-SNE strategische opties (§4.15)
# ──────────────────────────────────────────────────────────────────────
if 'df_options' in dir() and 'option_embeddings' in dir() and len(df_options) > 4:
    from sklearn.manifold import TSNE
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(5, len(option_embeddings)-1))
    coords = tsne.fit_transform(option_embeddings)
    df_options_plot = df_options.copy()
    df_options_plot["x"] = coords[:, 0]
    df_options_plot["y"] = coords[:, 1]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for exp in EXPERIMENTS:
        sub = df_options_plot[df_options_plot["experiment"] == exp]
        axes[0].scatter(sub["x"], sub["y"], color=palette[exp], s=100,
                        label=f"{exp.upper()} ({EXP_LABELS[exp]})", alpha=0.8, edgecolors="white")
    axes[0].set_title("t-SNE: strategische opties per experiment")
    axes[0].legend(fontsize=9)
    axes[0].set_xlabel("t-SNE 1")
    axes[0].set_ylabel("t-SNE 2")

    case_colors = plt.cm.Set2(np.linspace(0, 1, len(cases)))
    for i, case in enumerate(cases):
        sub = df_options_plot[df_options_plot["case_id"] == case]
        axes[1].scatter(sub["x"], sub["y"], color=case_colors[i], s=100,
                        label=case, alpha=0.8, edgecolors="white")
    axes[1].set_title("t-SNE: strategische opties per case")
    axes[1].legend(fontsize=9)
    axes[1].set_xlabel("t-SNE 1")
    axes[1].set_ylabel("t-SNE 2")

    plt.suptitle("§4.15 Convergentie strategische opties", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    savefig(fig, "fig_4_15_tsne_opties")
else:
    print("  ⚠️ df_options/option_embeddings niet beschikbaar — t-SNE overgeslagen")

# ──────────────────────────────────────────────────────────────────────
# 14. Case difficulty (§4.16)
# ──────────────────────────────────────────────────────────────────────
if 'df_difficulty' in dir():
    fig, ax = plt.subplots(figsize=(10, 6))
    df_diff_sorted = df_difficulty.sort_values("difficulty_index", ascending=False)
    bars = ax.barh(df_diff_sorted["case_id"], df_diff_sorted["difficulty_index"],
                   color=["#2ecc71" if v > 0.6 else "#f39c12" if v > 0.4 else "#e74c3c"
                          for v in df_diff_sorted["difficulty_index"]])
    ax.set_xlabel("Difficulty Index (0 = moeilijkst, 1 = makkelijkst)")
    ax.set_title("Case moeilijkheid per restaurant", fontsize=14)
    ax.set_xlim(0, 1)
    for i, (_, row) in enumerate(df_diff_sorted.iterrows()):
        ax.text(row["difficulty_index"] + 0.02, i, f"{row['difficulty_index']:.2f}",
                va="center", fontweight="bold")
    plt.tight_layout()
    savefig(fig, "fig_4_16_case_difficulty")
else:
    print("  ⚠️ df_difficulty niet beschikbaar — case difficulty overgeslagen")

# ──────────────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"KLAAR — Alle figuren opgeslagen in: {FIG_DIR}")
print(f"{'='*70}")
print(f"\nBestanden:")
for f in sorted(os.listdir(FIG_DIR)):
    if f.endswith(".png"):
        size_kb = os.path.getsize(os.path.join(FIG_DIR, f)) / 1024
        print(f"  📊 {f} ({size_kb:.0f} KB)")

In [0]:
from IPython.display import Image, display
display(Image(filename="/Workspace/Users/rgoedings@hanos.nl/AI_op_het_menu/figures/fig_13_Analyse_v2.png"))

In [0]:
from IPython.display import Image, display
display(Image(filename="/Workspace/Users/rgoedings@hanos.nl/AI_op_het_menu/figures/fig_thema_dekking_per_architectuur.png"))

In [0]:
from IPython.display import Image, display
display(Image(filename="/Workspace/Users/rgoedings@hanos.nl/AI_op_het_menu/figures/fig_thema_dekking_totaal.png"))